In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[:20]:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/brjapon/cwru-bearing-datasets/feature_time_48k_2048_load_1.csv')
print(df.shape)
print(df.columns.tolist())
print(df.head())
# find the label column — likely 'fault' or 'label' or last column
print(df.iloc[:, -1].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

X = df.drop(columns=['fault'])
le = LabelEncoder()
y = le.fit_transform(df['fault'])          # XGBoost needs numeric labels

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)

clf = XGBClassifier(random_state=42, eval_metric='mlogloss')
clf.fit(Xtr_s, ytr)

pred = clf.predict(Xte_s)
print(classification_report(yte, pred, target_names=le.classes_))

In [ ]:
import shap
import matplotlib.pyplot as plt

# SHAP explainer for the tree model
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(Xte_s)

# Global feature importance across all classes
shap.summary_plot(shap_values, Xte_s, feature_names=X.columns.tolist(),
                  plot_type="bar", show=True)

In [ ]:
# Detailed: how each feature pushes predictions (beeswarm), for one class
# shap_values is a list (one array per class); pick a class index
class_idx = list(le.classes_).index('Ball_021_1')   # the hard class
shap.summary_plot(shap_values[class_idx], Xte_s,
                  feature_names=X.columns.tolist(), show=True)

In [ ]:
import numpy as np
print(type(shap_values), np.array(shap_values).shape)  # confirm the shape first

In [ ]:
class_idx = list(le.classes_).index('Ball_021_1')
# new SHAP: shape is (samples, features, classes) -> slice last axis
shap.summary_plot(shap_values[:, :, class_idx], Xte_s,
                  feature_names=X.columns.tolist(), show=True)

In [ ]:
import matplotlib.pyplot as plt

# Global bar plot, saved at 300 DPI
shap.summary_plot(shap_values, Xte_s, feature_names=X.columns.tolist(),
                  plot_type="bar", show=False)
plt.tight_layout()
plt.savefig('/kaggle/working/shap_global_bar.png', dpi=300, bbox_inches='tight')
plt.close()

# Per-class beeswarm, saved at 300 DPI
shap.summary_plot(shap_values[:, :, class_idx], Xte_s,
                  feature_names=X.columns.tolist(), show=False)
plt.tight_layout()
plt.savefig('/kaggle/working/shap_ball021_beeswarm.png', dpi=300, bbox_inches='tight')
plt.close()

print("saved")

In [ ]:
import numpy as np

# Predicted class probabilities
proba = clf.predict_proba(Xte_s)
confidence = proba.max(axis=1)          # how confident on each sample
pred = proba.argmax(axis=1)
correct = (pred == yte)

# Are wrong predictions less confident than correct ones?
print("Mean confidence when CORRECT:", confidence[correct].mean())
print("Mean confidence when WRONG:  ", confidence[~correct].mean())

In [ ]:
import matplotlib.pyplot as plt

thresholds = np.linspace(0.3, 1.0, 30)
accs, coverage = [], []
for t in thresholds:
    keep = confidence >= t
    if keep.sum() > 0:
        accs.append((pred[keep] == yte[keep]).mean())
        coverage.append(keep.mean())

plt.figure()
plt.plot(coverage, accs, marker='o')
plt.xlabel('Coverage (fraction of samples predicted)')
plt.ylabel('Accuracy on accepted samples')
plt.title('Accuracy–Coverage trade-off (rejection curve)')
plt.grid(True)
plt.savefig('/kaggle/working/rejection_curve.png', dpi=300, bbox_inches='tight')
plt.savefig('/kaggle/working/rejection_curve.pdf', bbox_inches='tight')
plt.show()

In [ ]:
import shutil, os
os.makedirs('/kaggle/working/figures', exist_ok=True)
for f in os.listdir('/kaggle/working'):
    if f.endswith(('.png', '.pdf')):
        shutil.copy(f'/kaggle/working/{f}', f'/kaggle/working/figures/{f}')
shutil.make_archive('/kaggle/working/all_figures', 'zip', '/kaggle/working/figures')
print("Created all_figures.zip")

In [ ]:
with open('/kaggle/working/results_log.txt', 'a') as f:
    f.write("Mean confidence CORRECT: 0.9897\n")
    f.write("Mean confidence WRONG:   0.8843\n")
    f.write("Gap indicates meaningful predictive uncertainty.\n")

In [ ]:
import shutil, os

src = '/kaggle/working'
dst = '/kaggle/working/figures'
os.makedirs(dst, exist_ok=True)

count = 0
for f in os.listdir(src):
    full = os.path.join(src, f)
    if os.path.isfile(full) and f.endswith(('.png', '.pdf')):
        shutil.copy(full, os.path.join(dst, f))
        count += 1

print(f"Copied {count} figure files")

if count > 0:
    shutil.make_archive('/kaggle/working/all_figures', 'zip', dst)
    print("Created all_figures.zip")
else:
    print("No .png or .pdf files found in /kaggle/working yet — save some figures first")

In [ ]:
import os
print(os.listdir('/kaggle/working'))

In [ ]:
import numpy as np

data = np.load('/kaggle/input/datasets/brjapon/cwru-bearing-datasets/CWRU_48k_load_1_CNN_data.npz')
print(data.files)                    # what arrays are inside
for k in data.files:
    print(k, data[k].shape)

In [ ]:
import numpy as np
print("labels:", np.unique(data['labels']))   # expect 0-9
print("data range:", data['data'].min(), data['data'].max())

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

X = data['data'].astype('float32')
y = data['labels'].astype('int')

# normalize to roughly [0,1] or standardize
X = (X - X.mean()) / (X.std() + 1e-8)
X = X[..., np.newaxis]          # add channel dim -> (4600, 32, 32, 1)

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

model = models.Sequential([
    layers.Input((32, 32, 1)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),                 # dropout also enables MC-dropout uncertainty later
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax'),
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

history = model.fit(Xtr, ytr, validation_split=0.2,
                    epochs=30, batch_size=64, verbose=1)

In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

X = data['data'].astype('float32')

le_cnn = LabelEncoder()
y = le_cnn.fit_transform(data['labels'])   # strings -> 0..9
print("classes:", le_cnn.classes_)         # note the order for later
print("num classes:", len(le_cnn.classes_))

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

# --- Load and prep ---
data = np.load('/kaggle/input/datasets/brjapon/cwru-bearing-datasets/CWRU_48k_load_1_CNN_data.npz')

X = data['data'].astype('float32')
le_cnn = LabelEncoder()
y = le_cnn.fit_transform(data['labels'])        # strings -> 0..9

X = (X - X.mean()) / (X.std() + 1e-8)           # standardize
X = X[..., np.newaxis]                           # (4600, 32, 32, 1)

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

# --- Model ---
model = models.Sequential([
    layers.Input((32, 32, 1)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),                          # enables MC-dropout later
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax'),
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

# --- Train ---
history = model.fit(Xtr, ytr, validation_split=0.2,
                    epochs=30, batch_size=64, verbose=1)

# --- Evaluate ---
pred = model.predict(Xte).argmax(axis=1)
print(classification_report(yte, pred, target_names=le_cnn.classes_))

# --- Save training curve ---
plt.figure()
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.xlabel('epoch'); plt.ylabel('accuracy'); plt.legend(); plt.grid(True)
plt.title('CNN training accuracy')
plt.savefig('/kaggle/working/cnn_training_curve.png', dpi=300, bbox_inches='tight')
plt.savefig('/kaggle/working/cnn_training_curve.pdf', bbox_inches='tight')
plt.show()

In [ ]:
import shutil, os
dst = '/kaggle/working/figures'
os.makedirs(dst, exist_ok=True)
for f in os.listdir('/kaggle/working'):
    if f.endswith(('.png', '.pdf')):
        shutil.copy(f'/kaggle/working/{f}', f'{dst}/{f}')
shutil.make_archive('/kaggle/working/all_figures', 'zip', dst)
print("re-zipped all figures")

In [ ]:
import numpy as np

# Enable dropout at inference by calling model with training=True, many times
T = 50
preds = np.stack([model(Xte, training=True).numpy() for _ in range(T)])  # (T, N, 10)
mean_pred = preds.mean(axis=0)
pred_class = mean_pred.argmax(axis=1)
uncertainty = preds.std(axis=0).mean(axis=1)   # spread across passes = uncertainty

correct = (pred_class == yte)
print("Mean uncertainty when CORRECT:", uncertainty[correct].mean())
print("Mean uncertainty when WRONG:  ", uncertainty[~correct].mean())

In [ ]:
with open('/kaggle/working/results_log.txt', 'a') as f:
    f.write("\nCNN accuracy: 0.97\n")
    f.write("CNN MC-dropout uncertainty CORRECT: 0.0073\n")
    f.write("CNN MC-dropout uncertainty WRONG:   0.0390 (5x higher)\n")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# reload fresh so this cell is self-contained
df = pd.read_csv('/kaggle/input/datasets/brjapon/cwru-bearing-datasets/feature_time_48k_2048_load_1.csv')

Xall = df.drop(columns=['fault'])
yall = LabelEncoder().fit_transform(df['fault'])

results = []
for feats, name in [(Xall.columns.tolist(), 'All 9 features'),
                    (['sd', 'mean', 'kurtosis', 'form'], 'Top-4 SHAP features'),
                    (['max', 'min'], 'Weak features only')]:
    Xs = Xall[feats]
    Xtr, Xte, ytr, yte = train_test_split(Xs, yall, test_size=0.2,
                                          stratify=yall, random_state=42)
    sc = StandardScaler().fit(Xtr)
    clf = XGBClassifier(random_state=42, eval_metric='mlogloss')
    clf.fit(sc.transform(Xtr), ytr)
    acc = accuracy_score(yte, clf.predict(sc.transform(Xte)))
    results.append((name, acc))
    print(f"{name}: {acc:.3f}")

In [ ]:
with open('/kaggle/working/results_log.txt', 'a') as f:
    f.write("\nABLATION (XGBoost):\n")
    f.write("All 9 features:      0.943\n")
    f.write("Top-4 SHAP features: 0.946\n")
    f.write("Weak features only:  0.663\n")

In [ ]:
with open('/kaggle/working/results_log.txt', 'a') as f:
    f.write("\nABLATION (XGBoost):\n")
    f.write("All 9 features:      0.943\n")
    f.write("Top-4 SHAP features: 0.946\n")
    f.write("Weak features only:  0.663\n")
print("logged")

In [ ]:
import os

pb_root = '/kaggle/input/datasets/dippatel/paderborn-db'  # adjust if the path differs
count = 0
for dirname, _, filenames in os.walk('/kaggle/input'):
    if 'paderborn' in dirname.lower() or 'paderborn' in ''.join(filenames).lower():
        for f in filenames[:15]:
            print(os.path.join(dirname, f))
            count += 1
    if count > 40:
        break

In [ ]:
import scipy.io as sio
import numpy as np

f = '/kaggle/input/datasets/dippatel03/paderborn-db/K004/N15_M07_F10_K004_2.mat'
mat = sio.loadmat(f)
print("top keys:", [k for k in mat.keys() if not k.startswith('__')])

# Paderborn nests everything under a struct named like the file. Explore it:
key = [k for k in mat.keys() if not k.startswith('__')][0]
print("struct dtype names:", mat[key].dtype.names if mat[key].dtype.names else "flat array")

In [ ]:
import scipy.io as sio
import numpy as np

f = '/kaggle/input/datasets/dippatel03/paderborn-db/K004/N15_M07_F10_K004_2.mat'
mat = sio.loadmat(f)
key = 'N15_M07_F10_K004_2'
struct = mat[key][0, 0]

Y = struct['Y']            # the signal container
print("Y shape:", Y.shape)
print("Y[0,0] field names:", Y[0, 0].dtype.names if Y[0,0].dtype.names else "none")

# Paderborn Y is a list of channels; each has a 'Name' and 'Data'
for i in range(Y.shape[1]):
    name = Y[0, i]['Name'][0] if 'Name' in Y[0,i].dtype.names else '?'
    data = Y[0, i]['Data']
    print(i, "->", name, "| data shape:", np.array(data).shape)

In [ ]:
import os
root = '/kaggle/input/datasets/dippatel03/paderborn-db'
print(sorted([d for d in os.listdir(root) if os.path.isdir(f'{root}/{d}')]))

In [ ]:
import scipy.io as sio
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
import os, glob

ROOT = '/kaggle/input/datasets/dippatel03/paderborn-db'
COND = 'N15_M07_F10'          # fixed operating condition
WIN  = 2048                   # same window as CWRU features

# Balanced bearing selection: 2 codes per class
SELECTION = {
    'Normal': ['K001', 'K002'],
    'OR':     ['KA04', 'KA07'],   # real (fatigue) outer-race damage
    'IR':     ['KI04', 'KI07'],   # real inner-race damage
}

def extract_features(sig):
    """Same 9 time-domain features as the CWRU CSV."""
    mx, mn = sig.max(), sig.min()
    mean = sig.mean()
    sd = sig.std()
    rms = np.sqrt(np.mean(sig**2))
    sk = skew(sig)
    kt = kurtosis(sig)                       # Fisher (excess) kurtosis
    crest = mx / (rms + 1e-12)
    form = rms / (abs(mean) + 1e-12)
    return [mx, mn, mean, sd, rms, sk, kt, crest, form]

COLS = ['max','min','mean','sd','rms','skewness','kurtosis','crest','form']

rows, labels = [], []
for label, codes in SELECTION.items():
    for code in codes:
        files = glob.glob(f'{ROOT}/{code}/{COND}_{code}_*.mat')
        for f in files:
            key = os.path.splitext(os.path.basename(f))[0]
            try:
                mat = sio.loadmat(f)
                sig = np.array(mat[key][0,0]['Y'][0,6]['Data']).squeeze()  # vibration_1
            except Exception as e:
                continue
            # segment into non-overlapping windows
            n = len(sig) // WIN
            for i in range(n):
                seg = sig[i*WIN:(i+1)*WIN]
                rows.append(extract_features(seg))
                labels.append(label)

pb = pd.DataFrame(rows, columns=COLS)
pb['fault'] = labels
print(pb.shape)
print(pb['fault'].value_counts())
pb.head()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# reload CWRU 3-class subset to match Paderborn's label space
cwru = pd.read_csv('/kaggle/input/datasets/brjapon/cwru-bearing-datasets/feature_time_48k_2048_load_1.csv')

# map CWRU's fine labels to 3 classes; drop Ball
def to3(x):
    if x.startswith('IR'): return 'IR'
    if x.startswith('OR'): return 'OR'
    if x.startswith('Normal'): return 'Normal'
    return None
cwru['fault3'] = cwru['fault'].map(to3)
cwru3 = cwru[cwru['fault3'].notna()].copy()
print("CWRU 3-class counts:\n", cwru3['fault3'].value_counts())

# compare per-feature means, Normal class only
feats = ['max','min','mean','sd','rms','skewness','kurtosis','crest','form']
print("\nFeature means — CWRU Normal vs Paderborn Normal:")
comp = pd.DataFrame({
    'CWRU':   cwru3[cwru3['fault3']=='Normal'][feats].mean(),
    'Paderborn': pb[pb['fault']=='Normal'][feats].mean()
})
print(comp)

In [ ]:
import scipy.io as sio, numpy as np, pandas as pd, glob, os, re
from scipy.stats import skew, kurtosis

RAW = '/kaggle/input/datasets/brjapon/cwru-bearing-datasets/raw'
WIN = 2048

def extract_features(sig):
    mx, mn = sig.max(), sig.min()
    mean = sig.mean(); sd = sig.std()
    rms = np.sqrt(np.mean(sig**2))
    return [mx, mn, mean, sd, rms, skew(sig), kurtosis(sig),
            mx/(rms+1e-12), rms/(abs(mean)+1e-12)]

COLS = ['max','min','mean','sd','rms','skewness','kurtosis','crest','form']

def label3(fname):
    b = os.path.basename(fname)
    if b.startswith('IR'): return 'IR'
    if b.startswith('OR'): return 'OR'
    if b.startswith('Time_Normal') or b.startswith('Normal'): return 'Normal'
    return None   # skip Ball

rows, labels = [], []
for f in glob.glob(f'{RAW}/*.mat'):
    lab = label3(f)
    if lab is None:
        continue
    mat = sio.loadmat(f)
    # CWRU signal key ends in _DE_time (drive-end accelerometer)
    de_keys = [k for k in mat if k.endswith('DE_time')]
    if not de_keys:
        continue
    sig = mat[de_keys[0]].squeeze()
    n = len(sig) // WIN
    for i in range(n):
        rows.append(extract_features(sig[i*WIN:(i+1)*WIN]))
        labels.append(lab)

cwru_re = pd.DataFrame(rows, columns=COLS)
cwru_re['fault'] = labels
print(cwru_re.shape)
print(cwru_re['fault'].value_counts())

# now compare Normal means again — same pipeline both sides
comp = pd.DataFrame({
    'CWRU_re': cwru_re[cwru_re['fault']=='Normal'][COLS].mean(),
    'Paderborn': pb[pb['fault']=='Normal'][COLS].mean()
})
print(comp)

In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

COLS = ['max','min','mean','sd','rms','skewness','kurtosis','crest','form']

le = LabelEncoder().fit(['IR','Normal','OR'])
Xc, yc = cwru_re[COLS].values, le.transform(cwru_re['fault'])
Xp, yp = pb[COLS].values,      le.transform(pb['fault'])

# --- Experiment 1: cold transfer, RAW features ---
clf = XGBClassifier(random_state=42, eval_metric='mlogloss')
clf.fit(Xc, yc)
acc_raw = accuracy_score(yp, clf.predict(Xp))
print("COLD TRANSFER (raw features): %.3f" % acc_raw)
print(confusion_matrix(yp, clf.predict(Xp)))

In [ ]:
# --- Experiment 2: cold transfer, per-dataset standardized ---
sc_c = StandardScaler().fit(Xc)
sc_p = StandardScaler().fit(Xp)
clf2 = XGBClassifier(random_state=42, eval_metric='mlogloss')
clf2.fit(sc_c.transform(Xc), yc)
acc_std = accuracy_score(yp, clf2.predict(sc_p.transform(Xp)))
print("COLD TRANSFER (per-dataset standardized): %.3f" % acc_std)
print(confusion_matrix(yp, clf2.predict(sc_p.transform(Xp))))

In [ ]:
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

sc_c = StandardScaler().fit(Xc)
sc_p = StandardScaler().fit(Xp)
clf2 = XGBClassifier(random_state=42, eval_metric='mlogloss')
clf2.fit(sc_c.transform(Xc), yc)
pred2 = clf2.predict(sc_p.transform(Xp))
acc_std = accuracy_score(yp, pred2)
print("COLD TRANSFER (per-dataset standardized): %.3f" % acc_std)
print(confusion_matrix(yp, pred2))

In [ ]:
with open('/kaggle/working/results_log.txt', 'a') as f:
    f.write("\nCROSS-TRANSFER CWRU->Paderborn:\n")
    f.write("Raw features: 0.325 (Normal class fully collapses to OR)\n")

In [ ]:
with open('/kaggle/working/results_log.txt', 'a') as f:
    f.write("Per-dataset standardized: 0.337 (no improvement - gap is structural)\n")

In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from scipy.linalg import sqrtm

COLS = ['max','min','mean','sd','rms','skewness','kurtosis','crest','form']
le = LabelEncoder().fit(['IR','Normal','OR'])
Xc, yc = cwru_re[COLS].values.astype(float), le.transform(cwru_re['fault'])
Xp, yp = pb[COLS].values.astype(float),      le.transform(pb['fault'])

def coral(Xs, Xt):
    """Align source features Xs to target Xt via covariance whitening+recoloring."""
    Xs = (Xs - Xs.mean(0)) / (Xs.std(0) + 1e-8)
    Xt = (Xt - Xt.mean(0)) / (Xt.std(0) + 1e-8)
    Cs = np.cov(Xs, rowvar=False) + np.eye(Xs.shape[1])
    Ct = np.cov(Xt, rowvar=False) + np.eye(Xt.shape[1])
    Xs_white = Xs @ np.real(sqrtm(np.linalg.inv(Cs)))     # whiten source
    Xs_align = Xs_white @ np.real(sqrtm(Ct))              # recolor to target
    Xt_std = Xt
    return Xs_align, Xt_std

Xc_aligned, Xp_std = coral(Xc, Xp)

clf = XGBClassifier(random_state=42, eval_metric='mlogloss')
clf.fit(Xc_aligned, yc)
pred = clf.predict(Xp_std)
acc_coral = accuracy_score(yp, pred)
print("COLD TRANSFER (CORAL domain adaptation): %.3f" % acc_coral)
print(confusion_matrix(yp, pred))

In [ ]:
import scipy.io as sio, numpy as np, glob, os
from scipy.stats import zscore

WIN = 2048   # window length (samples)

# ---------- CWRU raw windows (3-class, from raw/ .mat) ----------
RAW = '/kaggle/input/datasets/brjapon/cwru-bearing-datasets/raw'
def cwru_label(f):
    b = os.path.basename(f)
    if b.startswith('IR'): return 'IR'
    if b.startswith('OR'): return 'OR'
    if b.startswith('Time_Normal') or b.startswith('Normal'): return 'Normal'
    return None

Xc_raw, yc_raw = [], []
for f in glob.glob(f'{RAW}/*.mat'):
    lab = cwru_label(f)
    if lab is None: continue
    mat = sio.loadmat(f)
    de = [k for k in mat if k.endswith('DE_time')]
    if not de: continue
    sig = mat[de[0]].squeeze().astype(float)
    for i in range(len(sig)//WIN):
        seg = sig[i*WIN:(i+1)*WIN]
        Xc_raw.append((seg - seg.mean())/(seg.std()+1e-8))  # per-window z-norm
        yc_raw.append(lab)

# ---------- Paderborn raw windows (3-class, one condition) ----------
ROOT = '/kaggle/input/datasets/dippatel03/paderborn-db'
COND = 'N15_M07_F10'
SEL = {'Normal':['K001','K002'], 'OR':['KA04','KA07'], 'IR':['KI04','KI07']}

Xp_raw, yp_raw = [], []
for lab, codes in SEL.items():
    for code in codes:
        for f in glob.glob(f'{ROOT}/{code}/{COND}_{code}_*.mat'):
            key = os.path.splitext(os.path.basename(f))[0]
            try:
                mat = sio.loadmat(f)
                sig = np.array(mat[key][0,0]['Y'][0,6]['Data']).squeeze().astype(float)
            except Exception:
                continue
            for i in range(len(sig)//WIN):
                seg = sig[i*WIN:(i+1)*WIN]
                Xp_raw.append((seg - seg.mean())/(seg.std()+1e-8))
                yp_raw.append(lab)

Xc_raw = np.array(Xc_raw); Xp_raw = np.array(Xp_raw)
print("CWRU raw:", Xc_raw.shape, "| Paderborn raw:", Xp_raw.shape)
import collections
print("CWRU:", collections.Counter(yc_raw))
print("PB:  ", collections.Counter(yp_raw))

In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder().fit(['IR','Normal','OR'])
yc = le.transform(yc_raw)
yp = le.transform(yp_raw)

# --- subsample Paderborn to ~match CWRU size, keep classes balanced ---
rng = np.random.default_rng(42)
target_per_class = 700   # close to CWRU's IR/OR count
idx_keep = []
for c in range(3):
    ci = np.where(yp == c)[0]
    idx_keep.extend(rng.choice(ci, size=min(target_per_class, len(ci)), replace=False))
idx_keep = np.array(idx_keep); rng.shuffle(idx_keep)

Xp_bal = Xp_raw[idx_keep]
yp_bal = yp[idx_keep]

print("CWRU:", Xc_raw.shape[0], "| Paderborn balanced:", Xp_bal.shape[0])
import collections
print("PB balanced classes:", collections.Counter(le.inverse_transform(yp_bal)))

# reshape for 1D-CNN: (N, 2048, 1)
Xc = Xc_raw[..., None].astype('float32')
Xp = Xp_bal[..., None].astype('float32')

# class weights for CWRU label classifier (Normal is scarce)
from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.arange(3), y=yc)
class_weight = {i: w for i, w in enumerate(cw)}
print("class weights:", class_weight)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np

# ---- Gradient Reversal Layer ----
@tf.custom_gradient
def grad_reverse(x, lamb):
    y = tf.identity(x)
    def grad(dy):
        return -lamb * dy, None
    return y, grad

class GRL(layers.Layer):
    def __init__(self, **kw): super().__init__(**kw); self.lamb = tf.Variable(1.0, trainable=False)
    def call(self, x): return grad_reverse(x, self.lamb)

# ---- Shared feature extractor ----
def build_dann():
    inp = layers.Input((2048, 1))
    x = layers.Conv1D(32, 16, strides=2, activation='relu', padding='same')(inp)
    x = layers.MaxPool1D(2)(x)
    x = layers.Conv1D(64, 8, strides=2, activation='relu', padding='same')(x)
    x = layers.MaxPool1D(2)(x)
    x = layers.Conv1D(128, 4, activation='relu', padding='same')(x)
    x = layers.GlobalAveragePooling1D()(x)
    feat = layers.Dense(128, activation='relu', name='features')(x)

    # label head (3 classes)
    lp = layers.Dropout(0.3)(feat)
    label_out = layers.Dense(3, activation='softmax', name='label')(lp)

    # domain head (2 domains) via gradient reversal
    grl = GRL()(feat)
    dp = layers.Dense(64, activation='relu')(grl)
    domain_out = layers.Dense(2, activation='softmax', name='domain')(dp)

    return Model(inp, [label_out, domain_out]), Model(inp, label_out)

dann, label_model = build_dann()
dann.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss={'label':'sparse_categorical_crossentropy',
          'domain':'sparse_categorical_crossentropy'},
    loss_weights={'label':1.0, 'domain':1.0},
    metrics={'label':'accuracy'})
dann.summary()

In [ ]:
import numpy as np
import tensorflow as tf

# domain labels: CWRU=0, Paderborn=1
Xc_t = tf.constant(Xc); yc_t = tf.constant(yc, dtype=tf.int32)
Xp_t = tf.constant(Xp); yp_t = tf.constant(yp_bal, dtype=tf.int32)

n_c, n_p = len(Xc), len(Xp)
BATCH = 64
EPOCHS = 60
steps = max(n_c, n_p) // BATCH

opt = tf.keras.optimizers.Adam(1e-3)
cce = tf.keras.losses.SparseCategoricalCrossentropy()
cw_vec = tf.constant([class_weight[0], class_weight[1], class_weight[2]], dtype=tf.float32)

def sample(X, y, k):
    idx = np.random.randint(0, len(X), k)
    return tf.gather(X, idx), tf.gather(y, idx)

@tf.function
def train_step(xc, yc_b, xp_b, lamb):
    dann.get_layer(index=[i for i,l in enumerate(dann.layers) if l.name=='grl' or isinstance(l,GRL)][0]).lamb.assign(lamb) if False else None
    with tf.GradientTape() as tape:
        # label loss on CWRU (weighted)
        lbl_pred, dom_pred_c = dann(xc, training=True)
        w = tf.gather(cw_vec, yc_b)
        label_loss = tf.reduce_mean(w * tf.keras.losses.sparse_categorical_crossentropy(yc_b, lbl_pred))
        # domain loss on both (CWRU=0, PB=1)
        _, dom_pred_p = dann(xp_b, training=True)
        dom_c = tf.zeros(tf.shape(xc)[0], dtype=tf.int32)
        dom_p = tf.ones(tf.shape(xp_b)[0], dtype=tf.int32)
        domain_loss = (cce(dom_c, dom_pred_c) + cce(dom_p, dom_pred_p)) / 2.0
        total = label_loss + domain_loss
    grads = tape.gradient(total, dann.trainable_variables)
    opt.apply_gradients(zip(grads, dann.trainable_variables))
    return label_loss, domain_loss

# find the GRL layer to set lambda each epoch
grl_layer = [l for l in dann.layers if isinstance(l, GRL)][0]

from sklearn.metrics import accuracy_score
for ep in range(EPOCHS):
    p = ep / EPOCHS
    lamb = 2.0/(1.0+np.exp(-10*p)) - 1.0          # ramp 0->1 (standard DANN schedule)
    grl_layer.lamb.assign(float(lamb))
    ll = dl = 0.0
    for _ in range(steps):
        xc_b, yc_b = sample(Xc_t, yc_t, BATCH)
        xp_b, _    = sample(Xp_t, yp_t, BATCH)
        a, b = train_step(xc_b, yc_b, xp_b, float(lamb))
        ll += float(a); dl += float(b)
    if ep % 5 == 0 or ep == EPOCHS-1:
        pb_pred = label_model.predict(Xp, verbose=0).argmax(1)
        acc = accuracy_score(yp_bal, pb_pred)
        print(f"ep {ep:2d} | lambda {lamb:.2f} | label {ll/steps:.3f} | domain {dl/steps:.3f} | PB acc {acc:.3f}")

In [ ]:
opt = tf.keras.optimizers.Adam(3e-4)          # lower LR
grl_layer = [l for l in dann.layers if isinstance(l, GRL)][0]

from sklearn.metrics import accuracy_score
LAMBDA_MAX = 0.3                               # much softer adversary
EPOCHS = 80

for ep in range(EPOCHS):
    p = ep / EPOCHS
    lamb = LAMBDA_MAX * (2.0/(1.0+np.exp(-5*p)) - 1.0)   # slow ramp, low ceiling
    grl_layer.lamb.assign(float(lamb))
    ll = dl = 0.0
    for _ in range(steps):
        xc_b, yc_b = sample(Xc_t, yc_t, BATCH)
        xp_b, _    = sample(Xp_t, yp_t, BATCH)
        a, b = train_step(xc_b, yc_b, xp_b, float(lamb))
        ll += float(a); dl += float(b)
    if ep % 5 == 0 or ep == EPOCHS-1:
        pb_pred = label_model.predict(Xp, verbose=0).argmax(1)
        acc = accuracy_score(yp_bal, pb_pred)
        print(f"ep {ep:2d} | lambda {lamb:.3f} | label {ll/steps:.3f} | domain {dl/steps:.3f} | PB acc {acc:.3f}")

In [ ]:
import tensorflow as tf, numpy as np
from sklearn.metrics import accuracy_score

# 1) rebuild model fresh (resets weights)
dann, label_model = build_dann()
grl_layer = [l for l in dann.layers if isinstance(l, GRL)][0]

# 2) new optimizer
opt = tf.keras.optimizers.Adam(3e-4)
cw_vec = tf.constant([class_weight[0], class_weight[1], class_weight[2]], dtype=tf.float32)

# 3) redefine train_step AFTER optimizer exists (fresh trace)
@tf.function
def train_step(xc, yc_b, xp_b):
    with tf.GradientTape() as tape:
        lbl_pred, dom_pred_c = dann(xc, training=True)
        w = tf.gather(cw_vec, yc_b)
        label_loss = tf.reduce_mean(
            w * tf.keras.losses.sparse_categorical_crossentropy(yc_b, lbl_pred))
        _, dom_pred_p = dann(xp_b, training=True)
        dom_c = tf.zeros(tf.shape(xc)[0], dtype=tf.int32)
        dom_p = tf.ones(tf.shape(xp_b)[0], dtype=tf.int32)
        domain_loss = (
            tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(dom_c, dom_pred_c)) +
            tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(dom_p, dom_pred_p))
        ) / 2.0
        total = label_loss + domain_loss
    grads = tape.gradient(total, dann.trainable_variables)
    opt.apply_gradients(zip(grads, dann.trainable_variables))
    return label_loss, domain_loss

# 4) data tensors
Xc_t = tf.constant(Xc); yc_t = tf.constant(yc, dtype=tf.int32)
Xp_t = tf.constant(Xp); yp_t = tf.constant(yp_bal, dtype=tf.int32)
def sample(X, y, k):
    idx = np.random.randint(0, len(X), k)
    return tf.gather(X, idx), tf.gather(y, idx)

BATCH = 64; EPOCHS = 80; LAMBDA_MAX = 0.3
steps = max(len(Xc), len(Xp)) // BATCH

# 5) train
for ep in range(EPOCHS):
    p = ep / EPOCHS
    lamb = LAMBDA_MAX * (2.0/(1.0+np.exp(-5*p)) - 1.0)
    grl_layer.lamb.assign(float(lamb))
    ll = dl = 0.0
    for _ in range(steps):
        xc_b, yc_b = sample(Xc_t, yc_t, BATCH)
        xp_b, _    = sample(Xp_t, yp_t, BATCH)
        a, b = train_step(xc_b, yc_b, xp_b)
        ll += float(a); dl += float(b)
    if ep % 5 == 0 or ep == EPOCHS-1:
        acc = accuracy_score(yp_bal, label_model.predict(Xp, verbose=0).argmax(1))
        print(f"ep {ep:2d} | lambda {lamb:.3f} | label {ll/steps:.3f} | domain {dl/steps:.3f} | PB acc {acc:.3f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from tensorflow.keras import Model

# --- extract learned features from the DANN feature extractor ---
feat_model = Model(dann.input, dann.get_layer('features').output)

Fc = feat_model.predict(Xc, verbose=0)   # CWRU features  (n_c, 128)
Fp = feat_model.predict(Xp, verbose=0)   # Paderborn features (n_p, 128)

F_all = np.vstack([Fc, Fp])
domain_all = np.array([0]*len(Fc) + [1]*len(Fp))          # 0=CWRU, 1=PB
label_all  = np.concatenate([yc, yp_bal])                 # fault class 0/1/2

# --- t-SNE to 2D ---
emb = TSNE(n_components=2, perplexity=30, random_state=42,
           init='pca').fit_transform(F_all)

# --- Figure 1: colored by DATASET (the domain-gap story) ---
plt.figure(figsize=(6,5))
for d, name, c in [(0,'CWRU','#1f77b4'), (1,'Paderborn','#d62728')]:
    m = domain_all == d
    plt.scatter(emb[m,0], emb[m,1], s=8, alpha=0.5, label=name, color=c)
plt.legend(); plt.title('Learned features by dataset (after DANN)')
plt.xlabel('t-SNE 1'); plt.ylabel('t-SNE 2'); plt.tight_layout()
plt.savefig('/kaggle/working/tsne_by_dataset.png', dpi=300, bbox_inches='tight')
plt.savefig('/kaggle/working/tsne_by_dataset.pdf', bbox_inches='tight')
plt.show()

# --- Figure 2: colored by FAULT CLASS (do classes align across domains?) ---
plt.figure(figsize=(6,5))
names = ['IR','Normal','OR']
for lab, c in zip([0,1,2], ['#2ca02c','#ff7f0e','#9467bd']):
    m = label_all == lab
    plt.scatter(emb[m,0], emb[m,1], s=8, alpha=0.5, label=names[lab], color=c)
plt.legend(); plt.title('Learned features by fault class')
plt.xlabel('t-SNE 1'); plt.ylabel('t-SNE 2'); plt.tight_layout()
plt.savefig('/kaggle/working/tsne_by_class.png', dpi=300, bbox_inches='tight')
plt.savefig('/kaggle/working/tsne_by_class.pdf', bbox_inches='tight')
plt.show()

In [ ]:
import shutil, os
dst = '/kaggle/working/figures'
os.makedirs(dst, exist_ok=True)
for f in os.listdir('/kaggle/working'):
    if f.endswith(('.png', '.pdf')):
        shutil.copy(f'/kaggle/working/{f}', f'{dst}/{f}')
shutil.make_archive('/kaggle/working/all_figures', 'zip', dst)
print("done — download all_figures.zip from the Output panel")

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import wasserstein_distance
import matplotlib.pyplot as plt

COLS = ['max','min','mean','sd','rms','skewness','kurtosis','crest','form']

# --- 1) SHAP importance per feature (CWRU XGBoost model) ---
# Retrain the 9-feature XGBoost on CWRU (3-class) to get clean SHAP importances
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import shap

le3 = LabelEncoder().fit(['IR','Normal','OR'])
Xc9 = cwru_re[COLS].values
yc9 = le3.transform(cwru_re['fault'])

xgb = XGBClassifier(random_state=42, eval_metric='mlogloss').fit(Xc9, yc9)
expl = shap.TreeExplainer(xgb)
sv = expl.shap_values(Xc9)                      # list or array
sv = np.array(sv)
# mean |SHAP| per feature, averaged over classes & samples
if sv.ndim == 3:      # (classes, samples, features) or (samples, features, classes)
    imp = np.abs(sv).mean(axis=tuple(range(sv.ndim-1))) if sv.shape[-1]==len(COLS) else np.abs(sv).mean(axis=(0,1))
else:
    imp = np.abs(sv).mean(axis=0)
imp = np.asarray(imp).ravel()[:len(COLS)]

# --- 2) Distribution divergence per feature: CWRU vs Paderborn ---
# standardize each dataset per-feature first so divergence isn't just scale
def zt(a): return (a - a.mean(0)) / (a.std(0) + 1e-8)
Cz = zt(cwru_re[COLS].values)
Pz = zt(pb[COLS].values)
div = [wasserstein_distance(Cz[:,i], Pz[:,i]) for i in range(len(COLS))]
div = np.array(div)

# --- 3) Combine and correlate ---
tab = pd.DataFrame({'feature':COLS,
                    'shap_importance':imp/imp.sum(),
                    'domain_divergence':div}).sort_values('shap_importance', ascending=False)
from scipy.stats import pearsonr, spearmanr
r_p, p_p = pearsonr(tab['shap_importance'], tab['domain_divergence'])
r_s, p_s = spearmanr(tab['shap_importance'], tab['domain_divergence'])
print(tab.to_string(index=False))
print(f"\nPearson r = {r_p:.3f} (p={p_p:.3f}) | Spearman r = {r_s:.3f} (p={p_s:.3f})")

# --- 4) Scatter figure ---
plt.figure(figsize=(6,5))
plt.scatter(tab['shap_importance'], tab['domain_divergence'], s=60, color='#d62728')
for _, row in tab.iterrows():
    plt.annotate(row['feature'],
                 (row['shap_importance'], row['domain_divergence']),
                 xytext=(4,4), textcoords='offset points', fontsize=9)
plt.xlabel('SHAP importance (normalized)')
plt.ylabel('CWRU↔Paderborn divergence (Wasserstein)')
plt.title('Do the features the model relies on transfer?')
plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('/kaggle/working/shap_vs_divergence.png', dpi=300, bbox_inches='tight')
plt.savefig('/kaggle/working/shap_vs_divergence.pdf', bbox_inches='tight')
plt.show()

In [ ]:
print("sv shape:", np.array(sv).shape)
print("imp shape:", np.asarray(imp).shape, "| len(COLS):", len(COLS))
print("div shape:", np.array(div).shape)

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import wasserstein_distance, pearsonr, spearmanr
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import shap

COLS = ['max','min','mean','sd','rms','skewness','kurtosis','crest','form']

le3 = LabelEncoder().fit(['IR','Normal','OR'])
Xc9 = cwru_re[COLS].values
yc9 = le3.transform(cwru_re['fault'])

xgb = XGBClassifier(random_state=42, eval_metric='mlogloss').fit(Xc9, yc9)
expl = shap.TreeExplainer(xgb)
sv = expl.shap_values(Xc9)

# --- robust: reduce SHAP output to exactly one importance per feature ---
sv = np.array(sv)
# collapse every axis that ISN'T the length-9 feature axis
abs_sv = np.abs(sv)
# find the axis whose size == number of features
feat_axis = [ax for ax, s in enumerate(abs_sv.shape) if s == len(COLS)]
if not feat_axis:
    raise ValueError(f"No axis of length {len(COLS)} in SHAP shape {abs_sv.shape}")
fax = feat_axis[-1]
# mean over all other axes
other = tuple(ax for ax in range(abs_sv.ndim) if ax != fax)
imp = abs_sv.mean(axis=other)
imp = np.asarray(imp).ravel()
print("imp length:", len(imp))          # must be 9

# --- distribution divergence per feature (standardized) ---
def zt(a): return (a - a.mean(0)) / (a.std(0) + 1e-8)
Cz = zt(cwru_re[COLS].values)
Pz = zt(pb[COLS].values)
div = np.array([wasserstein_distance(Cz[:,i], Pz[:,i]) for i in range(len(COLS))])

assert len(imp) == len(div) == len(COLS), f"lengths: imp={len(imp)}, div={len(div)}, cols={len(COLS)}"

tab = pd.DataFrame({'feature':COLS,
                    'shap_importance':imp/imp.sum(),
                    'domain_divergence':div}).sort_values('shap_importance', ascending=False)
r_p,p_p = pearsonr(tab.shap_importance, tab.domain_divergence)
r_s,p_s = spearmanr(tab.shap_importance, tab.domain_divergence)
print(tab.to_string(index=False))
print(f"\nPearson r={r_p:.3f} (p={p_p:.3f}) | Spearman r={r_s:.3f} (p={p_s:.3f})")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6,5))
plt.scatter(tab.shap_importance, tab.domain_divergence, s=60, color='#d62728')
for _, r in tab.iterrows():
    plt.annotate(r.feature, (r.shap_importance, r.domain_divergence),
                 xytext=(4,4), textcoords='offset points', fontsize=9)
plt.xlabel('SHAP importance (normalized)')
plt.ylabel('CWRU↔Paderborn divergence (Wasserstein)')
plt.title(f'Importance vs. transferability (Pearson r={r_p:.2f}, n.s.)')
plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('/kaggle/working/shap_vs_divergence.png', dpi=300, bbox_inches='tight')
plt.savefig('/kaggle/working/shap_vs_divergence.pdf', bbox_inches='tight')
plt.show()

In [ ]:
import shutil, os
dst = '/kaggle/working/figures'
os.makedirs(dst, exist_ok=True)
for f in os.listdir('/kaggle/working'):
    if f.endswith(('.png', '.pdf')):
        shutil.copy(f'/kaggle/working/{f}', f'{dst}/{f}')
shutil.make_archive('/kaggle/working/all_figures', 'zip', dst)
print("done — download all_figures.zip from the Output panel")

In [ ]:
import pandas as pd

# reload CWRU pre-extracted features (the CSV)
df = pd.read_csv('/kaggle/input/datasets/brjapon/cwru-bearing-datasets/feature_time_48k_2048_load_1.csv')
print(df.shape)
print(df['fault'].value_counts())

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from xgboost import XGBClassifier

X = df.drop(columns=['fault']).values
y = LabelEncoder().fit_transform(df['fault'])

pipe = make_pipeline(StandardScaler(),
                     XGBClassifier(random_state=42, eval_metric='mlogloss'))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=skf, scoring='accuracy')

print("XGBoost 5-fold accuracies:", np.round(scores, 4))
print(f"Mean ± std: {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
with open('/kaggle/working/results_log.txt', 'a') as f:
    f.write("\nXGBoost 5-fold CV: 0.9609 +/- 0.0077\n")
    f.write("Folds: 0.950, 0.959, 0.970, 0.970, 0.957\n")

In [ ]:
import numpy as np, tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

data = np.load('/kaggle/input/datasets/brjapon/cwru-bearing-datasets/CWRU_48k_load_1_CNN_data.npz')
X = data['data'].astype('float32')
X = (X - X.mean())/(X.std()+1e-8)
X = X[..., None]
y = LabelEncoder().fit_transform(data['labels'])

def make_cnn():
    m = models.Sequential([
        layers.Input((32,32,1)),
        layers.Conv2D(32,3,activation='relu',padding='same'), layers.MaxPooling2D(),
        layers.Conv2D(64,3,activation='relu',padding='same'), layers.MaxPooling2D(),
        layers.Conv2D(128,3,activation='relu',padding='same'), layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3), layers.Dense(64,activation='relu'), layers.Dense(10,activation='softmax')])
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
accs = []
for i,(tr,te) in enumerate(skf.split(X,y)):
    m = make_cnn()
    m.fit(X[tr], y[tr], epochs=30, batch_size=64, verbose=0)
    acc = m.evaluate(X[te], y[te], verbose=0)[1]
    accs.append(acc); print(f"fold {i+1}: {acc:.4f}")
accs = np.array(accs)
print(f"CNN 5-fold: {accs.mean():.4f} +/- {accs.std():.4f}")

In [ ]:
import os
os.makedirs('/kaggle/working', exist_ok=True)
with open('/kaggle/working/results_log.txt', 'a') as f:
    f.write("XGBoost 5-fold CV: 0.9609 +/- 0.0077\n")
    f.write("CNN 5-fold CV: 0.9700 +/- 0.0067\n")
print("written OK")
print(os.path.exists('/kaggle/working/results_log.txt'))

In [ ]:
import os
count = 0
for dirname, _, filenames in os.walk('/kaggle/input'):
    if 'mfpt' in dirname.lower() or 'rolling' in dirname.lower():
        for f in filenames[:25]:
            print(os.path.join(dirname, f))
            count += 1
    if count > 30:
        break

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    if 'mfpt' in dirname.lower() and 'rolling' not in dirname.lower():
        for f in filenames[:30]:
            print(os.path.join(dirname, f))

In [ ]:
import scipy.io as sio
import numpy as np

f = '/kaggle/input/datasets/emperorpein/mfpt-fault-datasets/MFPT Fault Data Sets/1 - Three Baseline Conditions/baseline_1.mat'
mat = sio.loadmat(f)
print("keys:", [k for k in mat.keys() if not k.startswith('__')])

key = [k for k in mat.keys() if not k.startswith('__')][0]
print("struct fields:", mat[key].dtype.names)

# MFPT stores signal under 'bearing' struct -> 'gs' (acceleration) and 'sr' (sample rate)
b = mat[key][0,0]
for fld in mat[key].dtype.names:
    v = b[fld]
    print(fld, "->", np.array(v).shape, np.array(v).ravel()[:3] if np.array(v).size < 5 else "signal array")

In [ ]:
import scipy.io as sio, numpy as np, pandas as pd, glob, os
from scipy.stats import skew, kurtosis

BASE = '/kaggle/input/datasets/emperorpein/mfpt-fault-datasets/MFPT Fault Data Sets'
WIN = 2048

def extract_features(sig):
    mx, mn = sig.max(), sig.min()
    mean = sig.mean(); sd = sig.std()
    rms = np.sqrt(np.mean(sig**2))
    return [mx, mn, mean, sd, rms, skew(sig), kurtosis(sig),
            mx/(rms+1e-12), rms/(abs(mean)+1e-12)]

COLS = ['max','min','mean','sd','rms','skewness','kurtosis','crest','form']

# folders -> class labels
FOLDERS = {
    'Normal': ['1 - Three Baseline Conditions'],
    'OR':     ['2 - Three Outer Race Fault Conditions',
               '3 - Seven More Outer Race Fault Conditions'],
    'IR':     ['4 - Seven Inner Race Fault Conditions'],
}

def load_gs(f):
    mat = sio.loadmat(f)
    return np.array(mat['bearing'][0,0]['gs']).squeeze().astype(float)

rows, labels = [], []
for lab, folders in FOLDERS.items():
    for folder in folders:
        for f in glob.glob(f'{BASE}/{folder}/*.mat'):
            try:
                sig = load_gs(f)
            except Exception:
                continue
            n = len(sig)//WIN
            for i in range(n):
                seg = sig[i*WIN:(i+1)*WIN]
                rows.append(extract_features(seg))
                labels.append(lab)

mfpt = pd.DataFrame(rows, columns=COLS)
mfpt['fault'] = labels
print(mfpt.shape)
print(mfpt['fault'].value_counts())
mfpt.head()

In [ ]:
import scipy.io as sio, numpy as np, pandas as pd, glob, os
from scipy.stats import skew, kurtosis
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

WIN = 2048
COLS = ['max','min','mean','sd','rms','skewness','kurtosis','crest','form']

def extract_features(sig):
    mx, mn = sig.max(), sig.min(); mean = sig.mean(); sd = sig.std()
    rms = np.sqrt(np.mean(sig**2))
    return [mx, mn, mean, sd, rms, skew(sig), kurtosis(sig),
            mx/(rms+1e-12), rms/(abs(mean)+1e-12)]

# --- rebuild CWRU (3-class) from raw ---
RAW = '/kaggle/input/datasets/brjapon/cwru-bearing-datasets/raw'
def cwru_label(f):
    b = os.path.basename(f)
    if b.startswith('IR'): return 'IR'
    if b.startswith('OR'): return 'OR'
    if b.startswith('Time_Normal') or b.startswith('Normal'): return 'Normal'
    return None
rows, labs = [], []
for f in glob.glob(f'{RAW}/*.mat'):
    lab = cwru_label(f)
    if lab is None: continue
    mat = sio.loadmat(f)
    de = [k for k in mat if k.endswith('DE_time')]
    if not de: continue
    sig = mat[de[0]].squeeze().astype(float)
    for i in range(len(sig)//WIN):
        rows.append(extract_features(sig[i*WIN:(i+1)*WIN])); labs.append(lab)
cwru_re = pd.DataFrame(rows, columns=COLS); cwru_re['fault'] = labs

le = LabelEncoder().fit(['IR','Normal','OR'])
Xc, yc = cwru_re[COLS].values, le.transform(cwru_re['fault'])
Xm, ym = mfpt[COLS].values,    le.transform(mfpt['fault'])

# --- Experiment: CWRU -> MFPT transfer ---
# raw features
clf = XGBClassifier(random_state=42, eval_metric='mlogloss').fit(Xc, yc)
acc_raw = accuracy_score(ym, clf.predict(Xm))

# per-dataset standardized
sc_c, sc_m = StandardScaler().fit(Xc), StandardScaler().fit(Xm)
clf2 = XGBClassifier(random_state=42, eval_metric='mlogloss').fit(sc_c.transform(Xc), yc)
pred_std = clf2.predict(sc_m.transform(Xm))
acc_std = accuracy_score(ym, pred_std)

print(f"CWRU->MFPT raw:          {acc_raw:.3f}")
print(f"CWRU->MFPT standardized: {acc_std:.3f}")
print("Confusion (standardized), rows=true [IR,Normal,OR]:")
print(confusion_matrix(ym, pred_std))
print("chance level = 0.333")

In [ ]:
import numpy as np
from scipy.linalg import sqrtm
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

def coral(Xs, Xt):
    Xs = (Xs - Xs.mean(0))/(Xs.std(0)+1e-8)
    Xt = (Xt - Xt.mean(0))/(Xt.std(0)+1e-8)
    Cs = np.cov(Xs, rowvar=False) + np.eye(Xs.shape[1])
    Ct = np.cov(Xt, rowvar=False) + np.eye(Xt.shape[1])
    Xs_a = Xs @ np.real(sqrtm(np.linalg.inv(Cs))) @ np.real(sqrtm(Ct))
    return Xs_a, Xt

Xc_a, Xm_s = coral(Xc, Xm)
clf3 = XGBClassifier(random_state=42, eval_metric='mlogloss').fit(Xc_a, yc)
acc_coral = accuracy_score(ym, clf3.predict(Xm_s))
print(f"CWRU->MFPT CORAL: {acc_coral:.3f}")

In [ ]:
import scipy.io as sio, numpy as np, pandas as pd, glob, os
from scipy.stats import skew, kurtosis

WIN = 2048
COLS = ['max','min','mean','sd','rms','skewness','kurtosis','crest','form']

def extract_features(sig):
    mx, mn = sig.max(), sig.min(); mean = sig.mean(); sd = sig.std()
    rms = np.sqrt(np.mean(sig**2))
    return [mx, mn, mean, sd, rms, skew(sig), kurtosis(sig),
            mx/(rms+1e-12), rms/(abs(mean)+1e-12)]

# ---------- CWRU ----------
RAW='/kaggle/input/datasets/brjapon/cwru-bearing-datasets/raw'
def cwru_label(f):
    b=os.path.basename(f)
    if b.startswith('IR'):return 'IR'
    if b.startswith('OR'):return 'OR'
    if b.startswith(('Time_Normal','Normal')):return 'Normal'
    return None
r,l=[],[]
for f in glob.glob(f'{RAW}/*.mat'):
    lab=cwru_label(f)
    if lab is None:continue
    m=sio.loadmat(f); de=[k for k in m if k.endswith('DE_time')]
    if not de:continue
    s=m[de[0]].squeeze().astype(float)
    for i in range(len(s)//WIN):
        r.append(extract_features(s[i*WIN:(i+1)*WIN])); l.append(lab)
cwru=pd.DataFrame(r,columns=COLS); cwru['fault']=l

# ---------- Paderborn ----------
ROOT='/kaggle/input/datasets/dippatel03/paderborn-db'; COND='N15_M07_F10'
SEL={'Normal':['K001','K002'],'OR':['KA04','KA07'],'IR':['KI04','KI07']}
r,l=[],[]
for lab,codes in SEL.items():
    for code in codes:
        for f in glob.glob(f'{ROOT}/{code}/{COND}_{code}_*.mat'):
            key=os.path.splitext(os.path.basename(f))[0]
            try: s=np.array(sio.loadmat(f)[key][0,0]['Y'][0,6]['Data']).squeeze().astype(float)
            except: continue
            for i in range(len(s)//WIN):
                r.append(extract_features(s[i*WIN:(i+1)*WIN])); l.append(lab)
pb=pd.DataFrame(r,columns=COLS); pb['fault']=l

# ---------- MFPT ----------
BASE='/kaggle/input/datasets/emperorpein/mfpt-fault-datasets/MFPT Fault Data Sets'
FOLDERS={'Normal':['1 - Three Baseline Conditions'],
         'OR':['2 - Three Outer Race Fault Conditions','3 - Seven More Outer Race Fault Conditions'],
         'IR':['4 - Seven Inner Race Fault Conditions']}
r,l=[],[]
for lab,folders in FOLDERS.items():
    for folder in folders:
        for f in glob.glob(f'{BASE}/{folder}/*.mat'):
            try: s=np.array(sio.loadmat(f)['bearing'][0,0]['gs']).squeeze().astype(float)
            except: continue
            for i in range(len(s)//WIN):
                r.append(extract_features(s[i*WIN:(i+1)*WIN])); l.append(lab)
mfpt=pd.DataFrame(r,columns=COLS); mfpt['fault']=l

print("CWRU:", cwru.shape[0], "| Paderborn:", pb.shape[0], "| MFPT:", mfpt.shape[0])

In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, StratifiedKFold

datasets = {'CWRU':cwru, 'Paderborn':pb, 'MFPT':mfpt}
le = LabelEncoder().fit(['IR','Normal','OR'])
names = list(datasets)
mat = np.zeros((3,3))

for i, src in enumerate(names):
    Xs = datasets[src][COLS].values
    ys = le.transform(datasets[src]['fault'])
    sc = StandardScaler().fit(Xs)
    clf = XGBClassifier(random_state=42, eval_metric='mlogloss').fit(sc.transform(Xs), ys)
    for j, tgt in enumerate(names):
        Xt = datasets[tgt][COLS].values
        yt = le.transform(datasets[tgt]['fault'])
        if src == tgt:
            # within-dataset: 5-fold CV accuracy
            skf = StratifiedKFold(5, shuffle=True, random_state=42)
            mat[i,j] = cross_val_score(
                XGBClassifier(random_state=42, eval_metric='mlogloss'),
                StandardScaler().fit_transform(Xt), yt, cv=skf).mean()
        else:
            sct = StandardScaler().fit(Xt)
            mat[i,j] = accuracy_score(yt, clf.predict(sct.transform(Xt)))

tm = pd.DataFrame(mat, index=[f'train:{n}' for n in names],
                  columns=[f'test:{n}' for n in names]).round(3)
print(tm)
print("\nDiagonal = within-dataset (5-fold CV). Off-diagonal = cross-dataset transfer. Chance=0.333")

In [ ]:
import matplotlib.pyplot as plt, numpy as np
fig, ax = plt.subplots(figsize=(5.5,4.5))
im = ax.imshow(mat, cmap='RdYlGn', vmin=0, vmax=1)
names = ['CWRU','Paderborn','MFPT']
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels([f'test:\n{n}' for n in names])
ax.set_yticklabels([f'train: {n}' for n in names])
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{mat[i,j]:.3f}', ha='center', va='center',
                color='black', fontweight='bold')
ax.set_title('Cross-dataset transfer accuracy\n(diagonal=within-dataset CV; chance=0.333)')
plt.colorbar(im, label='accuracy'); plt.tight_layout()
plt.savefig('/kaggle/working/transfer_matrix.png', dpi=300, bbox_inches='tight')
plt.savefig('/kaggle/working/transfer_matrix.pdf', bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import wasserstein_distance, pearsonr
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import shap

le = LabelEncoder().fit(['IR','Normal','OR'])

def zt(a): return (a - a.mean(0))/(a.std(0)+1e-8)

def shap_importance(dfset):
    X = dfset[COLS].values; y = le.transform(dfset['fault'])
    m = XGBClassifier(random_state=42, eval_metric='mlogloss').fit(X, y)
    sv = np.abs(np.array(shap.TreeExplainer(m).shap_values(X)))
    fax = [ax for ax,s in enumerate(sv.shape) if s==len(COLS)][-1]
    other = tuple(ax for ax in range(sv.ndim) if ax!=fax)
    imp = sv.mean(axis=other).ravel()
    return imp/imp.sum()

imp_cwru = shap_importance(cwru)

pairs = {'CWRU-Paderborn':(cwru,pb), 'CWRU-MFPT':(cwru,mfpt), 'Paderborn-MFPT':(pb,mfpt)}
print("Per-pair domain gap + SHAP-divergence correlation:\n")
for name,(A,B) in pairs.items():
    Az, Bz = zt(A[COLS].values), zt(B[COLS].values)
    div = np.array([wasserstein_distance(Az[:,i], Bz[:,i]) for i in range(len(COLS))])
    gap = div.mean()                                  # single gap metric
    r,p = pearsonr(imp_cwru, div)                     # importance vs divergence
    print(f"{name:16s} | mean divergence (gap)={gap:.3f} | SHAP-div corr r={r:+.3f} (p={p:.2f})")

In [ ]:
import shutil, os
dst='/kaggle/working/figures'; os.makedirs(dst, exist_ok=True)
for f in os.listdir('/kaggle/working'):
    if f.endswith(('.png','.pdf')):
        shutil.copy(f'/kaggle/working/{f}', f'{dst}/{f}')
shutil.make_archive('/kaggle/working/all_figures','zip',dst)
print("download all_figures.zip")

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

COLS = ['max','min','mean','sd','rms','skewness','kurtosis','crest','form']
le = LabelEncoder().fit(['IR','Normal','OR'])

def zt(a): return (a - a.mean(0))/(a.std(0)+1e-8)

def per_feature_divergence(A, B):
    Az, Bz = zt(A[COLS].values), zt(B[COLS].values)
    return np.array([wasserstein_distance(Az[:,i], Bz[:,i]) for i in range(len(COLS))])

def transfer_acc(src, tgt, feats):
    Xs = src[feats].values; ys = le.transform(src['fault'])
    Xt = tgt[feats].values; yt = le.transform(tgt['fault'])
    sc_s, sc_t = StandardScaler().fit(Xs), StandardScaler().fit(Xt)
    clf = XGBClassifier(random_state=42, eval_metric='mlogloss').fit(sc_s.transform(Xs), ys)
    return accuracy_score(yt, clf.predict(sc_t.transform(Xt)))

pairs = {'CWRU->Paderborn': (cwru, pb),
         'CWRU->MFPT':      (cwru, mfpt),
         'Paderborn->MFPT': (pb, mfpt)}

print("Explainability-guided feature pruning (drop highest-divergence features):\n")
for name, (A, B) in pairs.items():
    div = per_feature_divergence(A, B)
    order = np.argsort(div)          # ascending: lowest-divergence (most transferable) first
    print(f"=== {name} ===")
    print("  divergence rank (low->high):", [COLS[i] for i in order])
    for k in [9, 7, 5, 3]:           # keep k lowest-divergence features
        keep = [COLS[i] for i in order[:k]]
        acc = transfer_acc(A, B, keep)
        print(f"  keep {k} lowest-divergence features -> transfer acc = {acc:.3f}")
    print()

In [ ]:
import matplotlib.pyplot as plt
ks = [9,7,5,3]
res = {'CWRU→Paderborn':[0.363,0.338,0.486,0.574],
       'CWRU→MFPT':[0.124,0.149,0.157,0.160],
       'Paderborn→MFPT':[0.186,0.016,0.039,0.083]}
plt.figure(figsize=(6,4.5))
for name,v in res.items():
    plt.plot(ks, v, marker='o', label=name)
plt.axhline(0.333, ls='--', color='gray', label='chance')
plt.gca().invert_xaxis()
plt.xlabel('Number of features kept (lowest-divergence first)')
plt.ylabel('Transfer accuracy')
plt.title('Explainability-guided feature pruning')
plt.legend(fontsize=8); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('/kaggle/working/pruning_recovery.png', dpi=300, bbox_inches='tight')
plt.savefig('/kaggle/working/pruning_recovery.pdf', bbox_inches='tight')
plt.show()

In [ ]:
import shutil, os
dst='/kaggle/working/figures'; os.makedirs(dst, exist_ok=True)
for f in os.listdir('/kaggle/working'):
    if f.endswith(('.png','.pdf')):
        shutil.copy(f'/kaggle/working/{f}', f'{dst}/{f}')
shutil.make_archive('/kaggle/working/all_figures','zip',dst)
print("download all_figures.zip")

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

COLS = ['max','min','mean','sd','rms','skewness','kurtosis','crest','form']
le = LabelEncoder().fit(['IR','Normal','OR'])

# CWRU -> Paderborn, standardized (a representative failed transfer)
Xc, yc = cwru[COLS].values, le.transform(cwru['fault'])
Xp, yp = pb[COLS].values,   le.transform(pb['fault'])
sc_c, sc_p = StandardScaler().fit(Xc), StandardScaler().fit(Xp)
clf = XGBClassifier(random_state=42, eval_metric='mlogloss').fit(sc_c.transform(Xc), yc)
pred = clf.predict(sc_p.transform(Xp))

cm = confusion_matrix(yp, pred)
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
fig, ax = plt.subplots(figsize=(4,3.5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('CWRU→Paderborn transfer (standardized)')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_cwru_paderborn.png', dpi=300, bbox_inches='tight')
plt.savefig('/kaggle/working/confusion_cwru_paderborn.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CROSS-DOMAIN CONFIDENCE & UNCERTAINTY AUDIT
# Trains: CWRU model -> evaluate confidence on CWRU (in-domain,
# held-out), Paderborn, MFPT (foreign domains)
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

# ---------- domain definitions ----------
domains = {
    "CWRU":      (Xte_cwru_feat,  Xte_cwru_img,  yte_cwru),   # held-out CWRU test split
    "Paderborn": (X_pad_feat,     X_pad_img,     y_pad),
    "MFPT":      (X_mfpt_feat,    X_mfpt_img,    y_mfpt),
}
# xgb_model : trained on CWRU train split (9 features, per-dataset standardized)
# cnn_model : trained on CWRU train split (Keras, has Dropout layers)
# -----------------------------------------------------------------

N_MC = 30  # MC dropout forward passes

def mc_dropout_predict(model, X, n_passes=N_MC, batch=256):
    """Keras: force dropout at inference with training=True."""
    preds = np.stack([
        model(X, training=True).numpy() for _ in range(n_passes)
    ])                                   # (n_passes, N, 3)
    mean_p = preds.mean(axis=0)          # predictive mean
    # uncertainty = mean std of class probs across passes
    unc = preds.std(axis=0).mean(axis=1) # (N,)
    return mean_p, unc

rows = []
store = {}  # keep raw arrays for OOD-AUROC later

for name, (Xf, Xi, y) in domains.items():
    # ----- XGBoost -----
    p_xgb   = xgb_model.predict_proba(Xf)
    conf_x  = p_xgb.max(axis=1)
    pred_x  = p_xgb.argmax(axis=1)
    acc_x   = balanced_accuracy_score(y, pred_x)

    # ----- CNN + MC dropout -----
    mean_p, unc = mc_dropout_predict(cnn_model, Xi)
    conf_c  = mean_p.max(axis=1)
    pred_c  = mean_p.argmax(axis=1)
    acc_c   = balanced_accuracy_score(y, pred_c)

    store[name] = dict(conf_x=conf_x, conf_c=conf_c, unc=unc)

    rows.append(dict(
        domain=name,
        xgb_bal_acc=round(acc_x, 3),
        xgb_mean_conf=round(conf_x.mean(), 3),
        xgb_conf_p90=round(np.quantile(conf_x, 0.9), 3),
        xgb_frac_conf_gt95=round((conf_x > 0.95).mean(), 3),
        cnn_bal_acc=round(acc_c, 3),
        cnn_mean_conf=round(conf_c.mean(), 3),
        cnn_mean_mc_unc=round(unc.mean(), 4),
    ))

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# ============================================================
# OOD detection: can uncertainty flag foreign-domain samples?
# score = MC-dropout uncertainty (CNN) or 1 - confidence (XGB)
# label = 0 for in-domain (CWRU), 1 for foreign
# ============================================================
for foreign in ["Paderborn", "MFPT"]:
    # CNN uncertainty as OOD score
    s   = np.concatenate([store["CWRU"]["unc"], store[foreign]["unc"]])
    lab = np.concatenate([np.zeros(len(store["CWRU"]["unc"])),
                          np.ones(len(store[foreign]["unc"]))])
    auroc_cnn = roc_auc_score(lab, s)

    # XGB (1 - confidence) as OOD score
    s2   = np.concatenate([1 - store["CWRU"]["conf_x"],
                           1 - store[foreign]["conf_x"]])
    auroc_xgb = roc_auc_score(lab, s2)

    # significance of uncertainty shift
    u, p = mannwhitneyu(store["CWRU"]["unc"], store[foreign]["unc"])
    print(f"CWRU vs {foreign}:  OOD-AUROC  CNN(MC-unc)={auroc_cnn:.3f}  "
          f"XGB(1-conf)={auroc_xgb:.3f}   Mann-Whitney p={p:.2e}")

# ============================================================
# Figure: confidence distributions in-domain vs foreign
# ============================================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
bins = np.linspace(0.3, 1.0, 40)
for name in domains:
    axes[0].hist(store[name]["conf_x"], bins=bins, alpha=0.5,
                 density=True, label=name)
axes[0].set_title("XGBoost max-probability confidence")
axes[0].set_xlabel("confidence"); axes[0].legend()

ubins = np.linspace(0, max(s.max() for s in
                    [store[n]["unc"] for n in domains]), 40)
for name in domains:
    axes[1].hist(store[name]["unc"], bins=ubins, alpha=0.5,
                 density=True, label=name)
axes[1].set_title("CNN MC-dropout uncertainty")
axes[1].set_xlabel("uncertainty"); axes[1].legend()

plt.tight_layout()
plt.savefig("cross_domain_confidence.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
for k, v in list(globals().items()):
    if k.startswith("_"): 
        continue
    if isinstance(v, np.ndarray):
        print(f"{k:35s} ndarray  shape={v.shape}  dtype={v.dtype}")
    elif type(v).__name__ in ("DataFrame", "Series"):
        print(f"{k:35s} {type(v).__name__}  shape={v.shape}")
    elif "XGB" in type(v).__name__ or "Booster" in type(v).__name__:
        print(f"{k:35s} XGBoost model")
    elif "Model" in type(v).__name__ or "Sequential" in type(v).__name__:
        print(f"{k:35s} Keras model")
    elif type(v).__module__.startswith("torch") and hasattr(v, "parameters"):
        print(f"{k:35s} PyTorch model")

In [ ]:
# ============================================================
# CROSS-DOMAIN CONFIDENCE AUDIT (XGBoost, self-contained)
# ============================================================
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from scipy.stats import mannwhitneyu
from xgboost import XGBClassifier

# ---- identify label column (checks common names, else last col) ----
def split_df(d):
    lab = next((c for c in d.columns if str(c).lower() in
                ("label", "class", "y", "target", "fault")), d.columns[-1])
    feats = [c for c in d.columns if c != lab]
    return d[feats].values.astype(float), d[lab].values.astype(int), feats

Xc_raw, yc_, feat_names = split_df(cwru)
Xp_raw, yp_, _          = split_df(pb)
Xm_raw, ym_, _          = split_df(mfpt)
print("label col detected -> features:", feat_names)
print("class counts  CWRU:", np.bincount(yc_), " PB:", np.bincount(yp_),
      " MFPT:", np.bincount(ym_))

# ---- per-dataset standardization (matches the transfer protocol) ----
z = lambda X: (X - X.mean(0)) / (X.std(0) + 1e-12)

# ---- honest in-domain split on CWRU ----
Xtr, Xte, ytr, yte = train_test_split(Xc_raw, yc_, test_size=0.25,
                                      stratify=yc_, random_state=42)
mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-12   # standardize CWRU by TRAIN stats
Xtr_z, Xte_z = (Xtr - mu) / sd, (Xte - mu) / sd
Xp_z, Xm_z   = z(Xp_raw), z(Xm_raw)        # foreign sets: own stats

model = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                      eval_metric="mlogloss", random_state=42)
model.fit(Xtr_z, ytr)

domains = {"CWRU (held-out)": (Xte_z, yte),
           "Paderborn":       (Xp_z,  yp_),
           "MFPT":            (Xm_z,  ym_)}

rows, store = [], {}
for name, (Xd, yd) in domains.items():
    p    = model.predict_proba(Xd)
    conf = p.max(1); pred = p.argmax(1)
    store[name] = conf
    rows.append(dict(domain=name,
        bal_acc=round(balanced_accuracy_score(yd, pred), 3),
        mean_conf=round(conf.mean(), 3),
        median_conf=round(np.median(conf), 3),
        frac_conf_gt90=round((conf > 0.90).mean(), 3),
        frac_conf_gt95=round((conf > 0.95).mean(), 3)))
print(pd.DataFrame(rows).to_string(index=False))

# ---- can (1 - confidence) detect foreign-domain samples? ----
for foreign in ["Paderborn", "MFPT"]:
    s   = np.concatenate([1 - store["CWRU (held-out)"], 1 - store[foreign]])
    lab = np.concatenate([np.zeros(len(store["CWRU (held-out)"])),
                          np.ones(len(store[foreign]))])
    auroc = roc_auc_score(lab, s)
    _, pval = mannwhitneyu(store["CWRU (held-out)"], store[foreign])
    print(f"CWRU vs {foreign}:  OOD-AUROC={auroc:.3f}   Mann-Whitney p={pval:.2e}")

# ---- figure: confidence distributions ----
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
bins = np.linspace(0.3, 1.0, 45)
for name, c in store.items():
    plt.hist(c, bins=bins, alpha=0.55, density=True, label=name)
plt.xlabel("max predicted probability (confidence)")
plt.ylabel("density"); plt.legend()
plt.title("XGBoost confidence: in-domain vs foreign domains")
plt.tight_layout()
plt.savefig("cross_domain_confidence_xgb.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
print(cwru.columns.tolist())
print(cwru.iloc[0])

In [ ]:
LABEL_MAP = {"Normal": 0, "IR": 1, "OR": 2}   # fixed order across datasets

def split_df(d):
    lab = next((c for c in d.columns if str(c).lower() in
                ("label", "class", "y", "target", "fault")), d.columns[-1])
    feats = [c for c in d.columns if c != lab]
    yraw = d[lab]
    if yraw.dtype == object:
        unmapped = set(yraw.unique()) - set(LABEL_MAP)
        if unmapped:
            raise ValueError(f"Unmapped labels {unmapped} — extend LABEL_MAP")
        yy = yraw.map(LABEL_MAP).values.astype(int)
    else:
        yy = yraw.values.astype(int)
    return d[feats].values.astype(float), yy, feats

In [ ]:
# ============================================================
# CROSS-DOMAIN CONFIDENCE AUDIT — single cell, hardcoded schema
# ============================================================
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from scipy.stats import mannwhitneyu
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

FEATS = ['max','min','mean','sd','rms','skewness','kurtosis','crest','form']

# --- check label vocabulary across datasets first ---
print("cwru :", sorted(cwru['fault'].unique()))
print("pb   :", sorted(pb['fault'].unique()))
print("mfpt :", sorted(mfpt['fault'].unique()))

# extend this map if the printout shows other spellings
LABEL_MAP = {"Normal": 0, "IR": 1, "OR": 2,
             "normal": 0, "baseline": 0, "Baseline": 0,
             "healthy": 0, "Healthy": 0,
             "inner": 1, "Inner": 1, "outer": 2, "Outer": 2}

def get_xy(d):
    yy = d['fault'].map(LABEL_MAP)
    if yy.isna().any():
        raise ValueError(f"Unmapped labels: {sorted(d['fault'][yy.isna()].unique())}")
    return d[FEATS].values.astype(float), yy.values.astype(int)

Xc_raw, yc_ = get_xy(cwru)
Xp_raw, yp_ = get_xy(pb)
Xm_raw, ym_ = get_xy(mfpt)
print("counts  CWRU:", np.bincount(yc_), " PB:", np.bincount(yp_),
      " MFPT:", np.bincount(ym_))

# --- per-dataset standardization (matches transfer protocol) ---
z = lambda X: (X - X.mean(0)) / (X.std(0) + 1e-12)

Xtr, Xte, ytr, yte = train_test_split(Xc_raw, yc_, test_size=0.25,
                                      stratify=yc_, random_state=42)
mu, sd_ = Xtr.mean(0), Xtr.std(0) + 1e-12
Xtr_z, Xte_z = (Xtr - mu)/sd_, (Xte - mu)/sd_
Xp_z, Xm_z = z(Xp_raw), z(Xm_raw)

model = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                      eval_metric="mlogloss", random_state=42)
model.fit(Xtr_z, ytr)

domains = {"CWRU (held-out)": (Xte_z, yte),
           "Paderborn":       (Xp_z, yp_),
           "MFPT":            (Xm_z, ym_)}

rows, store = [], {}
for name, (Xd, yd) in domains.items():
    p = model.predict_proba(Xd)
    conf, pred = p.max(1), p.argmax(1)
    store[name] = conf
    rows.append(dict(domain=name,
        bal_acc=round(balanced_accuracy_score(yd, pred), 3),
        mean_conf=round(conf.mean(), 3),
        median_conf=round(np.median(conf), 3),
        frac_conf_gt90=round((conf > 0.90).mean(), 3),
        frac_conf_gt95=round((conf > 0.95).mean(), 3)))
print(pd.DataFrame(rows).to_string(index=False))

for foreign in ["Paderborn", "MFPT"]:
    s   = np.concatenate([1-store["CWRU (held-out)"], 1-store[foreign]])
    lab = np.concatenate([np.zeros(len(store["CWRU (held-out)"])),
                          np.ones(len(store[foreign]))])
    _, pval = mannwhitneyu(store["CWRU (held-out)"], store[foreign])
    print(f"CWRU vs {foreign}:  OOD-AUROC={roc_auc_score(lab, s):.3f}   "
          f"Mann-Whitney p={pval:.2e}")

plt.figure(figsize=(7,4))
bins = np.linspace(0.3, 1.0, 45)
for name, c in store.items():
    plt.hist(c, bins=bins, alpha=0.55, density=True, label=name)
plt.xlabel("max predicted probability (confidence)")
plt.ylabel("density"); plt.legend()
plt.title("XGBoost confidence: in-domain vs foreign domains")
plt.tight_layout()
plt.savefig("cross_domain_confidence_xgb.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from IPython.display import FileLink
FileLink("cross_domain_confidence_xgb.png")

In [ ]:
seed_rows = []
for seed in [0, 1, 2, 3, 4]:
    Xtr, Xte, ytr, yte = train_test_split(Xc_raw, yc_, test_size=0.25,
                                          stratify=yc_, random_state=seed)
    mu, sd_ = Xtr.mean(0), Xtr.std(0) + 1e-12
    m = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                      eval_metric="mlogloss", random_state=seed)
    m.fit((Xtr-mu)/sd_, ytr)
    for name, (Xd, yd) in {"CWRU": ((Xte-mu)/sd_, yte),
                           "Paderborn": (z(Xp_raw), yp_),
                           "MFPT": (z(Xm_raw), ym_)}.items():
        p = m.predict_proba(Xd); conf = p.max(1)
        seed_rows.append(dict(seed=seed, domain=name,
            bal_acc=balanced_accuracy_score(yd, p.argmax(1)),
            mean_conf=conf.mean(), frac_gt95=(conf > 0.95).mean()))

sdf = pd.DataFrame(seed_rows)
print(sdf.groupby("domain").agg(['mean','std']).round(3))

In [ ]:
from scipy.stats import wasserstein_distance

def per_class_gap(Xa, ya, Xb, yb):
    Xa, Xb = z(Xa), z(Xb)
    return {c: np.mean([wasserstein_distance(Xa[ya==ci][:,f], Xb[yb==ci][:,f])
                        for f in range(9)])
            for ci, c in enumerate(["Normal","IR","OR"])}

for nm, (Xa,ya,Xb,yb) in {"CWRU-PB":(Xc_raw,yc_,Xp_raw,yp_),
                          "CWRU-MFPT":(Xc_raw,yc_,Xm_raw,ym_),
                          "PB-MFPT":(Xp_raw,yp_,Xm_raw,ym_)}.items():
    print(nm, {k: round(v,3) for k,v in per_class_gap(Xa,ya,Xb,yb).items()})

In [ ]:
# ============================================================
# PRUNING SWEEP: divergence-guided vs random, all 6 pairs, 5 seeds
# ============================================================
from scipy.stats import wasserstein_distance
from itertools import permutations

data = {"CWRU": (Xc_raw, yc_), "PB": (Xp_raw, yp_), "MFPT": (Xm_raw, ym_)}
rng_global = np.random.default_rng(0)
N_RANDOM = 10

def feat_divergence(Xa, Xb):
    Xa, Xb = z(Xa), z(Xb)
    return np.array([wasserstein_distance(Xa[:, f], Xb[:, f]) for f in range(9)])

def transfer_acc(Xs_, ys_, Xt_, yt_, cols, seed):
    m = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                      eval_metric="mlogloss", random_state=seed)
    m.fit(z(Xs_)[:, cols], ys_)
    return balanced_accuracy_score(yt_, m.predict(z(Xt_)[:, cols]))

results = []
for src, tgt in permutations(data, 2):
    Xs_, ys_ = data[src]; Xt_, yt_ = data[tgt]
    div = feat_divergence(Xs_, Xt_)
    rank = np.argsort(div)                      # lowest divergence first
    for k in range(9, 0, -1):
        guided_cols = rank[:k]
        g_accs = [transfer_acc(Xs_, ys_, Xt_, yt_, guided_cols, s) for s in range(5)]
        r_accs = []
        for _ in range(N_RANDOM):
            cols = rng_global.choice(9, size=k, replace=False)
            r_accs.append(transfer_acc(Xs_, ys_, Xt_, yt_, cols, 0))
        results.append(dict(pair=f"{src}->{tgt}", k=k,
                            guided_mean=np.mean(g_accs), guided_std=np.std(g_accs),
                            random_mean=np.mean(r_accs), random_std=np.std(r_accs)))
        print(f"{src}->{tgt} k={k}: guided={np.mean(g_accs):.3f}±{np.std(g_accs):.3f}  "
              f"random={np.mean(r_accs):.3f}±{np.std(r_accs):.3f}")

rdf = pd.DataFrame(results)
rdf.to_csv("pruning_sweep.csv", index=False)

# ---- figure: 6 panels ----
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
for ax, pair in zip(axes.flat, rdf["pair"].unique()):
    sub = rdf[rdf["pair"] == pair]
    ax.errorbar(sub["k"], sub["guided_mean"], yerr=sub["guided_std"],
                marker="o", label="divergence-guided", capsize=3)
    ax.errorbar(sub["k"], sub["random_mean"], yerr=sub["random_std"],
                marker="s", label="random subset", capsize=3, alpha=0.7)
    ax.axhline(1/3, ls="--", c="gray", lw=1, label="chance")
    ax.set_title(pair); ax.set_xlabel("features kept (lowest-divergence first)")
    ax.invert_xaxis()
axes[0, 0].set_ylabel("balanced transfer accuracy")
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.savefig("pruning_guided_vs_random.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from IPython.display import FileLink, display
display(FileLink("pruning_guided_vs_random.png"))
display(FileLink("pruning_sweep.csv"))
display(FileLink("cross_domain_confidence_xgb.png"))

In [ ]:
# ECE + reliability diagram for in-domain CWRU model
def ece(conf, correct, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1)
    e, xs, ys = 0.0, [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() == 0: continue
        acc_b, conf_b = correct[m].mean(), conf[m].mean()
        e += m.mean() * abs(acc_b - conf_b)
        xs.append(conf_b); ys.append(acc_b)
    return e, xs, ys

p = model.predict_proba(Xte_z)
conf, correct = p.max(1), (p.argmax(1) == yte).astype(float)
e, xs, ys = ece(conf, correct)
print(f"In-domain ECE = {e:.4f}")

# cross-domain ECE (the contrast that matters)
for name, (Xd, yd) in [("Paderborn", (Xp_z, yp_)), ("MFPT", (Xm_z, ym_))]:
    pd_ = model.predict_proba(Xd)
    ef, _, _ = ece(pd_.max(1), (pd_.argmax(1) == yd).astype(float))
    print(f"{name} ECE = {ef:.4f}")

plt.figure(figsize=(4.5, 4.5))
plt.plot([0, 1], [0, 1], "--", c="gray", label="perfect")
plt.plot(xs, ys, "o-", label=f"CWRU held-out (ECE={e:.3f})")
plt.xlabel("mean confidence"); plt.ylabel("accuracy")
plt.legend(); plt.title("Reliability diagram (in-domain)")
plt.tight_layout()
plt.savefig("reliability_diagram.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
print(rdf[rdf.pair == "MFPT->PB"].to_string())

In [ ]:
# ---- self-contained ECE, no stale state ----
from sklearn.model_selection import train_test_split
Xtr, Xte, ytr, yte = train_test_split(Xc_raw, yc_, test_size=0.25,
                                      stratify=yc_, random_state=42)
mu, sd_ = Xtr.mean(0), Xtr.std(0) + 1e-12
mdl = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                    eval_metric="mlogloss", random_state=42)
mdl.fit((Xtr - mu)/sd_, ytr)

sets = {"CWRU held-out": ((Xte - mu)/sd_, yte),
        "Paderborn":     (z(Xp_raw), yp_),
        "MFPT":          (z(Xm_raw), ym_)}

plt.figure(figsize=(4.8, 4.8))
plt.plot([0,1],[0,1],"--",c="gray",label="perfect")
for name,(Xd,yd) in sets.items():
    p = mdl.predict_proba(Xd)
    conf, correct = p.max(1), (p.argmax(1)==yd).astype(float)
    e, xs, ys = ece(conf, correct)
    print(f"{name:15s} ECE = {e:.4f}")
    plt.plot(xs, ys, "o-", label=f"{name} (ECE={e:.3f})")
plt.xlabel("mean confidence"); plt.ylabel("accuracy")
plt.legend(fontsize=8); plt.title("Reliability: in-domain vs foreign")
plt.tight_layout()
plt.savefig("reliability_all_domains.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from IPython.display import FileLink, display
display(FileLink("reliability_all_domains.png"))
display(FileLink("reliability_diagram.png"))   # the buggy one — not used

In [ ]:
# ============================================================
# CNN + MC-DROPOUT CROSS-DOMAIN AUDIT (self-contained)
# ============================================================
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt

WIN = 2048  # 2048 = 32*64 -> reshape needs 2048? No: 32*32=1024.
# NOTE: X is 32x32 = 1024 samples per image, so the image window is
# 1024 rather than the 2048 used for the hand-crafted features.
IMG_WIN = 1024   # samples per image window (32*32)

# ---------------------------------------------------------
# Return a list of (signal_1d_array, label_str)
# — same file-loading loops as the feature-extraction
# cells for each dataset, yield the raw signal per file.
# ---------------------------------------------------------
def load_cwru_signals():
    out = []
    # for each CWRU file:
    #   sig = <load .mat / array>;  out.append((sig, "Normal"/"IR"/"OR"))
    raise NotImplementedError("CWRU loading loop")
    return out

def load_pb_signals():
    raise NotImplementedError("Paderborn loading loop")

def load_mfpt_signals():
    raise NotImplementedError("MFPT loading loop")
# ---------------------------------------------------------

LM = {"Normal": 0, "IR": 1, "OR": 2}

def window_dataset(sig_label_list, win=IMG_WIN):
    Xs, ys = [], []
    for sig, lab in sig_label_list:
        sig = np.asarray(sig, float).ravel()
        n = len(sig) // win
        for i in range(n):
            w = sig[i*win:(i+1)*win]
            w = (w - w.mean()) / (w.std() + 1e-12)   # per-window standardize
            Xs.append(w.reshape(32, 32, 1))
            ys.append(LM[lab])
    return np.array(Xs, np.float32), np.array(ys, np.int64)

Xc_img, yc_img = window_dataset(load_cwru_signals())
Xp_img, yp_img = window_dataset(load_pb_signals())
Xm_img, ym_img = window_dataset(load_mfpt_signals())
print("windows  CWRU:", Xc_img.shape, " PB:", Xp_img.shape, " MFPT:", Xm_img.shape)

# ---- train CNN on CWRU (dropout layers = MC dropout capable) ----
Xtr, Xte, ytr, yte = train_test_split(Xc_img, yc_img, test_size=0.25,
                                      stratify=yc_img, random_state=42)
tf.random.set_seed(42)
cnn = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, 3, activation="relu", input_shape=(32,32,1)),
    tf.keras.layers.MaxPool2D(),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPool2D(),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(3, activation="softmax"),
])
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
            metrics=["accuracy"])
cnn.fit(Xtr, ytr, epochs=30, batch_size=64, validation_split=0.1, verbose=2)

# ---- MC dropout ----
N_MC = 30
def mc_predict(model, Xd, n=N_MC, bs=512):
    preds = np.stack([model(Xd[i:i+bs], training=True).numpy()
                      for _ in range(n)
                      for i in range(0, len(Xd), bs)], axis=0) \
            if False else None
    # simpler & correct: loop passes, full-batch via predict-on-batch chunks
    all_passes = []
    for _ in range(n):
        chunks = [model(Xd[i:i+bs], training=True).numpy()
                  for i in range(0, len(Xd), bs)]
        all_passes.append(np.concatenate(chunks, 0))
    P = np.stack(all_passes)              # (n, N, 3)
    return P.mean(0), P.std(0).mean(1)    # mean probs, uncertainty

domains = {"CWRU (held-out)": (Xte, yte),
           "Paderborn": (Xp_img, yp_img),
           "MFPT": (Xm_img, ym_img)}

rows, store = [], {}
for name, (Xd, yd) in domains.items():
    mp, unc = mc_predict(cnn, Xd)
    conf, pred = mp.max(1), mp.argmax(1)
    store[name] = dict(conf=conf, unc=unc)
    rows.append(dict(domain=name,
        bal_acc=round(balanced_accuracy_score(yd, pred), 3),
        mean_conf=round(conf.mean(), 3),
        mean_mc_unc=round(unc.mean(), 4),
        frac_conf_gt95=round((conf > 0.95).mean(), 3)))
print(pd.DataFrame(rows).to_string(index=False))

for foreign in ["Paderborn", "MFPT"]:
    n_in = len(store["CWRU (held-out)"]["unc"])
    lab = np.concatenate([np.zeros(n_in), np.ones(len(store[foreign]["unc"]))])
    a_unc  = roc_auc_score(lab, np.concatenate(
        [store["CWRU (held-out)"]["unc"],  store[foreign]["unc"]]))
    a_conf = roc_auc_score(lab, np.concatenate(
        [1-store["CWRU (held-out)"]["conf"], 1-store[foreign]["conf"]]))
    _, pv = mannwhitneyu(store["CWRU (held-out)"]["unc"], store[foreign]["unc"])
    print(f"CWRU vs {foreign}: OOD-AUROC  MC-unc={a_unc:.3f}  "
          f"(1-conf)={a_conf:.3f}  p={pv:.2e}")

# ---- figure ----
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name in domains:
    ax[0].hist(store[name]["conf"], bins=40, alpha=0.5, density=True, label=name)
    ax[1].hist(store[name]["unc"],  bins=40, alpha=0.5, density=True, label=name)
ax[0].set_title("CNN confidence"); ax[1].set_title("CNN MC-dropout uncertainty")
for a in ax: a.legend(fontsize=8)
plt.tight_layout()
plt.savefig("cnn_mc_dropout_audit.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.count("/") - 2
    if depth > 4: 
        continue
    print("  " * depth + root.split("/")[-1] + "/")
    for f in files[:8]:
        print("  " * (depth+1) + f)
    if len(files) > 8:
        print("  " * (depth+1) + f"... +{len(files)-8} more")

In [ ]:
import numpy as np, os, scipy.io as sio

# 1) inspect the CNN npz
d = np.load("/kaggle/input/datasets/brjapon/cwru-bearing-datasets/CWRU_48k_load_1_CNN_data.npz")
print("npz keys:", d.files)
for k in d.files:
    print(" ", k, d[k].shape, d[k].dtype)

# 2) what's inside the truncated calciumcarbonate dataset (MFPT source?)
base = "/kaggle/input/datasets/calciumcarbonate"
for root, dirs, files in os.walk(base):
    print(root)
    for f in files[:10]:
        print("   ", f)
    if len(files) > 10: print(f"    ... +{len(files)-10} more")

# 3) structure of one Paderborn .mat (they use nested structs)
p = "/kaggle/input/datasets/dippatel03/paderborn-db/K001/N15_M07_F10_K001_14.mat"
m = sio.loadmat(p)
print("\nPB mat keys:", [k for k in m.keys() if not k.startswith("__")])

In [ ]:
import numpy as np, os

# 1) npz label vocabulary
d = np.load("/kaggle/input/datasets/brjapon/cwru-bearing-datasets/CWRU_48k_load_1_CNN_data.npz")
lab = d["labels"]
print("unique labels:", np.unique(lab, return_counts=True))

# 2) FULL walk of MFPT dataset, no depth cap
for root, dirs, files in os.walk("/kaggle/input/datasets/emperorpein"):
    mats = [f for f in files if f.endswith((".mat", ".csv"))]
    if mats or not dirs:
        print(root, "->", mats[:6], f"(+{max(0,len(mats)-6)} more)" if len(mats) > 6 else "")

# 3) list everything attached, in case MFPT lives elsewhere
print("\nAll attached datasets:")
for p in os.listdir("/kaggle/input/datasets"):
    print(" ", p)

In [ ]:
# ============================================================
# CNN + MC-DROPOUT CROSS-DOMAIN AUDIT — final, self-contained
# ============================================================
import numpy as np, pandas as pd, os, glob, scipy.io as sio
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt

WIN = 1024   # 32*32, matches original npz windows
rng = np.random.default_rng(42)
MAX_WIN_PER_CLASS = 3000   # cap PB/MFPT so sizes stay sane

def window_signal(sig, win=WIN):
    sig = np.asarray(sig, float).ravel()
    n = len(sig) // win
    out = np.empty((n, 32, 32, 1), np.float32)
    for i in range(n):
        w = sig[i*win:(i+1)*win]
        out[i] = ((w - w.mean()) / (w.std() + 1e-12)).reshape(32, 32, 1)
    return out

# ---------- CWRU from npz (drop Ball, merge severities) ----------
d = np.load("/kaggle/input/datasets/brjapon/cwru-bearing-datasets/CWRU_48k_load_1_CNN_data.npz")
raw, labs = d["data"], d["labels"]
def cwru_map(s):
    if s == "Normal": return 0
    if s.startswith("IR"): return 1
    if s.startswith("OR"): return 2
    return -1  # Ball -> drop
yc_all = np.array([cwru_map(s) for s in labs])
keep = yc_all >= 0
Xc_img = raw[keep].reshape(-1, 32, 32, 1).astype(np.float32)
# per-window standardize to match PB/MFPT preprocessing
m_ = Xc_img.reshape(len(Xc_img), -1)
Xc_img = ((m_ - m_.mean(1, keepdims=True)) /
          (m_.std(1, keepdims=True) + 1e-12)).reshape(-1, 32, 32, 1)
yc_img = yc_all[keep]
print("CWRU:", Xc_img.shape, np.bincount(yc_img))

# ---------- Paderborn: N15_M07_F10 files ----------
PB_ROOT = "/kaggle/input/datasets/dippatel03/paderborn-db"
def pb_class(code):
    if code.startswith("K0"): return 0   # healthy
    if code.startswith("KI"): return 1   # inner race
    if code.startswith("KA"): return 2   # outer race
    return -1                            # KB combined -> drop

def pb_load_signal(path):
    m = sio.loadmat(path)
    key = [k for k in m if not k.startswith("__")][0]
    Y = m[key]["Y"][0, 0][0]             # array of channel structs
    for ch in Y:
        if "vib" in str(ch["Name"][0]).lower():
            return ch["Data"].ravel()
    return Y[-1]["Data"].ravel()         # fallback: last channel

Xp_list, yp_list = [], []
for code_dir in sorted(os.listdir(PB_ROOT)):
    c = pb_class(code_dir)
    if c < 0: continue
    files = sorted(glob.glob(f"{PB_ROOT}/{code_dir}/N15_M07_F10_*.mat"))[:3]
    for f in files:
        try:
            w = window_signal(pb_load_signal(f))
            Xp_list.append(w); yp_list.append(np.full(len(w), c))
        except Exception as e:
            print("PB skip", f.split('/')[-1], e)
Xp_img = np.concatenate(Xp_list); yp_img = np.concatenate(yp_list)
print("PB before cap:", Xp_img.shape, np.bincount(yp_img))

# ---------- MFPT ----------
MF_ROOT = "/kaggle/input/datasets/emperorpein/mfpt-fault-datasets/MFPT Fault Data Sets"
def mfpt_signal(path):
    m = sio.loadmat(path)
    b = m["bearing"]
    fld = b.dtype.names
    return b["gs"][0, 0].ravel() if "gs" in fld else b[0, 0][-1].ravel()

mf_groups = [(f"{MF_ROOT}/1 - Three Baseline Conditions", 0),
             (f"{MF_ROOT}/2 - Three Outer Race Fault Conditions", 2),
             (f"{MF_ROOT}/3 - Seven More Outer Race Fault Conditions", 2),
             (f"{MF_ROOT}/4 - Seven Inner Race Fault Conditions", 1)]
Xm_list, ym_list = [], []
for folder, c in mf_groups:
    for f in sorted(glob.glob(f"{folder}/*.mat")):
        try:
            w = window_signal(mfpt_signal(f))
            Xm_list.append(w); ym_list.append(np.full(len(w), c))
        except Exception as e:
            print("MFPT skip", f.split('/')[-1], e)
Xm_img = np.concatenate(Xm_list); ym_img = np.concatenate(ym_list)
print("MFPT before cap:", Xm_img.shape, np.bincount(ym_img))

def cap_per_class(Xd, yd, cap=MAX_WIN_PER_CLASS):
    idx = np.concatenate([
        rng.choice(np.where(yd == c)[0], min(cap, (yd == c).sum()), replace=False)
        for c in np.unique(yd)])
    rng.shuffle(idx)
    return Xd[idx], yd[idx]
Xp_img, yp_img = cap_per_class(Xp_img, yp_img)
Xm_img, ym_img = cap_per_class(Xm_img, ym_img)
print("PB capped:", Xp_img.shape, np.bincount(yp_img),
      " MFPT capped:", Xm_img.shape, np.bincount(ym_img))

# ---------- train CNN on CWRU ----------
Xtr, Xte, ytr, yte = train_test_split(Xc_img, yc_img, test_size=0.25,
                                      stratify=yc_img, random_state=42)
tf.random.set_seed(42)
cnn = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, 3, activation="relu", input_shape=(32, 32, 1)),
    tf.keras.layers.MaxPool2D(), tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPool2D(), tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(3, activation="softmax")])
cnn.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.fit(Xtr, ytr, epochs=30, batch_size=64, validation_split=0.1, verbose=2)

# ---------- MC dropout audit ----------
def mc_predict(model, Xd, n=30, bs=512):
    passes = []
    for _ in range(n):
        chunks = [model(Xd[i:i+bs], training=True).numpy()
                  for i in range(0, len(Xd), bs)]
        passes.append(np.concatenate(chunks, 0))
    P = np.stack(passes)
    return P.mean(0), P.std(0).mean(1)

domains = {"CWRU (held-out)": (Xte, yte),
           "Paderborn": (Xp_img, yp_img),
           "MFPT": (Xm_img, ym_img)}
rows, store = [], {}
for name, (Xd, yd) in domains.items():
    mp, unc = mc_predict(cnn, Xd)
    conf, pred = mp.max(1), mp.argmax(1)
    store[name] = dict(conf=conf, unc=unc)
    rows.append(dict(domain=name,
        bal_acc=round(balanced_accuracy_score(yd, pred), 3),
        mean_conf=round(conf.mean(), 3),
        mean_mc_unc=round(unc.mean(), 4),
        frac_conf_gt95=round((conf > 0.95).mean(), 3)))
print(pd.DataFrame(rows).to_string(index=False))

for foreign in ["Paderborn", "MFPT"]:
    base = store["CWRU (held-out)"]
    lab = np.concatenate([np.zeros(len(base["unc"])),
                          np.ones(len(store[foreign]["unc"]))])
    a_unc = roc_auc_score(lab, np.concatenate([base["unc"], store[foreign]["unc"]]))
    a_cnf = roc_auc_score(lab, np.concatenate([1-base["conf"], 1-store[foreign]["conf"]]))
    _, pv = mannwhitneyu(base["unc"], store[foreign]["unc"])
    print(f"CWRU vs {foreign}: OOD-AUROC  MC-unc={a_unc:.3f}  (1-conf)={a_cnf:.3f}  p={pv:.2e}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name in domains:
    ax[0].hist(store[name]["conf"], bins=40, alpha=0.5, density=True, label=name)
    ax[1].hist(store[name]["unc"],  bins=40, alpha=0.5, density=True, label=name)
ax[0].set_title("CNN softmax confidence"); ax[1].set_title("CNN MC-dropout uncertainty")
for a in ax: a.legend(fontsize=8)
plt.tight_layout()
plt.savefig("cnn_mc_dropout_audit.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from IPython.display import FileLink, display
for f in ["cnn_mc_dropout_audit.png", "reliability_all_domains.png",
          "pruning_guided_vs_random.png", "pruning_sweep.csv",
          "cross_domain_confidence_xgb.png"]:
    display(FileLink(f))

In [ ]:
# ============================================================
# NORMALIZATION REGIME ABLATION: per-window vs per-dataset
# CNN transfer under both regimes, 3 seeds
# Requires the previous cell's loaders in memory (window logic redefined here)
# ============================================================
import numpy as np, pandas as pd, os, glob, scipy.io as sio
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score

WIN = 1024

# ---------- rebuild RAW (unnormalized) windows ----------
def window_raw(sig, win=WIN):
    sig = np.asarray(sig, float).ravel()
    n = len(sig) // win
    return sig[:n*win].reshape(n, win).astype(np.float32)

# CWRU from npz — check if already normalized
d = np.load("/kaggle/input/datasets/brjapon/cwru-bearing-datasets/CWRU_48k_load_1_CNN_data.npz")
raw, labs = d["data"].reshape(len(d["data"]), -1), d["labels"]
w_means, w_stds = np.abs(raw.mean(1)).mean(), raw.std(1)
print(f"npz window |mean| avg={w_means:.4f}, std range=({w_stds.min():.3f},{w_stds.max():.3f})")
NPZ_PRENORMALIZED = (w_means < 1e-3) and (abs(w_stds.mean() - 1) < 0.05)
print("npz pre-normalized per window:", NPZ_PRENORMALIZED)

def cwru_map(s):
    return 0 if s == "Normal" else 1 if s.startswith("IR") else 2 if s.startswith("OR") else -1

if not NPZ_PRENORMALIZED:
    yc0 = np.array([cwru_map(s) for s in labs]); k = yc0 >= 0
    Wc_raw, yc_r = raw[k], yc0[k]
else:
    # regenerate from raw .mat files
    RAWDIR = "/kaggle/input/datasets/brjapon/cwru-bearing-datasets/raw"
    def cwru_file_class(f):
        f = os.path.basename(f)
        if "Normal" in f: return 0
        if f.startswith("IR"): return 1
        if f.startswith("OR"): return 2
        return -1
    Ws, ys = [], []
    for f in sorted(glob.glob(f"{RAWDIR}/*.mat")):
        c = cwru_file_class(f)
        if c < 0: continue
        m = sio.loadmat(f)
        key = next(k for k in m if k.endswith("DE_time"))
        w = window_raw(m[key])
        Ws.append(w); ys.append(np.full(len(w), c))
    Wc_raw, yc_r = np.concatenate(Ws), np.concatenate(ys)
print("CWRU raw windows:", Wc_raw.shape, np.bincount(yc_r))

# PB and MFPT raw windows (reuse loaders from previous cell)
def collect(loader_pairs):
    Ws, ys = [], []
    for sig, c in loader_pairs:
        w = window_raw(sig); Ws.append(w); ys.append(np.full(len(w), c))
    return np.concatenate(Ws), np.concatenate(ys)

PB_ROOT = "/kaggle/input/datasets/dippatel03/paderborn-db"
pb_pairs = []
for code_dir in sorted(os.listdir(PB_ROOT)):
    c = pb_class(code_dir)
    if c < 0: continue
    for f in sorted(glob.glob(f"{PB_ROOT}/{code_dir}/N15_M07_F10_*.mat"))[:3]:
        try: pb_pairs.append((pb_load_signal(f), c))
        except Exception: pass
Wp_raw, yp_r = collect(pb_pairs)

MF_ROOT = "/kaggle/input/datasets/emperorpein/mfpt-fault-datasets/MFPT Fault Data Sets"
mf_pairs = []
for folder, c in [(f"{MF_ROOT}/1 - Three Baseline Conditions", 0),
                  (f"{MF_ROOT}/2 - Three Outer Race Fault Conditions", 2),
                  (f"{MF_ROOT}/3 - Seven More Outer Race Fault Conditions", 2),
                  (f"{MF_ROOT}/4 - Seven Inner Race Fault Conditions", 1)]:
    for f in sorted(glob.glob(f"{folder}/*.mat")):
        try: mf_pairs.append((mfpt_signal(f), c))
        except Exception: pass
Wm_raw, ym_r = collect(mf_pairs)
print("PB:", Wp_raw.shape, np.bincount(yp_r), " MFPT:", Wm_raw.shape, np.bincount(ym_r))

# cap
rng = np.random.default_rng(0)
def cap(W, y, n=3000):
    idx = np.concatenate([rng.choice(np.where(y==c)[0], min(n,(y==c).sum()), replace=False)
                          for c in np.unique(y)])
    rng.shuffle(idx); return W[idx], y[idx]
Wp_raw, yp_r = cap(Wp_raw, yp_r); Wm_raw, ym_r = cap(Wm_raw, ym_r)

# ---------- normalization regimes ----------
def norm_perwindow(W, *_):
    return ((W - W.mean(1, keepdims=True)) / (W.std(1, keepdims=True) + 1e-12))
def norm_perdataset(W, mu, sd):
    return (W - mu) / (sd + 1e-12)

def to_img(W): return W.reshape(-1, 32, 32, 1)

def build_cnn(seed):
    tf.random.set_seed(seed)
    m = tf.keras.Sequential([
        tf.keras.layers.Input((32,32,1)),
        tf.keras.layers.Conv2D(32,3,activation="relu"),
        tf.keras.layers.MaxPool2D(), tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Conv2D(64,3,activation="relu"),
        tf.keras.layers.MaxPool2D(), tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128,activation="relu"), tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(3,activation="softmax")])
    m.compile("adam","sparse_categorical_crossentropy",metrics=["accuracy"])
    return m

results = []
for regime in ["per-window", "per-dataset"]:
    for seed in [0, 1, 2]:
        Wtr, Wte, ytr, yte = train_test_split(Wc_raw, yc_r, test_size=0.25,
                                              stratify=yc_r, random_state=seed)
        if regime == "per-window":
            Xtr, Xte = norm_perwindow(Wtr), norm_perwindow(Wte)
            Xp = norm_perwindow(Wp_raw); Xm = norm_perwindow(Wm_raw)
        else:  # per-dataset: source uses train stats; targets use own stats (transfer protocol)
            mu, sd = Wtr.mean(), Wtr.std()
            Xtr, Xte = norm_perdataset(Wtr, mu, sd), norm_perdataset(Wte, mu, sd)
            Xp = norm_perdataset(Wp_raw, Wp_raw.mean(), Wp_raw.std())
            Xm = norm_perdataset(Wm_raw, Wm_raw.mean(), Wm_raw.std())
        cnn = build_cnn(seed)
        cnn.fit(to_img(Xtr), ytr, epochs=25, batch_size=64, verbose=0)
        row = dict(regime=regime, seed=seed)
        for nm, (Xd, yd) in {"CWRU": (Xte, yte), "PB": (Xp, yp_r), "MFPT": (Xm, ym_r)}.items():
            pred = cnn.predict(to_img(Xd), verbose=0).argmax(1)
            row[nm] = balanced_accuracy_score(yd, pred)
        results.append(row)
        print(row)

rdf2 = pd.DataFrame(results)
print("\n", rdf2.groupby("regime")[["CWRU","PB","MFPT"]].agg(["mean","std"]).round(3))
rdf2.to_csv("normalization_ablation.csv", index=False)

In [ ]:
# 1) confirm degenerate predictions under per-dataset regime (retrain one, seed 0)
# ... after a per-dataset training:
for nm, Xd in [("PB", norm_perdataset(Wp_raw, Wp_raw.mean(), Wp_raw.std())),
               ("MFPT", norm_perdataset(Wm_raw, Wm_raw.mean(), Wm_raw.std()))]:
    print(nm, np.bincount(cnn.predict(to_img(Xd), verbose=0).argmax(1), minlength=3))

# 2) does per-window help the FEATURE model too? (control: it shouldn't much,
# since most features are scale-invariant — confirms the decomposition)
Wc_n, Wp_n, Wm_n = norm_perwindow(Wc_raw), norm_perwindow(Wp_raw), norm_perwindow(Wm_raw)
from scipy.stats import skew, kurtosis
def feats(W):
    rms = np.sqrt((W**2).mean(1))
    return np.column_stack([W.max(1), W.min(1), W.mean(1), W.std(1), rms,
        skew(W,1), kurtosis(W,1), W.max(1)/(rms+1e-12),
        rms/(np.abs(W).mean(1)+1e-12)])
Fc, Fp, Fm = feats(Wc_n), feats(Wp_n), feats(Wm_n)
mdl = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                    eval_metric="mlogloss", random_state=0)
mdl.fit(z(Fc), yc_r)
print("XGB on per-window-normalized features:",
      "PB", round(balanced_accuracy_score(yp_r, mdl.predict(z(Fp))), 3),
      "MFPT", round(balanced_accuracy_score(ym_r, mdl.predict(z(Fm))), 3))

In [ ]:
# ============================================================
# FIG 1 REGEN: SHAP summary with Normal/IR/OR labels
# Uses the clean CWRU model from the confidence-audit cell
# ============================================================
import shap
import matplotlib.pyplot as plt

FEATS = ['max','min','mean','sd','rms','skewness','kurtosis','crest','form']
CLASS_NAMES = ["Normal", "IR", "OR"]

# retrain the reference CWRU model (same as ECE cell) to be safe
Xtr, Xte, ytr, yte = train_test_split(Xc_raw, yc_, test_size=0.25,
                                      stratify=yc_, random_state=42)
mu, sd_ = Xtr.mean(0), Xtr.std(0) + 1e-12
mdl = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                    eval_metric="mlogloss", random_state=42)
mdl.fit((Xtr - mu)/sd_, ytr)

explainer = shap.TreeExplainer(mdl)
sv = explainer.shap_values((Xte - mu)/sd_)

plt.figure()
shap.summary_plot(sv, (Xte - mu)/sd_, feature_names=FEATS,
                  class_names=CLASS_NAMES, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("shap_summary_fixed.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# exact adaptation-baseline numbers, CWRU->PB and CWRU->MFPT
from sklearn.metrics import accuracy_score

def coral_align(Xs_, Xt_):
    # whiten source, recolor with target covariance
    cs = np.cov(Xs_, rowvar=False) + np.eye(9)*1e-6
    ct = np.cov(Xt_, rowvar=False) + np.eye(9)*1e-6
    import scipy.linalg as la
    A = la.fractional_matrix_power(cs, -0.5) @ la.fractional_matrix_power(ct, 0.5)
    return (Xs_ - Xs_.mean(0)) @ A.real + Xt_.mean(0)

for tname, (Xt_, yt_) in [("PB", (Xp_raw, yp_)), ("MFPT", (Xm_raw, ym_))]:
    # naive raw
    m1 = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                       eval_metric="mlogloss", random_state=42).fit(Xc_raw, yc_)
    a_raw = accuracy_score(yt_, m1.predict(Xt_))
    # per-dataset standardized
    m2 = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                       eval_metric="mlogloss", random_state=42).fit(z(Xc_raw), yc_)
    a_std = accuracy_score(yt_, m2.predict(z(Xt_)))
    # CORAL
    Xs_c = coral_align(z(Xc_raw), z(Xt_))
    m3 = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                       eval_metric="mlogloss", random_state=42).fit(Xs_c, yc_)
    a_cor = accuracy_score(yt_, m3.predict(z(Xt_)))
    print(f"CWRU->{tname}:  raw={a_raw:.3f}  std={a_std:.3f}  CORAL={a_cor:.3f}")

In [ ]:
from IPython.display import FileLink, display
for f in ["shap_summary_fixed.png",
          "cross_domain_confidence_xgb.png",
          "reliability_all_domains.png",
          "pruning_guided_vs_random.png",
          "pruning_sweep.csv",
          "cnn_mc_dropout_audit.png",
          "normalization_ablation.csv"]:
    try: display(FileLink(f))
    except Exception as e: print("missing:", f, e)

In [ ]:
top4 = [FEATS.index(f) for f in ["max","sd","mean","min"]]
low_feats = [FEATS.index(f) for f in ["rms","crest","skewness"]]  # bottom of new Fig.1
from sklearn.model_selection import cross_val_score
for name, cols in [("full 9", list(range(9))), ("top-4", top4), ("low-3", low_feats)]:
    m = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                      eval_metric="mlogloss", random_state=42)
    s = cross_val_score(m, z(Xc_raw)[:, cols], yc_, cv=5,
                        scoring="balanced_accuracy")
    print(f"{name}: {s.mean():.3f} ± {s.std():.3f}")

In [ ]:
for cols, nm in [([FEATS.index("rms")], "rms only"),
                 ([FEATS.index("crest")], "crest only"),
                 ([FEATS.index("skewness")], "skewness only")]:
    m = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                      eval_metric="mlogloss", random_state=42)
    s = cross_val_score(m, z(Xc_raw)[:, cols], yc_, cv=5, scoring="balanced_accuracy")
    print(f"{nm}: {s.mean():.3f} ± {s.std():.3f}")

In [ ]:
import os; os.remove("/kaggle/working/reliability_diagram.png")

In [ ]:
import pandas as pd, numpy as np

print("="*50)
print("VERIFICATION REPORT")
print("="*50)

# 1) pruning sweep
try:
    r = pd.read_csv("/kaggle/working/pruning_sweep.csv")
    v1 = r[(r.pair=="CWRU->PB") & (r.k==3)].guided_mean.iloc[0]
    v2 = r[(r.pair=="MFPT->CWRU") & (r.k==2)].guided_mean.iloc[0]
    print(f"pruning CWRU->PB k=3  = {v1:.3f}  ->", "PASS" if abs(v1-0.575)<0.03 else "CHECK")
    print(f"pruning MFPT->CWRU k=2 = {v2:.3f}  ->", "PASS" if abs(v2-0.461)<0.03 else "CHECK")
except Exception as e:
    print("pruning_sweep.csv problem:", e)

# 2) normalization ablation
try:
    a = pd.read_csv("/kaggle/working/normalization_ablation.csv")
    g = a.groupby("regime")[["PB","MFPT"]].mean()
    pw_pb, pw_mf = g.loc["per-window","PB"], g.loc["per-window","MFPT"]
    pd_pb = g.loc["per-dataset","PB"]
    print(f"ablation per-window PB = {pw_pb:.3f}  ->", "PASS" if abs(pw_pb-0.465)<0.06 else "CHECK")
    print(f"ablation per-window MFPT = {pw_mf:.3f} ->", "PASS" if abs(pw_mf-0.555)<0.08 else "CHECK")
    print(f"ablation per-dataset PB = {pd_pb:.3f} ->", "PASS" if abs(pd_pb-1/3)<0.01 else "CHECK")
except Exception as e:
    print("normalization_ablation.csv problem:", e)

# 3) files present
import os
need = ["shap_summary_fixed.png","cross_domain_confidence_xgb.png",
        "reliability_all_domains.png","pruning_guided_vs_random.png",
        "cnn_mc_dropout_audit.png","pruning_sweep.csv","normalization_ablation.csv"]
for f in need:
    print(("PASS  " if os.path.exists(f"/kaggle/working/{f}") else "MISSING ") + f)
print("="*50)

In [ ]:
import json
nb =
json.load(open("/kaggle/working/.virtual_documents/__notebook_source__.ipynb"))
for i, c in enumerate(nb["cells"]):
    src = "".join(c.get("source", []))
    if any(k in src.lower() for k in ["dann", "gradient_reversal", "grl", "adversarial"]):
        print(f"--- cell {i} ---")
        print(src[:800])
        for o in c.get("outputs", []):
            print("OUTPUT:", "".join(o.get("text", []))[:500])
            

In [ ]:
import json
path = "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
nb = json.load(open(path))
hits = 0
for i, c in enumerate(nb["cells"]):
    src = "".join(c.get("source", []))
    if any(k in src.lower() for k in ["dann", "gradient_reversal", "grl", "adversarial"]):
        hits += 1
        print(f"--- cell {i} ---")
        print(src[:800])
        for o in c.get("outputs", []):
            txt = "".join(o.get("text", []))
            if txt: print("OUTPUT:", txt[:500])
print("cells found:", hits)

In [2]:
import os

def show_tree(root, max_depth=3, max_items=12):
    root = root.rstrip('/')
    base_depth = root.count(os.sep)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.count(os.sep) - base_depth
        if depth > max_depth:
            dirnames[:] = []
            continue
        indent = '  ' * depth
        print(f'{indent}{os.path.basename(dirpath) or dirpath}/')
        for f in filenames[:max_items]:
            print(f'{indent}  {f}')
        if len(filenames) > max_items:
            print(f'{indent}  ... (+{len(filenames)-max_items} more files)')

# List everything Kaggle mounted so we get exact folder names
for d in sorted(os.listdir('/kaggle/input')):
    print('===', d, '===')

=== datasets ===


In [4]:
import os

root = '/kaggle/input/datasets'
for dirpath, dirnames, filenames in os.walk(root):
    depth = dirpath.replace(root, '').count(os.sep)
    if depth > 2:
        dirnames[:] = []
        continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(dirpath) or dirpath}/')
    for f in filenames[:8]:
        print(f'{indent}  {f}')
    if len(filenames) > 8:
        print(f'{indent}  ... (+{len(filenames)-8} more)')

datasets/
  brjapon/
    cwru-bearing-datasets/
      CWRU_48k_load_1_CNN_data.npz
      feature_time_48k_2048_load_1.csv
  dippatel03/
    paderborn-db/
  sufian79/
    cwru-mat-full-dataset/
      296.mat
      124.mat
      145.mat
      112.mat
      125.mat
      132.mat
      169.mat
      203.mat
      ... (+153 more)
  emperorpein/
    mfpt-fault-datasets/
  mohdsufianbinothman/
    university-of-ottawa-rolling-element-dataset/
    hit-sm-bearing/
      README.md


In [5]:
import os

targets = [
    '/kaggle/input/datasets/mohdsufianbinothman/university-of-ottawa-rolling-element-dataset',
    '/kaggle/input/datasets/mohdsufianbinothman/hit-sm-bearing',
    '/kaggle/input/datasets/dippatel03/paderborn-db',
    '/kaggle/input/datasets/emperorpein/mfpt-fault-datasets',
]

for root in targets:
    print('\n' + '='*70)
    print(root)
    print('='*70)
    if not os.path.exists(root):
        print('  (path not found)')
        continue
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.replace(root, '').count(os.sep)
        if depth > 3:
            dirnames[:] = []
            continue
        indent = '  ' * depth
        print(f'{indent}{os.path.basename(dirpath) or "."}/')
        for f in filenames[:10]:
            print(f'{indent}  {f}')
        if len(filenames) > 10:
            print(f'{indent}  ... (+{len(filenames)-10} more)')


/kaggle/input/datasets/mohdsufianbinothman/university-of-ottawa-rolling-element-dataset
university-of-ottawa-rolling-element-dataset/
  University of Ottawa Rolling-element Dataset – Vibration and Acoustic Faults under Constant Load and Speed conditions (UORED-VAFCLS)/
    2_Excel_Raw_Data_Files (.xlsx)/
      2_Inner_Race_Faults/
        I_4_1.xlsx
        I_1_1.xlsx
        I_3_2.xlsx
        I_4_2.xlsx
        I_2_1.xlsx
        I_1_2.xlsx
        I_3_1.xlsx
        I_5_1.xlsx
        I_2_2.xlsx
        I_5_2.xlsx
      1_Healthy/
        H_3_0.xlsx
        H_4_0.xlsx
        H_10_0.xlsx
        H_8_0.xlsx
        H_9_0.xlsx
        H_14_0.xlsx
        H_11_0.xlsx
        H_5_0.xlsx
        H_2_0.xlsx
        H_13_0.xlsx
        ... (+10 more)
      5_Cage_Faults/
        C_16_2.xlsx
        C_19_2.xlsx
        C_18_1.xlsx
        C_17_1.xlsx
        C_16_1.xlsx
        C_20_1.xlsx
        C_18_2.xlsx
        C_17_2.xlsx
        C_20_2.xlsx
        C_19_1.xlsx
      3_Outer_Race_Fa

In [6]:
import scipy.io as sio
import numpy as np

def probe_mat(path):
    print('---', path)
    m = sio.loadmat(path)
    for k, v in m.items():
        if k.startswith('__'):
            continue
        arr = np.asarray(v)
        print(f'  key={k!r:20s} shape={arr.shape} dtype={arr.dtype}')

base = '/kaggle/input/datasets'

# Ottawa – one healthy .mat
probe_mat(f'{base}/mohdsufianbinothman/university-of-ottawa-rolling-element-dataset/'
          'University of Ottawa Rolling-element Dataset – Vibration and Acoustic Faults under Constant Load and Speed conditions (UORED-VAFCLS)/'
          '3_MatLab_Raw_Data_Files (.mat)/1_Healthy/H_3_0.mat')

# HIT-SM – one normal .mat from SpectraQuest
probe_mat(f'{base}/mohdsufianbinothman/hit-sm-bearing/SpectraQuest MFS dataset/Normal_900.mat')

# HIT-SM – one IR file to confirm same structure
probe_mat(f'{base}/mohdsufianbinothman/hit-sm-bearing/SpectraQuest MFS dataset/IR2_900.mat')

--- /kaggle/input/datasets/mohdsufianbinothman/university-of-ottawa-rolling-element-dataset/University of Ottawa Rolling-element Dataset – Vibration and Acoustic Faults under Constant Load and Speed conditions (UORED-VAFCLS)/3_MatLab_Raw_Data_Files (.mat)/1_Healthy/H_3_0.mat
  key='H_3_0'              shape=(420000, 5) dtype=float64
--- /kaggle/input/datasets/mohdsufianbinothman/hit-sm-bearing/SpectraQuest MFS dataset/Normal_900.mat
  key='VarName3'           shape=(262144, 1) dtype=float64
--- /kaggle/input/datasets/mohdsufianbinothman/hit-sm-bearing/SpectraQuest MFS dataset/IR2_900.mat
  key='z15_1'              shape=(262144, 1) dtype=float64


In [7]:
import scipy.io as sio
import numpy as np

p = ('/kaggle/input/datasets/mohdsufianbinothman/university-of-ottawa-rolling-element-dataset/'
     'University of Ottawa Rolling-element Dataset – Vibration and Acoustic Faults under Constant Load and Speed conditions (UORED-VAFCLS)/'
     '3_MatLab_Raw_Data_Files (.mat)/1_Healthy/H_3_0.mat')

m = sio.loadmat(p)
sig = m['H_3_0']            # (420000, 5)
print('shape', sig.shape)
for c in range(sig.shape[1]):
    col = sig[:, c]
    print(f'col {c}: mean={col.mean(): .4e}  std={col.std(): .4e}  '
          f'min={col.min(): .3e}  max={col.max(): .3e}  '
          f'kurt={((col-col.mean())**4).mean()/col.std()**4: .2f}')

shape (420000, 5)
col 0: mean= 2.2560e+00  std= 2.7654e+00  min=-1.115e+01  max= 1.593e+01  kurt= 3.14
col 1: mean= 1.6093e-02  std= 2.2925e-02  min=-5.977e-02  max= 1.335e-01  kurt= 3.45
col 2: mean= 4.2619e-03  std= 2.7620e+00  min= 0.000e+00  max= 1.790e+03  kurt= 419998.00
col 3: mean= 9.5238e-04  std= 6.1721e-01  min= 0.000e+00  max= 4.000e+02  kurt= 419998.00
col 4: mean= 1.1704e+00  std= 6.7737e-01  min=-2.136e+00  max= 4.423e+00  kurt= 3.00


In [9]:
import glob, os
base = ('/kaggle/input/datasets/mohdsufianbinothman/university-of-ottawa-rolling-element-dataset')
for p in glob.glob(base + '/**/*', recursive=True):
    name = os.path.basename(p).lower()
    if name.endswith(('.md','.txt','.pdf')) or 'readme' in name or 'description' in name:
        print(p)

KeyboardInterrupt: 

In [10]:
import os, glob
import numpy as np
import scipy.io as sio

# ----- 9 time-domain features (max,min,mean,sd,rms,skew,kurt,crest,form) -----
# One shared extractor, so every dataset uses identical feature code.
def extract_9_features(sig, win=2048):
    from scipy.stats import skew, kurtosis
    sig = np.asarray(sig, dtype=np.float64).ravel()
    n = (len(sig) // win) * win
    W = sig[:n].reshape(-1, win)
    rms  = np.sqrt((W**2).mean(1))
    sd   = W.std(1)
    mx   = W.max(1); mn = W.min(1); me = W.mean(1)
    peak = np.abs(W).max(1)
    crest = np.divide(peak, rms, out=np.zeros_like(rms), where=rms>0)
    form  = np.divide(rms, np.abs(W).mean(1)+1e-12, out=np.zeros_like(rms), where=True)
    feats = np.column_stack([mx, mn, me, sd, rms,
                             skew(W, axis=1), kurtosis(W, axis=1),
                             crest, form])
    return feats  # shape (n_windows, 9)

FEATURE_COLS = ['max','min','mean','sd','rms','skew','kurt','crest','form']

def _first_signal_var(matdict):
    """Return the first real variable array in a .mat dict."""
    for k, v in matdict.items():
        if not k.startswith('__'):
            return np.asarray(v)
    raise ValueError('no signal variable found')

# ============================ OTTAWA (UORED) ============================
OTTAWA_ROOT = ('/kaggle/input/datasets/mohdsufianbinothman/'
    'university-of-ottawa-rolling-element-dataset/'
    'University of Ottawa Rolling-element Dataset – Vibration and Acoustic Faults '
    'under Constant Load and Speed conditions (UORED-VAFCLS)/'
    '3_MatLab_Raw_Data_Files (.mat)')

OTTAWA_CLASS_DIRS = {
    'Normal': '1_Healthy',
    'IR':     '2_Inner_Race_Faults',
    'OR':     '3_Outer_Race_Faults',
    # Ball (4_) and Cage (5_) intentionally dropped to keep the 3-class space
}
OTTAWA_VIB_COL = 0   # locked from the channel analysis

def load_ottawa(win=2048):
    X, y = [], []
    for cls, sub in OTTAWA_CLASS_DIRS.items():
        for fp in sorted(glob.glob(os.path.join(OTTAWA_ROOT, sub, '*.mat'))):
            arr = _first_signal_var(sio.loadmat(fp))   # (420000, 5)
            vib = arr[:, OTTAWA_VIB_COL]
            feats = extract_9_features(vib, win)
            X.append(feats); y += [cls]*len(feats)
    return np.vstack(X), np.array(y)

# ============================ HIT-SM ============================
HITSM_ROOT = ('/kaggle/input/datasets/mohdsufianbinothman/'
              'hit-sm-bearing/SpectraQuest MFS dataset')   # one rig only

# filename -> class, e.g. Normal_900.mat, IR2_900.mat, OR5_1200.mat
def _hitsm_class(fname):
    b = os.path.basename(fname).lower()
    if b.startswith('normal'): return 'Normal'
    if b.startswith('ir'):     return 'IR'
    if b.startswith('or'):     return 'OR'
    return None

HITSM_SPEED = None   # set to '900' to fix one speed; None = pool all speeds

def load_hitsm(win=2048, speed=HITSM_SPEED):
    X, y = [], []
    for fp in sorted(glob.glob(os.path.join(HITSM_ROOT, '*.mat'))):
        cls = _hitsm_class(fp)
        if cls is None:
            continue
        if speed is not None and f'_{speed}' not in os.path.basename(fp):
            continue
        vib = _first_signal_var(sio.loadmat(fp)).ravel()  # (262144,)
        feats = extract_9_features(vib, win)
        X.append(feats); y += [cls]*len(feats)
    return np.vstack(X), np.array(y)

# ------------------- quick sanity check -------------------
Xo, yo = load_ottawa()
print('Ottawa:', Xo.shape, dict(zip(*np.unique(yo, return_counts=True))))
Xh, yh = load_hitsm()
print('HIT-SM:', Xh.shape, dict(zip(*np.unique(yh, return_counts=True))))

Ottawa: (8200, 9) {np.str_('IR'): np.int64(2050), np.str_('Normal'): np.int64(4100), np.str_('OR'): np.int64(2050)}
HIT-SM: (2688, 9) {np.str_('IR'): np.int64(1152), np.str_('Normal'): np.int64(384), np.str_('OR'): np.int64(1152)}


In [11]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
# from xgboost import XGBClassifier

# ---- assemble all five datasets ----
# CWRU / Paderborn / MFPT: reuse the same loaders so features are identical
datasets = {
    'CWRU':      load_cwru(),
    'Paderborn': load_paderborn(),  # fixed condition N15_M07_F10
    'MFPT':      load_mfpt(),
    'Ottawa':    load_ottawa(),
    'HITSM':     load_hitsm(),      # add speed='900' to pin a single shaft speed
}
names = list(datasets)

def make_model():
    # return a fresh copy of the within-dataset classifier
    from xgboost import XGBClassifier
    return XGBClassifier(n_estimators=200, max_depth=4, verbosity=0)

# label encoding consistent across datasets
CLASSES = ['Normal','IR','OR']
enc = {c:i for i,c in enumerate(CLASSES)}
def y2i(y): return np.array([enc[c] for c in y])

# ---- transfer matrix ----
M = np.full((len(names), len(names)), np.nan)

for i, src in enumerate(names):
    Xs, ys = datasets[src]; ys = y2i(ys)
    for j, tgt in enumerate(names):
        Xt, yt = datasets[tgt]; yt = y2i(yt)
        if i == j:
            # diagonal: within-dataset 5-fold CV balanced accuracy
            skf = StratifiedKFold(5, shuffle=True, random_state=0)
            accs = []
            for tr, te in skf.split(Xs, ys):
                sc = StandardScaler().fit(Xs[tr])
                clf = make_model().fit(sc.transform(Xs[tr]), ys[tr])
                accs.append(balanced_accuracy_score(ys[te],
                            clf.predict(sc.transform(Xs[te]))))
            M[i, j] = np.mean(accs)
        else:
            # off-diagonal: train on full source, test on full target
            sc = StandardScaler().fit(Xs)          # per-dataset standardization
            clf = make_model().fit(sc.transform(Xs), ys)
            M[i, j] = balanced_accuracy_score(yt, clf.predict(sc.transform(Xt)))

# ---- display ----
import pandas as pd
T = pd.DataFrame(M, index=names, columns=names).round(3)
print(T)
T.to_csv('/kaggle/working/transfer_matrix_5x5.csv')

NameError: name 'load_cwru' is not defined

In [12]:
import os, glob
# check what the earlier run saved
print("working dir:", os.listdir('/kaggle/working'))
# the brjapon CWRU dataset ships a ready-made feature CSV:
print(glob.glob('/kaggle/input/datasets/brjapon/**/*.csv', recursive=True))
print(glob.glob('/kaggle/input/datasets/brjapon/**/*.npz', recursive=True))

working dir: ['.virtual_documents']
['/kaggle/input/datasets/brjapon/cwru-bearing-datasets/feature_time_48k_2048_load_1.csv']
['/kaggle/input/datasets/brjapon/cwru-bearing-datasets/CWRU_48k_load_1_CNN_data.npz']


In [13]:
import scipy.io as sio, numpy as np

def probe(path):
    print('---', os.path.basename(path))
    m = sio.loadmat(path)
    for k,v in m.items():
        if not k.startswith('__'):
            print(f'  {k!r:22s} {np.asarray(v).shape}')

# CWRU (.mat full dataset by sufian79) – pick any file
probe('/kaggle/input/datasets/sufian79/cwru-mat-full-dataset/124.mat')

# MFPT – baseline, inner, outer (different folders)
probe('/kaggle/input/datasets/emperorpein/mfpt-fault-datasets/MFPT Fault Data Sets/1 - Three Baseline Conditions/baseline_1.mat')
probe('/kaggle/input/datasets/emperorpein/mfpt-fault-datasets/MFPT Fault Data Sets/4 - Seven Inner Race Fault Conditions/InnerRaceFault_vload_1.mat')

# Paderborn – one healthy, one IR, one OR (fixed condition N15_M07_F10)
probe('/kaggle/input/datasets/dippatel03/paderborn-db/K002/N15_M07_F10_K002_2.mat')

--- 124.mat
  'X124_DE_time'         (486804, 1)
  'X124_FE_time'         (486804, 1)
  'X124RPM'              (1, 1)
--- baseline_1.mat
  'bearing'              (1, 1)
--- InnerRaceFault_vload_1.mat
  'bearing'              (1, 1)
--- N15_M07_F10_K002_2.mat
  'N15_M07_F10_K002_2'   (1, 1)


In [14]:
import scipy.io as sio, numpy as np

# MFPT struct contents
m = sio.loadmat('/kaggle/input/datasets/emperorpein/mfpt-fault-datasets/'
    'MFPT Fault Data Sets/1 - Three Baseline Conditions/baseline_1.mat')
b = m['bearing']
print('MFPT bearing dtype fields:', b.dtype.names)
for f in b.dtype.names:
    val = b[f][0,0]
    print(f'  {f}: shape={np.asarray(val).shape}')

# Paderborn nested struct
m2 = sio.loadmat('/kaggle/input/datasets/dippatel03/paderborn-db/K002/N15_M07_F10_K002_2.mat')
top = m2['N15_M07_F10_K002_2']
print('\nPaderborn top fields:', top.dtype.names)
Y = top['Y'][0,0]
print('  Y fields:', Y.dtype.names)
print('  Y length:', Y.shape)
# list channel names inside Y
for i in range(Y.shape[1]):
    nm = Y[0,i]['Name'][0] if 'Name' in Y.dtype.names else '?'
    print(f'    Y[{i}] Name = {nm}')

MFPT bearing dtype fields: ('sr', 'gs', 'load', 'rate')
  sr: shape=(1, 1)
  gs: shape=(585936, 1)
  load: shape=(1, 1)
  rate: shape=(1, 1)

Paderborn top fields: ('Info', 'X', 'Y', 'Description')
  Y fields: ('Name', 'Type', 'Data', 'Unit', 'Raster', 'Device', 'XIndex', 'DownSampling', 'Description', 'DisplayIdentifier', 'Path', 'Flags', 'Min', 'Max', 'MinWeak', 'MaxWeak')
  Y length: (1, 7)
    Y[0] Name = force
    Y[1] Name = phase_current_1
    Y[2] Name = phase_current_2
    Y[3] Name = speed
    Y[4] Name = temp_2_bearing_module
    Y[5] Name = torque
    Y[6] Name = vibration_1


In [24]:
import os, glob, numpy as np, scipy.io as sio

# ===================== PADERBORN =====================
# Fixed condition N15_M07_F10; vibration_1 channel (index 6).
# Class from bearing code: K00x=Normal, KIxx=IR, KAxx=OR (KBxx=combined -> drop)
PADERBORN_ROOT = '/kaggle/input/datasets/dippatel03/paderborn-db'

def _pb_class(code):
    if code.startswith('K0'):  return 'Normal'   # K001,K002...K006 healthy
    if code.startswith('KI'):  return 'IR'
    if code.startswith('KA'):  return 'OR'
    return None                                    # KB = combined, skip

def load_paderborn(win=2048, condition='N15_M07_F10'):
    X, y = [], []
    for code in sorted(os.listdir(PADERBORN_ROOT)):
        cls = _pb_class(code)
        if cls is None:
            continue
        cdir = os.path.join(PADERBORN_ROOT, code)
        if not os.path.isdir(cdir):
            continue
        for fp in sorted(glob.glob(os.path.join(cdir, f'{condition}_{code}_*.mat'))):
            m = sio.loadmat(fp)
            key = os.path.splitext(os.path.basename(fp))[0]
            vib = m[key]['Y'][0,0][0,6]['Data'].ravel()   # vibration_1
            feats = extract_9_features(vib, win)
            X.append(feats); y += [cls]*len(feats)
    return np.vstack(X), np.array(y)

# ===================== MFPT =====================
MFPT_ROOT = ('/kaggle/input/datasets/emperorpein/mfpt-fault-datasets/MFPT Fault Data Sets')
MFPT_FILES = {
    'Normal': ['1 - Three Baseline Conditions/baseline_1.mat',
               '1 - Three Baseline Conditions/baseline_2.mat',
               '1 - Three Baseline Conditions/baseline_3.mat'],
    'IR': sorted(glob.glob(os.path.join(MFPT_ROOT,
               '4 - Seven Inner Race Fault Conditions/InnerRaceFault_vload_*.mat'))),
    'OR': sorted(glob.glob(os.path.join(MFPT_ROOT,
               '3 - Seven More Outer Race Fault Conditions/OuterRaceFault_vload_*.mat'))),
}

def load_mfpt(win=2048, resample_to=None):
    """resample_to: target Hz to unify baseline(97656) vs fault(48828). 
       None = keep native sampling rates."""
    from scipy.signal import resample_poly
    X, y = [], []
    for cls, entries in MFPT_FILES.items():
        paths = entries if cls != 'Normal' else [os.path.join(MFPT_ROOT, e) for e in entries]
        for fp in paths:
            b = sio.loadmat(fp)['bearing']
            gs = np.asarray(b['gs'][0,0]).ravel()
            sr = float(b['sr'][0,0].ravel()[0])
            if resample_to is not None and abs(sr - resample_to) > 1:
                # rational resample to common rate
                from math import gcd
                up, down = int(resample_to), int(sr)
                g = gcd(up, down); up//=g; down//=g
                gs = resample_poly(gs, up, down)
            feats = extract_9_features(gs, win)
            X.append(feats); y += [cls]*len(feats)
    return np.vstack(X), np.array(y)

In [25]:
import os, glob, numpy as np, scipy.io as sio

# 1. Is the feature function defined?
print("extract_9_features defined:", 'extract_9_features' in dir())

# 2. Are the loaders defined?
for fn in ['load_ottawa','load_hitsm','load_paderborn','load_mfpt']:
    print(f"{fn} defined:", fn in dir())

# 3. CWRU files present?
cwru = sorted(glob.glob('/kaggle/input/datasets/sufian79/cwru-mat-full-dataset/*.mat'))
print("\nCWRU file count:", len(cwru))
print("CWRU file numbers:", [os.path.splitext(os.path.basename(f))[0] for f in cwru])

# 4. Quick test each loader that should work now
for name, fn in [('paderborn','load_paderborn'), ('mfpt','load_mfpt')]:
    try:
        X, y = eval(fn)()
        print(f"\n{name}: OK {X.shape}", dict(zip(*np.unique(y, return_counts=True))))
    except Exception as e:
        print(f"\n{name}: FAILED -> {type(e).__name__}: {e}")

extract_9_features defined: True
load_ottawa defined: True
load_hitsm defined: True
load_paderborn defined: True
load_mfpt defined: True

CWRU file count: 161
CWRU file numbers: ['100', '105', '106', '107', '108', '109', '110', '111', '112', '118', '119', '120', '121', '122', '123', '124', '125', '130', '131', '132', '133', '135', '136', '137', '138', '144', '145', '146', '147', '148', '149', '150', '151', '156', '158', '159', '160', '161', '162', '163', '164', '169', '170', '171', '172', '174', '175', '176', '177', '185', '186', '187', '188', '189', '190', '191', '192', '197', '198', '199', '200', '201', '202', '203', '204', '209', '210', '211', '212', '213', '214', '215', '217', '222', '223', '224', '225', '226', '227', '228', '229', '234', '235', '236', '237', '238', '239', '240', '241', '246', '247', '248', '249', '250', '251', '252', '253', '258', '259', '260', '261', '262', '263', '264', '265', '270', '271', '272', '273', '274', '275', '276', '277', '278', '279', '280', '281', '2

In [26]:
import os, glob, numpy as np, scipy.io as sio

CWRU_ROOT = '/kaggle/input/datasets/sufian79/cwru-mat-full-dataset'

# Standard CWRU drive-end file-number -> class map.
# Normal baseline: 97,98,99,100
# 12k Drive-End faults (0.007/0.014/0.021", loads 0-3):
#   IR: 105-108, 169-172, 209-212
#   Ball:118-121,185-188,222-225  (DROPPED - keeps 3-class Normal/IR/OR)
#   OR(@6):130-133, 197-200, 234-237
#   OR(@3):144-147, ... ; OR(@12):156-159, ...
CWRU_NORMAL = {'97','98','99','100'}
CWRU_IR = {'105','106','107','108','169','170','171','172','209','210','211','212',
           '3001','3002','3003','3004'}                       # +48k IR if present
CWRU_OR = {'130','131','132','133','197','198','199','200','234','235','236','237',
           '144','145','146','147','156','158','159','160','161',
           '246','247','248','249','258','259','260','261','262','263','264','265',
           '3005','3006','3007','3008'}                       # OR @3/@6/@12 + 48k

def _cwru_class(num):
    if num in CWRU_NORMAL: return 'Normal'
    if num in CWRU_IR:     return 'IR'
    if num in CWRU_OR:     return 'OR'
    return None            # ball faults + anything unmapped -> skip

def load_cwru(win=2048):
    X, y = [], []
    for fp in sorted(glob.glob(os.path.join(CWRU_ROOT, '*.mat'))):
        num = os.path.splitext(os.path.basename(fp))[0]
        cls = _cwru_class(num)
        if cls is None:
            continue
        m = sio.loadmat(fp)
        de = [k for k in m if k.endswith('_DE_time')]   # drive-end channel
        if not de:
            continue
        sig = m[de[0]].ravel()
        feats = extract_9_features(sig, win)
        X.append(feats); y += [cls]*len(feats)
    return np.vstack(X), np.array(y)

Xc, yc = load_cwru()
print('CWRU:', Xc.shape, dict(zip(*np.unique(yc, return_counts=True))))

CWRU: (4496, 9) {np.str_('IR'): np.int64(944), np.str_('Normal'): np.int64(828), np.str_('OR'): np.int64(2724)}


In [27]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from xgboost import XGBClassifier
import numpy as np
enc={'Normal':0,'IR':1,'OR':2}
yi=np.array([enc[c] for c in yc])
Xs=StandardScaler().fit_transform(Xc)
sc=cross_val_score(XGBClassifier(n_estimators=200,max_depth=4,verbosity=0),
                   Xs,yi,cv=StratifiedKFold(5,shuffle=True,random_state=0),
                   scoring='balanced_accuracy')
print('CWRU within-dataset balanced acc:', sc.mean().round(3), '+/-', sc.std().round(3))

CWRU within-dataset balanced acc: 0.997 +/- 0.003


In [28]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

datasets = {
    'CWRU':      (Xc, yc),          # from load_cwru()  (already in memory)
    'Paderborn': load_paderborn(),
    'MFPT':      load_mfpt(),        # native sampling rates
    'Ottawa':    load_ottawa(),
    'HITSM':     load_hitsm(),
}
names = list(datasets)
enc = {'Normal':0,'IR':1,'OR':2}
def y2i(y): return np.array([enc[c] for c in y])

def make_model():
    return XGBClassifier(n_estimators=200, max_depth=4, verbosity=0)

M = np.full((len(names), len(names)), np.nan)
for i, src in enumerate(names):
    Xs, ys = datasets[src]; ys = y2i(ys)
    for j, tgt in enumerate(names):
        Xt, yt = datasets[tgt]; yt = y2i(yt)
        if i == j:
            skf = StratifiedKFold(5, shuffle=True, random_state=0)
            accs = []
            for tr, te in skf.split(Xs, ys):
                scaler = StandardScaler().fit(Xs[tr])
                clf = make_model().fit(scaler.transform(Xs[tr]), ys[tr])
                accs.append(balanced_accuracy_score(ys[te], clf.predict(scaler.transform(Xs[te]))))
            M[i, j] = np.mean(accs)
        else:
            scaler = StandardScaler().fit(Xs)           # standardize on source
            clf = make_model().fit(scaler.transform(Xs), ys)
            M[i, j] = balanced_accuracy_score(yt, clf.predict(scaler.transform(Xt)))

T = pd.DataFrame(M, index=names, columns=names).round(3)
print("Rows = train, Cols = test  (chance = 0.333)\n")
print(T)
T.to_csv('/kaggle/working/transfer_matrix_5x5.csv')

# quick summary
off = M[~np.eye(5, dtype=bool)]
print(f"\nDiagonal mean: {np.diag(M).mean():.3f}")
print(f"Off-diagonal mean: {off.mean():.3f}   (min {off.min():.3f}, max {off.max():.3f})")
print(f"Transfers at/below chance (<=0.333): {(off<=0.333).sum()} / {len(off)}")

Rows = train, Cols = test  (chance = 0.333)

            CWRU  Paderborn   MFPT  Ottawa  HITSM
CWRU       0.997      0.345  0.304   0.337  0.596
Paderborn  0.616      0.897  0.333   0.333  0.333
MFPT       0.625      0.306  0.997   0.482  0.470
Ottawa     0.333      0.333  0.333   0.991  0.333
HITSM      0.438      0.357  0.321   0.400  0.981

Diagonal mean: 0.972
Off-diagonal mean: 0.397   (min 0.304, max 0.625)
Transfers at/below chance (<=0.333): 3 / 20


In [29]:
import numpy as np, pandas as pd
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import StandardScaler

# standardize each dataset's features within-dataset, then per-feature Wasserstein
def gap(A, B):
    A = StandardScaler().fit_transform(A)
    B = StandardScaler().fit_transform(B)
    return np.mean([wasserstein_distance(A[:,k], B[:,k]) for k in range(A.shape[1])])

rows=[]
for i,s in enumerate(names):
    for j,t in enumerate(names):
        if i==j: continue
        g = gap(datasets[s][0], datasets[t][0])
        rows.append({'src':s,'tgt':t,'gap':round(g,3),'transfer':round(M[i,j],3)})
df = pd.DataFrame(rows)
print(df.sort_values('gap').to_string(index=False))

# does gap predict transfer?
from scipy.stats import spearmanr, pearsonr
print('\nSpearman gap vs transfer:', spearmanr(df.gap, df.transfer))
print('Pearson  gap vs transfer:', pearsonr(df.gap, df.transfer))
df.to_csv('/kaggle/working/gap_vs_transfer_20pairs.csv', index=False)

      src       tgt   gap  transfer
Paderborn    Ottawa 0.195     0.333
   Ottawa Paderborn 0.195     0.333
   Ottawa      CWRU 0.217     0.333
     CWRU    Ottawa 0.217     0.337
Paderborn      CWRU 0.218     0.616
     CWRU Paderborn 0.218     0.345
Paderborn     HITSM 0.241     0.333
    HITSM Paderborn 0.241     0.357
    HITSM    Ottawa 0.259     0.400
   Ottawa     HITSM 0.259     0.333
Paderborn      MFPT 0.264     0.333
     MFPT Paderborn 0.264     0.306
     CWRU     HITSM 0.288     0.596
    HITSM      CWRU 0.288     0.438
   Ottawa      MFPT 0.289     0.333
     MFPT    Ottawa 0.289     0.482
     MFPT     HITSM 0.318     0.470
    HITSM      MFPT 0.318     0.321
     MFPT      CWRU 0.336     0.625
     CWRU      MFPT 0.336     0.304

Spearman gap vs transfer: SignificanceResult(statistic=np.float64(0.09562027878366747), pvalue=np.float64(0.6884103574383822))
Pearson  gap vs transfer: PearsonRResult(statistic=np.float64(0.28400499579129534), pvalue=np.float64(0.224934556973

In [30]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy.stats import spearmanr, pearsonr

def directional_gaps(A, B):
    """A=source, B=target. Standardize on SOURCE (as in transfer protocol)."""
    sc = StandardScaler().fit(A)
    As, Bs = sc.transform(A), sc.transform(B)

    # 1. target-to-source coverage: mean dist from each target pt to nearest source pt
    nn = NearestNeighbors(n_neighbors=1).fit(As)
    d_t2s = nn.kneighbors(Bs)[0].mean()          # how well source covers target

    # 2. source-to-target (reverse) for contrast
    nn2 = NearestNeighbors(n_neighbors=1).fit(Bs)
    d_s2t = nn2.kneighbors(As)[0].mean()

    # 3. centroid shift in source-standardized space
    cshift = np.linalg.norm(As.mean(0) - Bs.mean(0))

    # 4. target spread relative to source (variance ratio)
    var_ratio = Bs.var(0).mean() / (As.var(0).mean() + 1e-9)
    return d_t2s, d_s2t, cshift, var_ratio

rows=[]
for i,s in enumerate(names):
    for j,t in enumerate(names):
        if i==j: continue
        d_t2s,d_s2t,cs,vr = directional_gaps(datasets[s][0], datasets[t][0])
        rows.append({'src':s,'tgt':t,'transfer':round(M[i,j],3),
                     'cover_t2s':round(d_t2s,3),'cover_s2t':round(d_s2t,3),
                     'cshift':round(cs,3),'var_ratio':round(vr,3)})
dd = pd.DataFrame(rows)
print(dd.to_string(index=False))
for col in ['cover_t2s','cover_s2t','cshift','var_ratio']:
    r=spearmanr(dd[col], dd.transfer)
    print(f'{col:12s} Spearman rho={r.statistic:+.3f}  p={r.pvalue:.3f}')

      src       tgt  transfer  cover_t2s  cover_s2t    cshift    var_ratio
     CWRU Paderborn     0.345      1.288      1.335     0.864 9.020000e-01
     CWRU      MFPT     0.304      7.079      3.592     9.361 5.059000e+00
     CWRU    Ottawa     0.337   1317.314     13.316  1305.258 4.106674e+05
     CWRU     HITSM     0.596      4.249      2.406     3.948 1.009100e+01
Paderborn      CWRU     0.616      2.144      1.337     1.903 4.869000e+00
Paderborn      MFPT     0.333      8.748      6.748    14.098 1.563400e+01
Paderborn    Ottawa     0.333   1062.611     35.127  1010.351 2.357739e+05
Paderborn     HITSM     0.333      3.944      3.977     6.230 5.447000e+00
     MFPT      CWRU     0.625      2.495      3.170     3.934 4.910000e-01
     MFPT Paderborn     0.306      2.261      2.993     4.115 1.970000e-01
     MFPT    Ottawa     0.482    495.448      8.905   489.004 5.644900e+04
     MFPT     HITSM     0.470      4.675      4.817     5.083 1.798000e+00
   Ottawa      CWRU     0

In [31]:
import numpy as np
for n in names:
    X = datasets[n][0]
    print(f'{n:10s} feat means:', np.round(X.mean(0),2))
    print(f'{" "*10} feat stds :', np.round(X.std(0),2))

CWRU       feat means: [ 2.34 -2.19  0.01  0.45  0.45  0.01  6.17  5.33  1.52]
           feat stds : [2.01 1.88 0.02 0.46 0.46 0.18 8.19 2.04 0.3 ]
Paderborn  feat means: [ 1.82 -1.8   0.01  0.32  0.32  0.    4.6   6.13  1.4 ]
           feat stds : [1.05 0.88 0.03 0.13 0.13 0.37 4.69 1.72 0.15]
MFPT       feat means: [ 7.78 -7.32 -0.15  1.17  1.18  0.22  8.19  5.96  1.45]
           feat stds : [ 6.98  6.21  0.06  0.46  0.46  0.51 12.36  2.92  0.28]
Ottawa     feat means: [ 1.1743e+02 -5.7890e+01  2.6600e+01  1.4320e+01  3.4630e+01  4.0000e-02
  5.4600e+00  4.6400e+00  1.2600e+00]
           feat stds : [118.43 100.53  39.2   13.4   37.81   0.53   9.06   2.97   0.2 ]
HITSM      feat means: [ 0.43 -0.34  0.    0.04  0.04  0.32 22.85  9.43  1.74]
           feat stds : [ 0.39  0.24  0.    0.03  0.03  1.6  21.26  3.81  0.34]


In [32]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy.stats import spearmanr

def dir_gap_scalefree(A, B):
    # standardize EACH dataset independently -> removes scale, keeps shape
    As = StandardScaler().fit_transform(A)
    Bs = StandardScaler().fit_transform(B)
    nn = NearestNeighbors(n_neighbors=1).fit(As)
    cover_t2s = nn.kneighbors(Bs)[0].mean()      # source covers target?
    # energy distance-ish centroid + covariance mismatch
    cshift = np.linalg.norm(As.mean(0)-Bs.mean(0))   # ~0 by construction; keep for check
    # covariance Frobenius distance (captures correlation-structure shift, directional-ish)
    cov_d = np.linalg.norm(np.cov(As.T)-np.cov(Bs.T))
    return cover_t2s, cov_d

rows=[]
for i,s in enumerate(names):
    for j,t in enumerate(names):
        if i==j: continue
        c,cov = dir_gap_scalefree(datasets[s][0], datasets[t][0])
        rows.append({'src':s,'tgt':t,'transfer':round(M[i,j],3),
                     'cover_t2s':round(c,3),'cov_d':round(cov,3)})
dd=pd.DataFrame(rows)
print(dd.to_string(index=False))
for col in ['cover_t2s','cov_d']:
    r=spearmanr(dd[col],dd.transfer); print(f'{col}: rho={r.statistic:+.3f} p={r.pvalue:.3f}')

      src       tgt  transfer  cover_t2s  cov_d
     CWRU Paderborn     0.345      1.156  2.543
     CWRU      MFPT     0.304      0.965  5.121
     CWRU    Ottawa     0.337      1.362  3.542
     CWRU     HITSM     0.596      1.170  2.589
Paderborn      CWRU     0.616      0.839  2.543
Paderborn      MFPT     0.333      0.573  3.290
Paderborn    Ottawa     0.333      0.990  3.429
Paderborn     HITSM     0.333      0.849  2.308
     MFPT      CWRU     0.625      1.312  5.121
     MFPT Paderborn     0.306      1.161  3.290
     MFPT    Ottawa     0.482      1.511  5.680
     MFPT     HITSM     0.470      1.354  3.705
   Ottawa      CWRU     0.333      1.222  3.542
   Ottawa Paderborn     0.333      1.288  3.429
   Ottawa      MFPT     0.333      1.287  5.680
   Ottawa     HITSM     0.333      1.363  3.839
    HITSM      CWRU     0.438      0.959  2.589
    HITSM Paderborn     0.357      0.969  2.308
    HITSM      MFPT     0.321      0.727  3.705
    HITSM    Ottawa     0.400      1.212

In [33]:
def extract_9_features_norm(sig, win=2048):
    from scipy.stats import skew, kurtosis
    sig = np.asarray(sig, float).ravel()
    n=(len(sig)//win)*win
    W=sig[:n].reshape(-1,win)
    W=(W - W.mean(1,keepdims=True))/(W.std(1,keepdims=True)+1e-8)  # per-window normalize
    rms=np.sqrt((W**2).mean(1)); sd=W.std(1)
    mx=W.max(1);mn=W.min(1);me=W.mean(1);peak=np.abs(W).max(1)
    crest=peak/(rms+1e-12); form=rms/(np.abs(W).mean(1)+1e-12)
    return np.column_stack([mx,mn,me,sd,rms,skew(W,axis=1),kurtosis(W,axis=1),crest,form])

In [34]:
import os, glob, numpy as np, pandas as pd, scipy.io as sio
from scipy.stats import skew, kurtosis
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

# ---- normalized feature extractor ----
def extract_9_features_norm(sig, win=2048):
    sig = np.asarray(sig, float).ravel()
    n = (len(sig)//win)*win
    if n == 0:
        return np.empty((0,9))
    W = sig[:n].reshape(-1, win)
    W = (W - W.mean(1, keepdims=True)) / (W.std(1, keepdims=True)+1e-8)
    rms=np.sqrt((W**2).mean(1)); sd=W.std(1)
    mx=W.max(1); mn=W.min(1); me=W.mean(1); peak=np.abs(W).max(1)
    crest=peak/(rms+1e-12); form=rms/(np.abs(W).mean(1)+1e-12)
    return np.column_stack([mx,mn,me,sd,rms,skew(W,axis=1),kurtosis(W,axis=1),crest,form])

# ---- rebuild each dataset by re-running loaders but swapping the extractor ----
# Simplest: temporarily point the global name the loaders call to the norm version.
_orig = extract_9_features
extract_9_features = extract_9_features_norm      # loaders call this name internally

datasets_norm = {
    'CWRU':      load_cwru(),
    'Paderborn': load_paderborn(),
    'MFPT':      load_mfpt(),
    'Ottawa':    load_ottawa(),
    'HITSM':     load_hitsm(),
}
extract_9_features = _orig                         # restore original

for n,(X,y) in datasets_norm.items():
    print(n, X.shape, dict(zip(*np.unique(y, return_counts=True))))

CWRU (4496, 9) {np.str_('IR'): np.int64(944), np.str_('Normal'): np.int64(828), np.str_('OR'): np.int64(2724)}
Paderborn (72669, 9) {np.str_('IR'): np.int64(27547), np.str_('Normal'): np.int64(15019), np.str_('OR'): np.int64(30103)}
MFPT (1852, 9) {np.str_('IR'): np.int64(497), np.str_('Normal'): np.int64(858), np.str_('OR'): np.int64(497)}
Ottawa (8200, 9) {np.str_('IR'): np.int64(2050), np.str_('Normal'): np.int64(4100), np.str_('OR'): np.int64(2050)}
HITSM (2688, 9) {np.str_('IR'): np.int64(1152), np.str_('Normal'): np.int64(384), np.str_('OR'): np.int64(1152)}


In [35]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

names = list(datasets_norm)
enc = {'Normal':0,'IR':1,'OR':2}
def y2i(y): return np.array([enc[c] for c in y])
def make_model(): return XGBClassifier(n_estimators=200, max_depth=4, verbosity=0)

Mn = np.full((5,5), np.nan)
for i,src in enumerate(names):
    Xs,ys = datasets_norm[src]; ys=y2i(ys)
    for j,tgt in enumerate(names):
        Xt,yt = datasets_norm[tgt]; yt=y2i(yt)
        if i==j:
            skf=StratifiedKFold(5,shuffle=True,random_state=0); accs=[]
            for tr,te in skf.split(Xs,ys):
                sc=StandardScaler().fit(Xs[tr])
                clf=make_model().fit(sc.transform(Xs[tr]),ys[tr])
                accs.append(balanced_accuracy_score(ys[te],clf.predict(sc.transform(Xs[te]))))
            Mn[i,j]=np.mean(accs)
        else:
            sc=StandardScaler().fit(Xs)
            clf=make_model().fit(sc.transform(Xs),ys)
            Mn[i,j]=balanced_accuracy_score(yt,clf.predict(sc.transform(Xt)))

Tn=pd.DataFrame(Mn,index=names,columns=names).round(3)
print("NORMALIZED features — Rows=train, Cols=test (chance=0.333)\n")
print(Tn)
off=Mn[~np.eye(5,dtype=bool)]
print(f"\nDiagonal mean: {np.diag(Mn).mean():.3f}")
print(f"Off-diagonal mean: {off.mean():.3f}  (min {off.min():.3f}, max {off.max():.3f})")
print(f"Transfers <=chance: {(off<=0.333).sum()}/20")
Tn.to_csv('/kaggle/working/transfer_matrix_5x5_normalized.csv')

NORMALIZED features — Rows=train, Cols=test (chance=0.333)

            CWRU  Paderborn   MFPT  Ottawa  HITSM
CWRU       0.996      0.360  0.232   0.363  0.405
Paderborn  0.367      0.805  0.628   0.301  0.163
MFPT       0.551      0.296  0.988   0.563  0.687
Ottawa     0.333      0.333  0.333   0.981  0.333
HITSM      0.438      0.341  0.570   0.199  0.976

Diagonal mean: 0.949
Off-diagonal mean: 0.390  (min 0.163, max 0.687)
Transfers <=chance: 5/20


In [36]:
import numpy as np, pandas as pd
from scipy.stats import wasserstein_distance, spearmanr
from sklearn.preprocessing import StandardScaler

def per_class_gap(A, yA, B, yB):
    """mean over classes of (mean per-feature Wasserstein between class-c in A vs B)."""
    A = StandardScaler().fit_transform(A); B = StandardScaler().fit_transform(B)
    gaps=[]
    for c in ['Normal','IR','OR']:
        Ac=A[yA==c]; Bc=B[yB==c]
        if len(Ac)==0 or len(Bc)==0: continue
        gaps.append(np.mean([wasserstein_distance(Ac[:,k],Bc[:,k]) for k in range(A.shape[1])]))
    return np.mean(gaps), gaps

# use RAW-feature datasets — swap to datasets_norm to compare
DS = datasets if 'datasets' in dir() else datasets_norm
rows=[]
for i,s in enumerate(names):
    for j,t in enumerate(names):
        if i==j: continue
        g,_ = per_class_gap(DS[s][0],DS[s][1],DS[t][0],DS[t][1])
        rows.append({'src':s,'tgt':t,'transfer':round(M[i,j],3),'perclass_gap':round(g,3)})
dp=pd.DataFrame(rows)
print(dp.sort_values('perclass_gap').to_string(index=False))
r=spearmanr(dp.perclass_gap,dp.transfer)
print(f'\nper-class gap vs transfer: rho={r.statistic:+.3f} p={r.pvalue:.3f}')

      src       tgt  transfer  perclass_gap
     CWRU     HITSM     0.596         0.464
    HITSM      CWRU     0.438         0.464
   Ottawa      CWRU     0.333         0.669
     CWRU    Ottawa     0.337         0.669
Paderborn      CWRU     0.616         0.677
     CWRU Paderborn     0.345         0.677
    HITSM      MFPT     0.321         0.685
     MFPT     HITSM     0.470         0.685
     CWRU      MFPT     0.304         0.763
     MFPT      CWRU     0.625         0.763
Paderborn     HITSM     0.333         0.786
    HITSM Paderborn     0.357         0.786
   Ottawa Paderborn     0.333         0.799
Paderborn    Ottawa     0.333         0.799
     MFPT    Ottawa     0.482         0.843
   Ottawa      MFPT     0.333         0.843
   Ottawa     HITSM     0.333         0.845
    HITSM    Ottawa     0.400         0.845
     MFPT Paderborn     0.306         1.007
Paderborn      MFPT     0.333         1.007

per-class gap vs transfer: rho=-0.379 p=0.099


In [37]:
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler

CLASSES=['Normal','IR','OR']
DS = datasets  # RAW-feature dicts + raw M must be in memory

def directional_features(A,yA,B,yB):
    # standardize on SOURCE (matches transfer protocol)
    sc=StandardScaler().fit(A); As=sc.transform(A); Bs=sc.transform(B)
    # per-class centroids
    muA={c:As[yA==c].mean(0) for c in CLASSES if (yA==c).any()}
    muB={c:Bs[yB==c].mean(0) for c in CLASSES if (yB==c).any()}
    # (a) how far each target class centroid sits from its OWN source class centroid
    same=np.mean([np.linalg.norm(muA[c]-muB[c]) for c in CLASSES if c in muA and c in muB])
    # (b) does target class land closer to the WRONG source class? (confusability)
    #     count target classes whose nearest source centroid is a different class
    wrong=0; tot=0
    for c in CLASSES:
        if c not in muB: continue
        dists={k:np.linalg.norm(muB[c]-muA[k]) for k in muA}
        nearest=min(dists,key=dists.get)
        wrong += (nearest!=c); tot+=1
    frac_wrong = wrong/max(tot,1)
    # (c) source within-class spread (broad source -> better generalization)
    spreadA=np.mean([As[yA==c].std(0).mean() for c in CLASSES if (yA==c).any()])
    return same, frac_wrong, spreadA

rows=[]
for i,s in enumerate(names):
    for j,t in enumerate(names):
        if i==j: continue
        same,fw,spread=directional_features(DS[s][0],DS[s][1],DS[t][0],DS[t][1])
        rows.append({'src':s,'tgt':t,'transfer':round(M[i,j],3),
                     'centroid_shift':round(same,3),'frac_wrong_nn':fw,
                     'src_spread':round(spread,3)})
dd=pd.DataFrame(rows)
print(dd.to_string(index=False))
for col in ['centroid_shift','frac_wrong_nn','src_spread']:
    r=spearmanr(dd[col],dd.transfer); print(f'{col:15s} rho={r.statistic:+.3f} p={r.pvalue:.3f}')

      src       tgt  transfer  centroid_shift  frac_wrong_nn  src_spread
     CWRU Paderborn     0.345           1.921       0.666667       0.588
     CWRU      MFPT     0.304          11.042       1.000000       0.588
     CWRU    Ottawa     0.337        1310.718       0.666667       0.588
     CWRU     HITSM     0.596           4.796       0.333333       0.588
Paderborn      CWRU     0.616           3.466       0.666667       0.943
Paderborn      MFPT     0.333          16.754       1.000000       0.943
Paderborn    Ottawa     0.333        1062.038       0.666667       0.943
Paderborn     HITSM     0.333           7.203       1.000000       0.943
     MFPT      CWRU     0.625           4.842       0.666667       0.419
     MFPT Paderborn     0.306           4.910       0.666667       0.419
     MFPT    Ottawa     0.482         494.036       0.666667       0.419
     MFPT     HITSM     0.470           5.470       0.666667       0.419
   Ottawa      CWRU     0.333           2.617      

In [38]:
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
CLASSES=['Normal','IR','OR']; DS=datasets

def containment(A,yA,B,yB):
    # standardize each independently (scale-free), per-class variance ratio src/tgt
    A=StandardScaler().fit_transform(A); B=StandardScaler().fit_transform(B)
    ratios=[]
    for c in CLASSES:
        Ac,Bc=A[yA==c],B[yB==c]
        if len(Ac)<5 or len(Bc)<5: continue
        # does source class c cover target class c? var(src)/var(tgt) per feature, mean
        ratios.append(np.mean(Ac.var(0)/(Bc.var(0)+1e-9)))
    return np.mean(ratios)   # >1 => source broader than target => expect better transfer

rows=[]
for i,s in enumerate(names):
    for j,t in enumerate(names):
        if i==j: continue
        rows.append({'src':s,'tgt':t,'transfer':round(M[i,j],3),
                     'containment':round(containment(DS[s][0],DS[s][1],DS[t][0],DS[t][1]),3)})
dc=pd.DataFrame(rows)
print(dc.sort_values('containment',ascending=False).to_string(index=False))
r=spearmanr(dc.containment,dc.transfer); print(f'\ncontainment vs transfer: rho={r.statistic:+.3f} p={r.pvalue:.3f}')

      src       tgt  transfer  containment
Paderborn     HITSM     0.333   452866.616
   Ottawa     HITSM     0.333   260609.976
     MFPT     HITSM     0.470     1490.388
Paderborn      CWRU     0.616     1406.952
   Ottawa      CWRU     0.333     1326.559
Paderborn      MFPT     0.333      658.314
     CWRU    Ottawa     0.337      272.098
    HITSM    Ottawa     0.400      264.773
Paderborn    Ottawa     0.333      186.220
     CWRU     HITSM     0.596       69.357
   Ottawa      MFPT     0.333       54.046
     MFPT    Ottawa     0.482       13.228
     MFPT      CWRU     0.625        7.212
     CWRU      MFPT     0.304        3.878
    HITSM      MFPT     0.321        3.133
    HITSM      CWRU     0.438        1.687
   Ottawa Paderborn     0.333        1.640
     CWRU Paderborn     0.345        0.631
     MFPT Paderborn     0.306        0.528
    HITSM Paderborn     0.357        0.520

containment vs transfer: rho=+0.120 p=0.615


In [39]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
enc={'Normal':0,'IR':1,'OR':2}; y2i=lambda y:np.array([enc[c] for c in y])
rows=[]
for i,s in enumerate(names):
    Xs,ys=datasets[s]; ys=y2i(ys)
    sc=StandardScaler().fit(Xs); clf=XGBClassifier(n_estimators=200,max_depth=4,verbosity=0).fit(sc.transform(Xs),ys)
    for j,t in enumerate(names):
        if i==j: continue
        Xt,yt=datasets[t]; yt=y2i(yt)
        P=clf.predict_proba(sc.transform(Xt)); conf=P.max(1).mean()
        from sklearn.metrics import balanced_accuracy_score
        acc=balanced_accuracy_score(yt,P.argmax(1))
        rows.append({'pair':f'{s}->{t}','transfer':round(acc,3),'mean_conf':round(conf,3),'overconf':round(conf-acc,3)})
dfc=pd.DataFrame(rows); print(dfc.sort_values('overconf',ascending=False).to_string(index=False))
print(f'\nmean overconfidence (conf - acc): {dfc.overconf.mean():.3f}')

             pair  transfer  mean_conf  overconf
    Ottawa->HITSM     0.333      1.000     0.666
     Ottawa->CWRU     0.333      1.000     0.666
Ottawa->Paderborn     0.333      1.000     0.666
     Ottawa->MFPT     0.333      0.999     0.665
  MFPT->Paderborn     0.306      0.968     0.662
 Paderborn->HITSM     0.333      0.991     0.658
Paderborn->Ottawa     0.333      0.989     0.655
      HITSM->MFPT     0.321      0.943     0.622
  CWRU->Paderborn     0.345      0.959     0.614
     CWRU->Ottawa     0.337      0.944     0.607
  Paderborn->MFPT     0.333      0.869     0.536
       CWRU->MFPT     0.304      0.832     0.528
 HITSM->Paderborn     0.357      0.884     0.527
      HITSM->CWRU     0.438      0.885     0.447
     MFPT->Ottawa     0.482      0.922     0.440
      MFPT->HITSM     0.470      0.895     0.426
    HITSM->Ottawa     0.400      0.818     0.418
       MFPT->CWRU     0.625      0.959     0.334
  Paderborn->CWRU     0.616      0.908     0.292
      CWRU->HITSM   

In [40]:
from scipy.stats import spearmanr, pearsonr
r = spearmanr(dfc.transfer, dfc.overconf)
print(f'transfer vs overconfidence: Spearman rho={r.statistic:+.3f} p={r.pvalue:.4f}')
print(f'                            Pearson  r  ={pearsonr(dfc.transfer,dfc.overconf)[0]:+.3f}')
# also: is raw confidence itself anti-correlated with accuracy?
r2 = spearmanr(dfc.transfer, dfc.mean_conf)
print(f'transfer vs mean_conf:      Spearman rho={r2.statistic:+.3f} p={r2.pvalue:.4f}')

transfer vs overconfidence: Spearman rho=-0.779 p=0.0001
                            Pearson  r  =-0.905
transfer vs mean_conf:      Spearman rho=-0.373 p=0.1058


In [41]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from xgboost import XGBClassifier
from scipy.stats import spearmanr
enc={'Normal':0,'IR':1,'OR':2}; y2i=lambda y:np.array([enc[c] for c in y])

rhos, overs = [], []
for seed in range(5):
    rows=[]
    for i,s in enumerate(names):
        Xs,ys=datasets[s]; ys=y2i(ys)
        sc=StandardScaler().fit(Xs)
        clf=XGBClassifier(n_estimators=200,max_depth=4,verbosity=0,random_state=seed).fit(sc.transform(Xs),ys)
        for j,t in enumerate(names):
            if i==j: continue
            Xt,yt=datasets[t]; yt=y2i(yt)
            P=clf.predict_proba(sc.transform(Xt))
            acc=balanced_accuracy_score(yt,P.argmax(1))
            rows.append((acc, P.max(1).mean()-acc))
    r=pd.DataFrame(rows,columns=['acc','over'])
    rhos.append(spearmanr(r.acc,r.over).statistic); overs.append(r.over.mean())
print(f'overconfidence mean: {np.mean(overs):.3f} ± {np.std(overs):.3f}')
print(f'transfer-vs-overconf rho: {np.mean(rhos):+.3f} ± {np.std(rhos):.3f}')

overconfidence mean: 0.530 ± 0.000
transfer-vs-overconf rho: -0.777 ± 0.000


In [42]:
import glob
print(glob.glob('/kaggle/working/*.py') + glob.glob('/kaggle/working/*.pt') +
      glob.glob('/kaggle/working/*.h5') + glob.glob('/kaggle/working/*.keras'))

[]


In [43]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

GPU available: True
Device: Tesla T4


In [44]:
import os, glob, numpy as np, scipy.io as sio, torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn, torch.nn.functional as F

def windows_32x32(sig, win=1024):
    sig = np.asarray(sig, float).ravel()
    n = (len(sig)//win)*win
    if n == 0: return np.empty((0,1,32,32), np.float32)
    W = sig[:n].reshape(-1, win)
    W = (W - W.mean(1,keepdims=True))/(W.std(1,keepdims=True)+1e-8)  # per-window norm
    return W.reshape(-1,1,32,32).astype(np.float32)

CWRU_ROOT='/kaggle/input/datasets/sufian79/cwru-mat-full-dataset'
PB_ROOT='/kaggle/input/datasets/dippatel03/paderborn-db'
MFPT_ROOT='/kaggle/input/datasets/emperorpein/mfpt-fault-datasets/MFPT Fault Data Sets'
OTT_ROOT=('/kaggle/input/datasets/mohdsufianbinothman/university-of-ottawa-rolling-element-dataset/'
 'University of Ottawa Rolling-element Dataset – Vibration and Acoustic Faults under Constant Load and Speed conditions (UORED-VAFCLS)/3_MatLab_Raw_Data_Files (.mat)')
HIT_ROOT='/kaggle/input/datasets/mohdsufianbinothman/hit-sm-bearing/SpectraQuest MFS dataset'

def raw_cwru():
    X,y=[],[]
    for fp in sorted(glob.glob(f'{CWRU_ROOT}/*.mat')):
        num=os.path.splitext(os.path.basename(fp))[0]; cls=_cwru_class(num)
        if cls is None: continue
        m=sio.loadmat(fp); de=[k for k in m if k.endswith('_DE_time')]
        if not de: continue
        W=windows_32x32(m[de[0]].ravel()); X.append(W); y+=[cls]*len(W)
    return np.concatenate(X), np.array(y)

def raw_paderborn(cond='N15_M07_F10'):
    X,y=[],[]
    for code in sorted(os.listdir(PB_ROOT)):
        cls=_pb_class(code); 
        if cls is None or not os.path.isdir(f'{PB_ROOT}/{code}'): continue
        for fp in sorted(glob.glob(f'{PB_ROOT}/{code}/{cond}_{code}_*.mat')):
            m=sio.loadmat(fp); key=os.path.splitext(os.path.basename(fp))[0]
            vib=m[key]['Y'][0,0][0,6]['Data'].ravel()
            W=windows_32x32(vib); X.append(W); y+=[cls]*len(W)
    return np.concatenate(X), np.array(y)

def raw_mfpt():
    files={'Normal':[f'{MFPT_ROOT}/1 - Three Baseline Conditions/baseline_{i}.mat' for i in (1,2,3)],
           'IR':sorted(glob.glob(f'{MFPT_ROOT}/4 - Seven Inner Race Fault Conditions/InnerRaceFault_vload_*.mat')),
           'OR':sorted(glob.glob(f'{MFPT_ROOT}/3 - Seven More Outer Race Fault Conditions/OuterRaceFault_vload_*.mat'))}
    X,y=[],[]
    for cls,ps in files.items():
        for fp in ps:
            gs=sio.loadmat(fp)['bearing']['gs'][0,0].ravel()
            W=windows_32x32(gs); X.append(W); y+=[cls]*len(W)
    return np.concatenate(X), np.array(y)

def raw_ottawa():
    dirs={'Normal':'1_Healthy','IR':'2_Inner_Race_Faults','OR':'3_Outer_Race_Faults'}
    X,y=[],[]
    for cls,sub in dirs.items():
        for fp in sorted(glob.glob(f'{OTT_ROOT}/{sub}/*.mat')):
            m=sio.loadmat(fp); arr=[v for k,v in m.items() if not k.startswith('__')][0]
            W=windows_32x32(np.asarray(arr)[:,0]); X.append(W); y+=[cls]*len(W)
    return np.concatenate(X), np.array(y)

def raw_hitsm():
    X,y=[],[]
    for fp in sorted(glob.glob(f'{HIT_ROOT}/*.mat')):
        b=os.path.basename(fp).lower()
        cls='Normal' if b.startswith('normal') else 'IR' if b.startswith('ir') else 'OR' if b.startswith('or') else None
        if cls is None: continue
        arr=[v for k,v in sio.loadmat(fp).items() if not k.startswith('__')][0]
        W=windows_32x32(np.asarray(arr).ravel()); X.append(W); y+=[cls]*len(W)
    return np.concatenate(X), np.array(y)

raw = {'CWRU':raw_cwru(),'Paderborn':raw_paderborn(),'MFPT':raw_mfpt(),
       'Ottawa':raw_ottawa(),'HITSM':raw_hitsm()}
for n,(X,yy) in raw.items():
    print(n, X.shape, dict(zip(*np.unique(yy,return_counts=True))))

CWRU (9020, 1, 32, 32) {np.str_('IR'): np.int64(1893), np.str_('Normal'): np.int64(1656), np.str_('OR'): np.int64(5471)}
Paderborn (145376, 1, 32, 32) {np.str_('IR'): np.int64(55109), np.str_('Normal'): np.int64(30049), np.str_('OR'): np.int64(60218)}
MFPT (3718, 1, 32, 32) {np.str_('IR'): np.int64(1001), np.str_('Normal'): np.int64(1716), np.str_('OR'): np.int64(1001)}
Ottawa (16400, 1, 32, 32) {np.str_('IR'): np.int64(4100), np.str_('Normal'): np.int64(8200), np.str_('OR'): np.int64(4100)}
HITSM (5376, 1, 32, 32) {np.str_('IR'): np.int64(2304), np.str_('Normal'): np.int64(768), np.str_('OR'): np.int64(2304)}


In [45]:
import torch, numpy as np, time
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score

dev = 'cuda'
enc={'Normal':0,'IR':1,'OR':2}
names=['CWRU','Paderborn','MFPT','Ottawa','HITSM']

class CNN(nn.Module):
    def __init__(self, nc=3):
        super().__init__()
        self.c1=nn.Conv2d(1,16,3,padding=1); self.b1=nn.BatchNorm2d(16)
        self.c2=nn.Conv2d(16,32,3,padding=1); self.b2=nn.BatchNorm2d(32)
        self.c3=nn.Conv2d(32,64,3,padding=1); self.b3=nn.BatchNorm2d(64)
        self.fc1=nn.Linear(64*4*4,128); self.bf=nn.BatchNorm1d(128)
        self.fc2=nn.Linear(128,nc)
    def forward(self,x):
        x=F.max_pool2d(F.relu(self.b1(self.c1(x))),2)   # 16
        x=F.max_pool2d(F.relu(self.b2(self.c2(x))),2)   # 8
        x=F.max_pool2d(F.relu(self.b3(self.c3(x))),2)   # 4
        x=x.flatten(1); x=F.relu(self.bf(self.fc1(x)))
        return self.fc2(x)

def subsample(X,y,cap=20000,seed=0):
    if len(X)<=cap: return X,y
    rng=np.random.RandomState(seed); idx=rng.choice(len(X),cap,replace=False)
    return X[idx],y[idx]

def train_cnn(X,y,epochs=12,bs=256,seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    yi=np.array([enc[c] for c in y])
    X,yi=subsample(X,yi,cap=20000,seed=seed)
    Xtr,Xte,ytr,yte=train_test_split(X,yi,test_size=0.2,stratify=yi,random_state=seed)
    tr=DataLoader(TensorDataset(torch.tensor(Xtr),torch.tensor(ytr)),batch_size=bs,shuffle=True)
    net=CNN().to(dev); opt=torch.optim.Adam(net.parameters(),1e-3)
    for ep in range(epochs):
        net.train()
        for xb,yb in tr:
            xb,yb=xb.to(dev),yb.to(dev)
            opt.zero_grad(); loss=F.cross_entropy(net(xb),yb); loss.backward(); opt.step()
    # within-dataset check
    net.eval()
    with torch.no_grad():
        pred=net(torch.tensor(Xte).to(dev)).argmax(1).cpu().numpy()
    return net, balanced_accuracy_score(yte,pred)

models={}
t0=time.time()
for n in names:
    X,y=raw[n]
    net,acc=train_cnn(X,y)
    models[n]=net
    print(f'{n:10s} within-dataset bal-acc={acc:.3f}   ({time.time()-t0:.0f}s elapsed)')
print('done')

CWRU       within-dataset bal-acc=0.999   (8s elapsed)
Paderborn  within-dataset bal-acc=0.752   (13s elapsed)
MFPT       within-dataset bal-acc=0.998   (14s elapsed)
Ottawa     within-dataset bal-acc=0.991   (19s elapsed)
HITSM      within-dataset bal-acc=0.986   (20s elapsed)
done


In [46]:
net,acc = train_cnn(*raw['Paderborn'], epochs=30, seed=0)
models['Paderborn']=net
print('Paderborn retrained bal-acc=', round(acc,3))

Paderborn retrained bal-acc= 0.739


In [47]:
import copy, torch, numpy as np, pandas as pd
import torch.nn.functional as F
from sklearn.metrics import balanced_accuracy_score

def collect_bn_params(net):
    ps=[]
    for m in net.modules():
        if isinstance(m,(torch.nn.BatchNorm2d,torch.nn.BatchNorm1d)):
            m.train()                        # use batch stats (TENT)
            if m.affine:
                ps += [m.weight, m.bias]
    return ps

@torch.no_grad()
def evaluate(net, X, yi):
    net.eval()
    P=[]
    for i in range(0,len(X),512):
        xb=torch.tensor(X[i:i+512]).to(dev)
        P.append(F.softmax(net(xb),1).cpu().numpy())
    P=np.concatenate(P)
    acc=balanced_accuracy_score(yi, P.argmax(1))
    return acc, P.max(1).mean()

def tent_adapt(base, X, steps=1, bs=256, lr=1e-3):
    net=copy.deepcopy(base)
    # freeze everything except BN affine params
    for p in net.parameters(): p.requires_grad=False
    ps=collect_bn_params(net)
    for p in ps: p.requires_grad=True
    opt=torch.optim.Adam(ps, lr=lr)
    Xt=torch.tensor(X)
    for _ in range(steps):
        perm=torch.randperm(len(Xt))
        for i in range(0,len(Xt),bs):
            xb=Xt[perm[i:i+bs]].to(dev)
            out=net(xb); p=F.softmax(out,1)
            ent=-(p*torch.log(p+1e-8)).sum(1).mean()   # entropy minimization
            opt.zero_grad(); ent.backward(); opt.step()
    return net

rows=[]
for s in names:
    for t in names:
        if s==t: continue
        Xt,yt=raw[t]; yti=np.array([enc[c] for c in yt])
        base=models[s]
        a0,c0=evaluate(base, Xt, yti)                       # source-only
        adapted=tent_adapt(base, Xt, steps=3)
        a1,c1=evaluate(adapted, Xt, yti)                    # TENT
        rows.append({'pair':f'{s}->{t}','src_only_acc':round(a0,3),'tent_acc':round(a1,3),
                     'delta_acc':round(a1-a0,3),'src_conf':round(c0,3),'tent_conf':round(c1,3)})
dt=pd.DataFrame(rows)
print(dt.to_string(index=False))
print(f'\nMean TENT delta-acc: {dt.delta_acc.mean():+.3f}')
print(f'TENT improved (>0.02): {(dt.delta_acc>0.02).sum()}/20   worsened (<-0.02): {(dt.delta_acc<-0.02).sum()}/20')
print(f'Mean confidence  src-only={dt.src_conf.mean():.3f} -> TENT={dt.tent_conf.mean():.3f}')
dt.to_csv('/kaggle/working/tent_results.csv',index=False)

             pair  src_only_acc  tent_acc  delta_acc  src_conf  tent_conf
  CWRU->Paderborn         0.363     0.366      0.003     0.787      0.985
       CWRU->MFPT         0.433     0.537      0.104     0.812      0.884
     CWRU->Ottawa         0.345     0.332     -0.012     0.987      0.942
      CWRU->HITSM         0.335     0.211     -0.124     0.954      0.884
  Paderborn->CWRU         0.208     0.272      0.064     0.955      0.947
  Paderborn->MFPT         0.366     0.257     -0.108     0.920      0.936
Paderborn->Ottawa         0.316     0.168     -0.148     0.958      0.951
 Paderborn->HITSM         0.632     0.698      0.066     0.979      0.953
       MFPT->CWRU         0.582     0.540     -0.042     0.892      0.935
  MFPT->Paderborn         0.205     0.260      0.055     0.933      0.975
     MFPT->Ottawa         0.419     0.405     -0.015     0.928      0.951
      MFPT->HITSM         0.369     0.350     -0.018     0.963      0.871
     Ottawa->CWRU         0.395     0.

In [48]:
import numpy as np, pandas as pd, torch
np.savez('/kaggle/working/all_raw_features.npz',
    **{f'{n}_X':raw[n][0] for n in names}, **{f'{n}_y':raw[n][1] for n in names})
dt.to_csv('/kaggle/working/tent_results.csv',index=False)
for n in names: torch.save(models[n].state_dict(), f'/kaggle/working/cnn_{n}.pt')
print('saved:', [f for f in __import__('os').listdir('/kaggle/working')])

saved: ['cnn_Ottawa.pt', '.virtual_documents', 'tent_results.csv', 'transfer_matrix_5x5.csv', 'cnn_CWRU.pt', 'all_raw_features.npz', 'cnn_MFPT.pt', 'cnn_Paderborn.pt', 'gap_vs_transfer_20pairs.csv', 'transfer_matrix_5x5_normalized.csv', 'cnn_HITSM.pt']


In [49]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt, numpy as np, pandas as pd, os
os.makedirs('/kaggle/working/figs', exist_ok=True)
names=['CWRU','Paderborn','MFPT','Ottawa','HITSM']

# --- Fig A: 5x5 transfer heatmap (raw M) ---
fig,ax=plt.subplots(figsize=(6,5))
im=ax.imshow(M,vmin=0.15,vmax=1.0,cmap='RdYlGn')
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels(names,rotation=45,ha='right'); ax.set_yticklabels(names)
ax.set_xlabel('Test (target)'); ax.set_ylabel('Train (source)')
for i in range(5):
    for j in range(5):
        ax.text(j,i,f'{M[i,j]:.2f}',ha='center',va='center',
                color='black',fontsize=9)
ax.set_title('Cross-dataset transfer (balanced accuracy)\nchance = 0.333')
plt.colorbar(im,fraction=0.046)
plt.tight_layout(); plt.savefig('/kaggle/working/figs/fig_transfer_matrix.png',dpi=200); plt.close()

# --- Fig B: overconfidence vs transfer scatter (headline) ---
fig,ax=plt.subplots(figsize=(6,4.5))
ax.scatter(dfc.transfer,dfc.overconf,s=55,edgecolor='k',color='#c0392b',zorder=3)
z=np.polyfit(dfc.transfer,dfc.overconf,1); xs=np.linspace(dfc.transfer.min(),dfc.transfer.max(),50)
ax.plot(xs,np.polyval(z,xs),'--',color='gray')
ax.axvline(0.333,color='k',ls=':',lw=1); ax.text(0.34,ax.get_ylim()[1]*0.95,'chance',fontsize=8)
ax.set_xlabel('Balanced transfer accuracy'); ax.set_ylabel('Overconfidence (conf − acc)')
ax.set_title('Overconfidence rises as transfer collapses\nρ = −0.78, p = 1e−4')
plt.tight_layout(); plt.savefig('/kaggle/working/figs/fig_overconfidence.png',dpi=200); plt.close()

# --- Fig C: TENT delta-acc per pair ---
d=dt.sort_values('delta_acc')
fig,ax=plt.subplots(figsize=(7,6))
colors=['#2ecc71' if v>0 else '#e74c3c' for v in d.delta_acc]
ax.barh(d.pair,d.delta_acc,color=colors,edgecolor='k',lw=0.4)
ax.axvline(0,color='k',lw=0.8)
ax.set_xlabel('TENT Δ balanced accuracy (adapted − source-only)')
ax.set_title(f'TENT is net-negative: mean Δ={dt.delta_acc.mean():+.3f}\n'
             f'{(dt.delta_acc>0.02).sum()} improved, {(dt.delta_acc<-0.02).sum()} worsened / 20')
plt.tight_layout(); plt.savefig('/kaggle/working/figs/fig_tent.png',dpi=200); plt.close()

# --- Fig D: raw vs normalized transfer (off-diagonals) ---
off=~np.eye(5,dtype=bool)
fig,ax=plt.subplots(figsize=(5.5,5.5))
ax.scatter(M[off],Mn[off],s=55,edgecolor='k',color='#2980b9',zorder=3)
lims=[0.1,0.9]; ax.plot(lims,lims,'--',color='gray')
ax.axhline(0.333,color='k',ls=':',lw=1); ax.axvline(0.333,color='k',ls=':',lw=1)
ax.set_xlabel('Transfer (raw features)'); ax.set_ylabel('Transfer (per-window normalized)')
ax.set_title('Normalization is pair-dependent\n(above line = helped, below = harmed)')
ax.set_xlim(lims); ax.set_ylim(lims)
plt.tight_layout(); plt.savefig('/kaggle/working/figs/fig_norm_vs_raw.png',dpi=200); plt.close()

print('saved:', os.listdir('/kaggle/working/figs'))

saved: ['fig_norm_vs_raw.png', 'fig_transfer_matrix.png', 'fig_overconfidence.png', 'fig_tent.png']


In [50]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt, numpy as np, pandas as pd, os
plt.rcParams.update({'font.size':11,'axes.titlesize':12,'savefig.dpi':300,
                     'figure.dpi':300,'font.family':'serif'})
os.makedirs('/kaggle/working/figs',exist_ok=True)
names=['CWRU','Paderborn','MFPT','Ottawa','HITSM']
DPI=300

# Fig: 5x5 transfer heatmap
fig,ax=plt.subplots(figsize=(6,5))
im=ax.imshow(M,vmin=0.15,vmax=1.0,cmap='RdYlGn')
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels(names,rotation=45,ha='right'); ax.set_yticklabels(names)
ax.set_xlabel('Test (target)'); ax.set_ylabel('Train (source)')
for i in range(5):
    for j in range(5):
        ax.text(j,i,f'{M[i,j]:.2f}',ha='center',va='center',fontsize=9,
                color='white' if M[i,j]<0.3 or M[i,j]>0.9 else 'black')
ax.set_title('Cross-dataset transfer (balanced accuracy), chance=0.333')
plt.colorbar(im,fraction=0.046,pad=0.04)
plt.tight_layout(); plt.savefig('/kaggle/working/figs/fig_transfer_matrix.png',dpi=DPI,bbox_inches='tight'); plt.close()

# Fig: overconfidence vs transfer (HEADLINE)
fig,ax=plt.subplots(figsize=(6,4.5))
ax.scatter(dfc.transfer,dfc.overconf,s=60,edgecolor='k',color='#c0392b',zorder=3)
z=np.polyfit(dfc.transfer,dfc.overconf,1); xs=np.linspace(dfc.transfer.min(),dfc.transfer.max(),50)
ax.plot(xs,np.polyval(z,xs),'--',color='gray',lw=1.5)
ax.axvline(0.333,color='k',ls=':',lw=1); ax.text(0.34,dfc.overconf.max(),'chance',fontsize=9)
ax.set_xlabel('Balanced transfer accuracy'); ax.set_ylabel('Overconfidence (conf − acc)')
ax.set_title('Overconfidence rises as transfer collapses\n$\\rho=-0.78,\\ p=1\\times10^{-4}$')
plt.tight_layout(); plt.savefig('/kaggle/working/figs/fig_overconfidence.png',dpi=DPI,bbox_inches='tight'); plt.close()

# Fig: TENT delta-acc per pair
d=dt.sort_values('delta_acc')
fig,ax=plt.subplots(figsize=(7,6.5))
colors=['#27ae60' if v>0 else '#c0392b' for v in d.delta_acc]
ax.barh(d.pair,d.delta_acc,color=colors,edgecolor='k',lw=0.4)
ax.axvline(0,color='k',lw=0.8)
ax.set_xlabel('TENT $\\Delta$ balanced accuracy (adapted − source-only)')
ax.set_title(f'TENT is net-negative (mean $\\Delta$={dt.delta_acc.mean():+.3f}): '
             f'{(dt.delta_acc>0.02).sum()} improved, {(dt.delta_acc<-0.02).sum()} worsened / 20')
plt.tight_layout(); plt.savefig('/kaggle/working/figs/fig_tent.png',dpi=DPI,bbox_inches='tight'); plt.close()

# Fig: raw vs normalized transfer
off=~np.eye(5,dtype=bool)
fig,ax=plt.subplots(figsize=(5.5,5.5))
ax.scatter(M[off],Mn[off],s=60,edgecolor='k',color='#2980b9',zorder=3)
lims=[0.1,0.9]; ax.plot(lims,lims,'--',color='gray',lw=1.5)
ax.axhline(0.333,color='k',ls=':',lw=1); ax.axvline(0.333,color='k',ls=':',lw=1)
ax.set_xlabel('Transfer (raw features)'); ax.set_ylabel('Transfer (per-window normalized)')
ax.set_title('Amplitude normalization is pair-dependent\n(above line = helped, below = harmed)')
ax.set_xlim(lims); ax.set_ylim(lims)
plt.tight_layout(); plt.savefig('/kaggle/working/figs/fig_norm_vs_raw.png',dpi=DPI,bbox_inches='tight'); plt.close()

print('Cell 1 saved:', sorted(os.listdir('/kaggle/working/figs')))

Cell 1 saved: ['fig_norm_vs_raw.png', 'fig_overconfidence.png', 'fig_tent.png', 'fig_transfer_matrix.png']


In [51]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
import numpy as np, matplotlib.pyplot as plt

# t-SNE colored by dataset (subsample for speed)
rng=np.random.RandomState(0); Xs,lab=[],[]
for n in names:
    X=StandardScaler().fit_transform(datasets[n][0])
    idx=rng.choice(len(X),min(400,len(X)),replace=False)
    Xs.append(X[idx]); lab+=[n]*len(idx)
Xs=np.vstack(Xs); lab=np.array(lab)
emb=TSNE(n_components=2,perplexity=30,init='pca',random_state=0).fit_transform(Xs)
fig,ax=plt.subplots(figsize=(6.5,5.5))
for n,c in zip(names,['#e74c3c','#3498db','#2ecc71','#9b59b6','#f39c12']):
    m=lab==n; ax.scatter(emb[m,0],emb[m,1],s=14,label=n,alpha=0.6,color=c)
ax.legend(markerscale=2,frameon=True); ax.set_xticks([]); ax.set_yticks([])
ax.set_title('t-SNE of features by dataset (each occupies a distinct region)')
plt.tight_layout(); plt.savefig('/kaggle/working/figs/fig_tsne.png',dpi=DPI,bbox_inches='tight'); plt.close()

# Confusion matrix for a dramatic collapse: Ottawa->CWRU
from sklearn.preprocessing import StandardScaler as SS
from xgboost import XGBClassifier
enc={'Normal':0,'IR':1,'OR':2}; y2i=lambda y:np.array([enc[c] for c in y])
Xs_,ys_=datasets['Ottawa']; Xt_,yt_=datasets['CWRU']
sc=SS().fit(Xs_); clf=XGBClassifier(n_estimators=200,max_depth=4,verbosity=0).fit(sc.transform(Xs_),y2i(ys_))
pred=clf.predict(sc.transform(Xt_))
cm=confusion_matrix(y2i(yt_),pred,normalize='true')
fig,ax=plt.subplots(figsize=(4.8,4.2))
im=ax.imshow(cm,vmin=0,vmax=1,cmap='Blues')
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(['Normal','IR','OR']); ax.set_yticklabels(['Normal','IR','OR'])
for i in range(3):
    for j in range(3):
        ax.text(j,i,f'{cm[i,j]:.2f}',ha='center',va='center',
                color='white' if cm[i,j]>0.5 else 'black')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion: Ottawa$\\rightarrow$CWRU (structured collapse)')
plt.colorbar(im,fraction=0.046,pad=0.04)
plt.tight_layout(); plt.savefig('/kaggle/working/figs/fig_confusion.png',dpi=DPI,bbox_inches='tight'); plt.close()
print('Cell 2 saved: fig_tsne.png, fig_confusion.png')

Cell 2 saved: fig_tsne.png, fig_confusion.png


In [52]:
# SHAP summary for within-CWRU tree model
import numpy as np, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
FEATS=['max','min','mean','sd','rms','skew','kurt','crest','form']
Xc_,yc_=datasets['CWRU']; enc={'Normal':0,'IR':1,'OR':2}
Xcs=StandardScaler().fit_transform(Xc_)
clf=XGBClassifier(n_estimators=200,max_depth=4,verbosity=0).fit(Xcs,np.array([enc[c] for c in yc_]))
try:
    import shap
    expl=shap.TreeExplainer(clf); sv=expl.shap_values(Xcs)
    imp=np.abs(np.array(sv)).mean(axis=(0,1)) if isinstance(sv,list) else np.abs(sv).mean(0)
except Exception:
    imp=clf.feature_importances_    # fallback if shap unavailable
order=np.argsort(imp)
fig,ax=plt.subplots(figsize=(5.5,4))
ax.barh([FEATS[i] for i in order],imp[order],color='#34495e',edgecolor='k')
ax.set_xlabel('Mean |SHAP| importance'); ax.set_title('Within-CWRU feature importance (tree model)')
plt.tight_layout(); plt.savefig('/kaggle/working/figs/fig_shap.png',dpi=DPI,bbox_inches='tight'); plt.close()

# Reliability diagram: CWRU in-domain vs foreign (using tree confidences)
from sklearn.model_selection import train_test_split
def reliability(probs,correct,bins=10):
    edges=np.linspace(0,1,bins+1); xs,ys=[],[]
    for b in range(bins):
        m=(probs>=edges[b])&(probs<edges[b+1])
        if m.sum()>0: xs.append(probs[m].mean()); ys.append(correct[m].mean())
    return np.array(xs),np.array(ys)
sc=StandardScaler().fit(Xc_); clf2=XGBClassifier(n_estimators=200,max_depth=4,verbosity=0)
Xtr,Xte,ytr,yte=train_test_split(sc.transform(Xc_),np.array([enc[c] for c in yc_]),test_size=0.3,random_state=0,stratify=yc_)
clf2.fit(Xtr,ytr)
fig,ax=plt.subplots(figsize=(5,5)); ax.plot([0,1],[0,1],'--',color='gray',label='perfect')
# in-domain
P=clf2.predict_proba(Xte); conf=P.max(1); corr=(P.argmax(1)==yte).astype(float)
xs,ys=reliability(conf,corr); ax.plot(xs,ys,'o-',label='CWRU (in-domain)',color='#27ae60')
# foreign: MFPT
Xm,ym=datasets['MFPT']; Pm=clf2.predict_proba(sc.transform(Xm))
confm=Pm.max(1); corrm=(Pm.argmax(1)==np.array([enc[c] for c in ym])).astype(float)
xs,ys=reliability(confm,corrm); ax.plot(xs,ys,'s-',label='MFPT (foreign)',color='#c0392b')
ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy'); ax.legend()
ax.set_title('Reliability: in-domain vs foreign (foreign is overconfident)')
plt.tight_layout(); plt.savefig('/kaggle/working/figs/fig_reliability.png',dpi=DPI,bbox_inches='tight'); plt.close()
print('Cell 3 saved: fig_shap.png, fig_reliability.png')

TypeError: only integer scalar arrays can be converted to a scalar index

In [53]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

FEATS=['max','min','mean','sd','rms','skew','kurt','crest','form']
Xc_,yc_=datasets['CWRU']; enc={'Normal':0,'IR':1,'OR':2}
yci=np.array([enc[c] for c in yc_])
Xcs=StandardScaler().fit_transform(Xc_)
clf=XGBClassifier(n_estimators=200,max_depth=4,verbosity=0).fit(Xcs,yci)

# --- robust per-feature importance ---
imp=None
try:
    import shap
    sv=shap.TreeExplainer(clf).shap_values(Xcs)
    arr=np.array(sv)                      # could be (C,N,F) or (N,F) or (N,F,C)
    arr=np.abs(arr)
    # collapse everything except the last axis of length 9
    while arr.ndim>1 and arr.shape[-1]!=len(FEATS):
        arr=arr.mean(0)
    if arr.ndim>1:                        # e.g. (N,9) -> mean over N
        arr=arr.mean(tuple(range(arr.ndim-1)))
    imp=np.asarray(arr).ravel()
    src='Mean |SHAP| importance'
except Exception as e:
    print('shap fallback:',e)
    imp=None

if imp is None or imp.shape[0]!=len(FEATS):
    imp=np.asarray(clf.feature_importances_).ravel()
    src='Gain-based importance'

imp=imp.astype(float)
order=np.argsort(imp)                     # 1-D int array now
fig,ax=plt.subplots(figsize=(5.5,4))
ax.barh([FEATS[i] for i in order],imp[order],color='#34495e',edgecolor='k')
ax.set_xlabel(src); ax.set_title('Within-CWRU feature importance (tree model)')
plt.tight_layout(); plt.savefig('/kaggle/working/figs/fig_shap.png',dpi=300,bbox_inches='tight'); plt.close()
print('saved fig_shap.png  |', src, '| values:', np.round(imp,3))

# ---------- reliability diagram (unchanged, robust) ----------
def reliability(probs,correct,bins=10):
    edges=np.linspace(0,1,bins+1); xs,ys=[],[]
    for b in range(bins):
        m=(probs>=edges[b])&(probs<edges[b+1])
        if m.sum()>0: xs.append(probs[m].mean()); ys.append(correct[m].mean())
    return np.array(xs),np.array(ys)

sc=StandardScaler().fit(Xc_)
Xtr,Xte,ytr,yte=train_test_split(sc.transform(Xc_),yci,test_size=0.3,random_state=0,stratify=yci)
clf2=XGBClassifier(n_estimators=200,max_depth=4,verbosity=0).fit(Xtr,ytr)
fig,ax=plt.subplots(figsize=(5,5)); ax.plot([0,1],[0,1],'--',color='gray',label='perfect')
P=clf2.predict_proba(Xte); xs,ys=reliability(P.max(1),(P.argmax(1)==yte).astype(float))
ax.plot(xs,ys,'o-',color='#27ae60',label='CWRU (in-domain)')
Xm,ym=datasets['MFPT']; Pm=clf2.predict_proba(sc.transform(Xm))
ymi=np.array([enc[c] for c in ym])
xs,ys=reliability(Pm.max(1),(Pm.argmax(1)==ymi).astype(float))
ax.plot(xs,ys,'s-',color='#c0392b',label='MFPT (foreign)')
ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy'); ax.legend()
ax.set_title('Reliability: in-domain vs foreign')
plt.tight_layout(); plt.savefig('/kaggle/working/figs/fig_reliability.png',dpi=300,bbox_inches='tight'); plt.close()
print('saved fig_reliability.png')

saved fig_shap.png  | Gain-based importance | values: [0.087 0.057 0.017 0.448 0.053 0.108 0.061 0.03  0.139]
saved fig_reliability.png


In [54]:
import shutil, os
shutil.make_archive('/kaggle/working/figures_export', 'zip', '/kaggle/working/figs')
print('created:', '/kaggle/working/figures_export.zip',
      '(', os.path.getsize('/kaggle/working/figures_export.zip')//1024, 'KB )')
print('contents:', sorted(os.listdir('/kaggle/working/figs')))

created: /kaggle/working/figures_export.zip ( 1488 KB )
contents: ['fig_confusion.png', 'fig_norm_vs_raw.png', 'fig_overconfidence.png', 'fig_reliability.png', 'fig_shap.png', 'fig_tent.png', 'fig_transfer_matrix.png', 'fig_tsne.png']


In [55]:
import shutil, os
os.makedirs('/kaggle/working/deliverables', exist_ok=True)
# copy figures
for f in os.listdir('/kaggle/working/figs'):
    shutil.copy(f'/kaggle/working/figs/{f}', '/kaggle/working/deliverables/')
# copy result tables
for f in ['transfer_matrix_5x5.csv','transfer_matrix_5x5_normalized.csv',
          'tent_results.csv','gap_vs_transfer_20pairs.csv']:
    p=f'/kaggle/working/{f}'
    if os.path.exists(p): shutil.copy(p, '/kaggle/working/deliverables/')
shutil.make_archive('/kaggle/working/results_bundle', 'zip', '/kaggle/working/deliverables')
print('created results_bundle.zip:', os.path.getsize('/kaggle/working/results_bundle.zip')//1024,'KB')
print('contents:', sorted(os.listdir('/kaggle/working/deliverables')))

created results_bundle.zip: 1490 KB
contents: ['fig_confusion.png', 'fig_norm_vs_raw.png', 'fig_overconfidence.png', 'fig_reliability.png', 'fig_shap.png', 'fig_tent.png', 'fig_transfer_matrix.png', 'fig_tsne.png', 'gap_vs_transfer_20pairs.csv', 'tent_results.csv', 'transfer_matrix_5x5.csv', 'transfer_matrix_5x5_normalized.csv']


In [56]:
import subprocess; subprocess.run(['pip','install','-q','shap'])

CompletedProcess(args=['pip', 'install', '-q', 'shap'], returncode=0)

In [57]:
import numpy as np, matplotlib.pyplot as plt, shap
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

FEATS=['max','min','mean','sd','rms','skew','kurt','crest','form']
Xc_,yc_=datasets['CWRU']; enc={'Normal':0,'IR':1,'OR':2}
yci=np.array([enc[c] for c in yc_])
Xcs=StandardScaler().fit_transform(Xc_)
clf=XGBClassifier(n_estimators=200,max_depth=4,verbosity=0).fit(Xcs,yci)

sv=shap.TreeExplainer(clf).shap_values(Xcs)
arr=np.abs(np.array(sv))
while arr.ndim>1 and arr.shape[-1]!=len(FEATS): arr=arr.mean(0)
if arr.ndim>1: arr=arr.mean(tuple(range(arr.ndim-1)))
imp=np.asarray(arr).ravel().astype(float)
order=np.argsort(imp)

plt.rcParams.update({'font.family':'serif','font.size':11})
fig,ax=plt.subplots(figsize=(5.5,4))
ax.barh([FEATS[i] for i in order],imp[order],color='#34495e',edgecolor='k')
ax.set_xlabel('Mean |SHAP| importance')
ax.set_title('Within-CWRU SHAP feature importance (tree model)')
plt.tight_layout()
plt.savefig('/kaggle/working/figs/fig_shap.png',dpi=300,bbox_inches='tight'); plt.close()
print('regenerated with SHAP | values:', np.round(imp,3))

# refresh the archive so it has the SHAP version
import shutil
shutil.make_archive('/kaggle/working/figures_export','zip','/kaggle/working/figs')
print('zip refreshed')

regenerated with SHAP | values: [0.465 0.744 0.734]
zip refreshed


In [1]:
import os
print([f for f in os.listdir('/kaggle/working') if f.endswith('.npz')])

[]


In [2]:
import numpy as np
names=['CWRU','Paderborn','MFPT','Ottawa','HITSM']
np.savez('/kaggle/working/all_features.npz',
    **{f'{n}_X':datasets[n][0] for n in names},
    **{f'{n}_y':datasets[n][1] for n in names})
print('saved features:', __import__('os').path.getsize('/kaggle/working/all_features.npz')//1024, 'KB')

NameError: name 'datasets' is not defined

In [3]:
for v in ['extract_9_features','load_cwru','load_paderborn','load_mfpt','load_ottawa','load_hitsm','datasets','raw']:
    print(v, '->', v in dir())

extract_9_features -> False
load_cwru -> False
load_paderborn -> False
load_mfpt -> False
load_ottawa -> False
load_hitsm -> False
datasets -> False
raw -> False


In [4]:
import os, glob, numpy as np, scipy.io as sio, matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

# --- 9-feature extractor ---
def extract_9_features(sig, win=2048):
    sig=np.asarray(sig,float).ravel()
    n=(len(sig)//win)*win
    W=sig[:n].reshape(-1,win)
    rms=np.sqrt((W**2).mean(1)); sd=W.std(1)
    mx=W.max(1);mn=W.min(1);me=W.mean(1);peak=np.abs(W).max(1)
    crest=peak/(rms+1e-12); form=rms/(np.abs(W).mean(1)+1e-12)
    return np.column_stack([mx,mn,me,sd,rms,skew(W,axis=1),kurtosis(W,axis=1),crest,form])

# --- CWRU class map (same as before) ---
CWRU_ROOT='/kaggle/input/datasets/sufian79/cwru-mat-full-dataset'
CWRU_NORMAL={'97','98','99','100'}
CWRU_IR={'105','106','107','108','169','170','171','172','209','210','211','212','3001','3002','3003','3004'}
CWRU_OR={'130','131','132','133','197','198','199','200','234','235','236','237','144','145','146','147','156','158','159','160','161','246','247','248','249','258','259','260','261','262','263','264','265','3005','3006','3007','3008'}
def _cwru_class(num):
    if num in CWRU_NORMAL: return 'Normal'
    if num in CWRU_IR: return 'IR'
    if num in CWRU_OR: return 'OR'
    return None

def load_cwru(win=2048):
    X,y=[],[]
    for fp in sorted(glob.glob(f'{CWRU_ROOT}/*.mat')):
        num=os.path.splitext(os.path.basename(fp))[0]; cls=_cwru_class(num)
        if cls is None: continue
        m=sio.loadmat(fp); de=[k for k in m if k.endswith('_DE_time')]
        if not de: continue
        feats=extract_9_features(m[de[0]].ravel(),win); X.append(feats); y+=[cls]*len(feats)
    return np.vstack(X), np.array(y)

Xc,yc=load_cwru()
print('CWRU loaded:', Xc.shape, dict(zip(*np.unique(yc,return_counts=True))))

# --- SHAP figure, all 9 features ---
FEATS=['max','min','mean','sd','rms','skew','kurt','crest','form']
enc={'Normal':0,'IR':1,'OR':2}; yci=np.array([enc[c] for c in yc])
Xcs=StandardScaler().fit_transform(Xc)
clf=XGBClassifier(n_estimators=200,max_depth=4,verbosity=0).fit(Xcs,yci)

import shap
sv=np.array(shap.TreeExplainer(clf).shap_values(Xcs))
absmean=np.abs(sv)
faxis=[i for i,s in enumerate(absmean.shape) if s==9][0]
other=tuple(i for i in range(absmean.ndim) if i!=faxis)
imp=absmean.mean(axis=other)
assert imp.shape[0]==9, imp.shape
order=np.argsort(imp)

os.makedirs('/kaggle/working/figs',exist_ok=True)
plt.rcParams.update({'font.family':'serif','font.size':11})
fig,ax=plt.subplots(figsize=(5.5,4))
ax.barh([FEATS[i] for i in order], imp[order], color='#34495e', edgecolor='k')
ax.set_xlabel('Mean |SHAP| importance')
ax.set_title('Within-CWRU SHAP feature importance (tree model)')
plt.tight_layout()
plt.savefig('/kaggle/working/figs/fig_shap.png',dpi=300,bbox_inches='tight'); plt.close()
print('9-feature SHAP saved | values:', dict(zip(FEATS, np.round(imp,3))))

CWRU loaded: (4496, 9) {np.str_('IR'): np.int64(944), np.str_('Normal'): np.int64(828), np.str_('OR'): np.int64(2724)}
9-feature SHAP saved | values: {'max': np.float32(0.668), 'min': np.float32(0.373), 'mean': np.float32(0.445), 'sd': np.float32(1.641), 'rms': np.float32(0.536), 'skew': np.float32(0.462), 'kurt': np.float32(0.726), 'crest': np.float32(0.272), 'form': np.float32(0.706)}


In [5]:
from PIL import Image
import os, shutil
p='/kaggle/working/figs/fig_shap.png'
im=Image.open(p).convert('RGBA')
bg=Image.new('RGBA',im.size,(255,255,255,255))
Image.alpha_composite(bg,im).convert('RGB').save(p)
print('fig_shap.png flattened to RGB (no alpha)')
# flatten any remaining figures that still have alpha:
d='/kaggle/working/figs'
for f in os.listdir(d):
    if f.endswith('.png'):
        img=Image.open(os.path.join(d,f)).convert('RGBA')
        b=Image.new('RGBA',img.size,(255,255,255,255))
        Image.alpha_composite(b,img).convert('RGB').save(os.path.join(d,f))
shutil.make_archive('/kaggle/working/figures_export','zip',d)
print('all figures flattened + zip refreshed — re-download figures_export.zip')

fig_shap.png flattened to RGB (no alpha)
all figures flattened + zip refreshed — re-download figures_export.zip
